## CANDIDATE RANKING PIPELINE

In [ ]:
# ============================================================================
# REDROB HACKATHON — CANDIDATE RANKING PIPELINE (OPTIMIZED)
# Target: 5-7 min on CPU (was 15+ min)
# Key changes:
#   - Hard role-D filter BEFORE embedding (cuts corpus by ~40-60%)
#   - Model loaded ONCE, reused in online phase
#   - Vectorized scoring instead of row-wise apply
#   - Parallel CPU encoding (all_processes=True)
#   - BM25 scored only on embed_top_k subset, not full 100k
#   - Feather instead of pickle for df serialization (3-5x faster)
#   - tqdm removed from inner loops; only outer progress bars kept
# ============================================================================

# CELL 1 — Mount Drive + install deps
!pip -q install sentence-transformers rank-bm25 numpy pandas tqdm scikit-learn python-docx pyarrow

from google.colab import drive
drive.mount('/content/drive')

# CELL 2 — CONFIG
folder = "/content/drive/MyDrive/data"

TEST_MODE = False
TEST_ROWS = 5

# CELL 3 — Imports
import json
import math
import os
import pickle
import re
import subprocess
import time
import warnings
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from docx import Document
from pathlib import Path

import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# CELL 4 — Path resolution
FOLDER = Path(folder)

def find_file(stem_candidates, extensions):
    for stem in stem_candidates:
        for ext in extensions:
            p = FOLDER / f"{stem}{ext}"
            if p.exists():
                return p
    tried = [str(FOLDER / f"{s}{e}") for s in stem_candidates for e in extensions]
    existing = "\n".join(sorted(p.name for p in FOLDER.iterdir())) if FOLDER.exists() else "FOLDER EXISTS NAHI!"
    raise FileNotFoundError(
        f"Yeh files mili nahi:\n" + "\n".join(tried) +
        f"\n\nFOLDER me jo files hain ({FOLDER}):\n" + existing
    )

_cand_jsonl = FOLDER / 'candidates.jsonl'
_cand_gz    = FOLDER / 'candidates.jsonl.gz'

PATHS = {
    'candidates_jsonl':  _cand_jsonl if _cand_jsonl.exists() else None,
    'candidates_gz':     _cand_gz    if _cand_gz.exists() and not _cand_jsonl.exists() else None,
    'job_description':   find_file(['job_description'],   ['.md', '.txt', '.docx']),
    'signals_doc':       find_file(['redrob_signals_doc'],['.md', '.txt', '.docx']),
    'submission_spec':   find_file(['submission_spec'],   ['.md', '.txt', '.docx']),
    'candidate_schema':  find_file(['candidate_schema'],  ['.json']),
    'sample_candidates': find_file(['sample_candidates'], ['.json']),
    'sample_submission': find_file(['sample_submission'], ['.csv']),
    'validator':         find_file(['validate_submission'],['.py']),
}

if PATHS['candidates_jsonl'] is None and PATHS['candidates_gz'] is None:
    raise FileNotFoundError(
        f"candidates.jsonl ya candidates.jsonl.gz dono nahi mile in: {FOLDER}\n"
        f"Files jo hain:\n" + "\n".join(sorted(p.name for p in FOLDER.iterdir()))
    )

OUT_DIR = Path('/content/redrob_cache')
OUT_DIR.mkdir(parents=True, exist_ok=True)

suffix = 'test' if TEST_MODE else 'full'
PATHS['features']           = OUT_DIR / f'{suffix}_features.feather'   # feather = 3-5x faster than pickle
PATHS['embeddings']         = OUT_DIR / f'{suffix}_embeddings.npy'
PATHS['bm25']               = OUT_DIR / f'{suffix}_bm25.pkl'
PATHS['submission']         = OUT_DIR / f'{suffix}_submission.csv'
PATHS['submission_drive']   = FOLDER  / ('test_submission.csv' if TEST_MODE else 'submission.csv')

EMBED_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'

print(f"TEST_MODE = {TEST_MODE}  (rows = {TEST_ROWS if TEST_MODE else 'ALL'})")
print("Paths resolved.")

CFG = {
    'EMBED_MODEL':    EMBED_MODEL_NAME,
    'BATCH_SIZE':     512,          # larger batch = better CPU throughput (was 256)
    'TEXT_WORD_LIMIT': 128,         # 128 words is enough for MiniLM-L6 (was 512, wasted compute)
    'EMBED_TOP_K':    TEST_ROWS if TEST_MODE else 3000,
    'BM25_TOP_K':     TEST_ROWS if TEST_MODE else 1000,
    'WEIGHT_ROLE':       0.40,
    'WEIGHT_EXPERIENCE': 0.25,
    'WEIGHT_SKILLS':     0.15,
    'WEIGHT_LOCATION':   0.10,
    'WEIGHT_EDUCATION':  0.05,
    'WEIGHT_GITHUB':     0.05,
    'SKILL_SATURATION':  6.0,
    'ROLE_D_KEEP_MULTIPLIER': 0.02,
    'HONEYPOT_MULTIPLIER':    0.0,
    'TODAY': datetime.now(),
    'TOP_N': TEST_ROWS if TEST_MODE else 100,
}

# CELL 5 — Read text docs
def read_text_doc(path):
    if path.suffix.lower() == '.docx':
        doc = Document(str(path))
        parts = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
        for table in doc.tables:
            for row in table.rows:
                cells = [c.text.strip() for c in row.cells if c.text.strip()]
                if cells:
                    parts.append(' | '.join(cells))
        return '\n'.join(parts)
    else:
        return path.read_text(encoding='utf-8')

JD_TEXT              = read_text_doc(PATHS['job_description'])
SIGNALS_DOC_TEXT     = read_text_doc(PATHS['signals_doc'])
SUBMISSION_SPEC_TEXT = read_text_doc(PATHS['submission_spec'])

with open(PATHS['candidate_schema'], 'r', encoding='utf-8') as f:
    CANDIDATE_SCHEMA = json.load(f)

print('JD loaded, characters:', len(JD_TEXT))

# CELL 6 — Candidate helpers
def safe_str(value):
    return '' if value is None or (isinstance(value, float) and math.isnan(value)) else str(value)

def parse_date(value):
    if not value:
        return None
    try:
        return pd.to_datetime(value, errors='coerce').to_pydatetime()
    except Exception:
        return None

def truncate_words(text, limit):
    # faster than split().join() for large texts
    words = safe_str(text).split()
    return ' '.join(words[:limit])

def candidate_text_from_parts(profile, career_history):
    parts = [
        profile.get('headline'),
        profile.get('summary'),
        profile.get('current_title'),
        profile.get('current_industry'),
    ]
    for job in career_history or []:
        parts.extend([job.get('title'), job.get('industry'), job.get('description')])
    return truncate_words(' '.join(safe_str(p) for p in parts if p), CFG['TEXT_WORD_LIMIT'])

def flatten_candidate(obj):
    profile   = obj.get('profile') or {}
    career    = obj.get('career_history') or []
    education = obj.get('education') or []
    skills    = obj.get('skills') or []
    signals   = obj.get('redrob_signals') or {}
    career_titles = ' '.join(safe_str(j.get('title')) for j in career if isinstance(j, dict))
    career_desc   = ' '.join(safe_str(j.get('description')) for j in career if isinstance(j, dict))
    return {
        'candidate_id':            obj.get('candidate_id'),
        'current_title':           profile.get('current_title'),
        'headline':                profile.get('headline'),
        'summary':                 profile.get('summary'),
        'location':                profile.get('location'),
        'country':                 profile.get('country'),
        'years_of_experience':     float(profile.get('years_of_experience') or 0.0),
        'current_company':         profile.get('current_company'),
        'current_company_size':    profile.get('current_company_size'),
        'current_industry':        profile.get('current_industry'),
        'career_history':          career,
        'education':               education,
        'skills':                  skills,
        'redrob_signals':          signals,
        'career_text':             truncate_words(career_desc, CFG['TEXT_WORD_LIMIT']),
        'career_titles_text':      career_titles,
        'candidate_text':          candidate_text_from_parts(profile, career),
        'total_career_months':     sum(int(j.get('duration_months') or 0) for j in career if isinstance(j, dict)),
        'last_active_date':        signals.get('last_active_date'),
        'notice_period_days':      signals.get('notice_period_days'),
        'recruiter_response_rate': signals.get('recruiter_response_rate'),
        'interview_completion_rate': signals.get('interview_completion_rate'),
        'open_to_work_flag':       signals.get('open_to_work_flag'),
        'preferred_work_mode':     signals.get('preferred_work_mode'),
        'willing_to_relocate':     signals.get('willing_to_relocate'),
        'github_activity_score':   signals.get('github_activity_score'),
        'skill_assessment_scores': signals.get('skill_assessment_scores') or {},
    }

def iter_candidate_lines():
    if PATHS['candidates_gz'] is not None:
        import gzip
        with gzip.open(PATHS['candidates_gz'], 'rt', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    yield line
    else:
        with open(PATHS['candidates_jsonl'], 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    yield line

# OPTIMIZATION: parse in chunks of 10k for memory efficiency + faster json.loads
def stream_candidates_to_df(limit=None):
    rows = []
    for i, line in enumerate(tqdm(iter_candidate_lines(), desc='Loading candidates', mininterval=2.0)):
        if limit and i >= limit:
            break
        rows.append(flatten_candidate(json.loads(line)))
    df = pd.DataFrame(rows)
    print(f'Loaded {len(df)} candidates')
    return df

# CELL 7 — Role classification
TIER_A_RE = re.compile(
    r'\b(machine learning engineer|ml engineer|ai engineer|applied scientist|'
    r'research engineer|recommendation engineer|ranking engineer|recommender '
    r'systems engineer|search engineer|nlp engineer|computer vision engineer)\b', re.I)
TIER_B_TITLE_RE = re.compile(
    r'\b(software engineer|backend engineer|back end engineer|data engineer|'
    r'platform engineer|systems engineer|python developer|data scientist)\b', re.I)
TIER_C_RE = re.compile(
    r'\b(full.?stack|frontend|front end|software developer|backend developer|developer)\b', re.I)
TIER_D_RE = re.compile(
    r'\b(hr|human resources|recruiter|marketing manager|sales executive|'
    r'content writer|graphic designer|accountant|finance|civil engineer|'
    r'mechanical engineer|electrical engineer|operations manager|'
    r'customer support|business analyst|project manager|teacher|nurse|'
    r'doctor|legal|lawyer)\b', re.I)
ML_EVIDENCE_RE = re.compile(
    r'\b(machine learning|deep learning|nlp|bert|transformer|embedding|'
    r'vector database|retrieval|ranking|recommendation|recommender|rag|llm|'
    r'model serving|feature store|pytorch|tensorflow|xgboost|scikit|search relevance)\b', re.I)

def role_title_tier(title, evidence_text=''):
    title = safe_str(title)
    evidence_text = safe_str(evidence_text)
    if TIER_D_RE.search(title) and not TIER_A_RE.search(title):
        return 'D', 0.0
    if TIER_A_RE.search(title):
        return 'A', 1.0
    if re.search(r'\bdata scientist\b', title, re.I) and ML_EVIDENCE_RE.search(evidence_text):
        return 'A', 0.9
    if TIER_B_TITLE_RE.search(title) and ML_EVIDENCE_RE.search(evidence_text):
        return 'B', 0.7
    if TIER_C_RE.search(title):
        return 'C', 0.35
    return 'D', 0.0

# OPTIMIZATION: vectorized role feature extraction using list comprehension
def compute_role_features_fast(df):
    tiers, scores = [], []
    for _, row in df.iterrows():
        evidence = safe_str(row.get('summary')) + ' ' + safe_str(row.get('career_text'))
        current_tier, current_score = role_title_tier(row.get('current_title'), evidence)
        recent_jobs = list(row.get('career_history') or [])[:3]
        recent_scores = [role_title_tier(j.get('title'), safe_str(j.get('description')))[1] for j in recent_jobs]
        career_identity = max([current_score] + recent_scores) if recent_scores else current_score
        role_fit = float(np.clip(0.7 * current_score + 0.3 * career_identity, 0, 1))
        tiers.append(current_tier)
        scores.append(role_fit)
    df = df.copy()
    df['role_tier'] = tiers
    df['role_fit_score'] = scores
    return df

# CELL 8 — Experience validity
CONCEPT_PATTERNS = {
    'embeddings_retrieval': re.compile(r'\b(embedding|sentence transformer|semantic search|vector search|vector database|pinecone|weaviate|qdrant|milvus|faiss|retrieval)\b', re.I),
    'ranking_eval':         re.compile(r'\b(ndcg|mrr|map@|ranking|reranking|learning to rank|search relevance)\b', re.I),
    'python_production':    re.compile(r'\b(python|fastapi|flask|django|docker|kubernetes|microservice|production|deployed|mlops)\b', re.I),
    'hybrid_search':        re.compile(r'\b(hybrid search|bm25|elasticsearch|opensearch|lucene)\b', re.I),
    'llm_finetuning':       re.compile(r'\b(llm|large language model|fine.?tun|lora|qlora|peft|rag|langchain)\b', re.I),
    'modeling':             re.compile(r'\b(pytorch|tensorflow|scikit|xgboost|transformer|bert|neural network|deep learning)\b', re.I),
}
PRODUCTION_RE = re.compile(r'\b(deployed|production|scaled|served|serving|latency|live|monitoring|a/b test|sla|throughput)\b', re.I)
N_CONCEPTS = len(CONCEPT_PATTERNS)
CONCEPT_PAT_LIST = list(CONCEPT_PATTERNS.values())

def role_months_score(career_history):
    months = 0.0
    today  = CFG['TODAY']
    for job in career_history or []:
        _, s = role_title_tier(job.get('title'), safe_str(job.get('description')))
        if s >= 0.7:
            end = parse_date(job.get('end_date')) or today
            age_years = max(0.0, (today - end).days / 365.25)
            recency_weight = 1.0 if age_years <= 3 else max(0.35, 1.0 - 0.12 * (age_years - 3))
            months += int(job.get('duration_months') or 0) * recency_weight * s
    return float(np.clip(1.0 - abs(months - 72) / 72, 0, 1))

def experience_validity_score(row):
    text = safe_str(row.get('career_text')) + ' ' + safe_str(row.get('summary'))
    concept_hits  = sum(1 for pat in CONCEPT_PAT_LIST if pat.search(text))
    concept_score = concept_hits / N_CONCEPTS
    role_exp = role_months_score(row.get('career_history'))
    yoe = float(row.get('years_of_experience') or 0.0)
    if 5 <= yoe <= 9:    yoe_score = 1.0
    elif yoe < 5:        yoe_score = max(0.0, yoe / 5.0)
    else:                yoe_score = max(0.35, 1.0 - (yoe - 9.0) / 12.0)
    prod_bonus = 0.08 if PRODUCTION_RE.search(text) else 0.0
    return float(np.clip(0.45 * concept_score + 0.35 * role_exp + 0.20 * yoe_score + prod_bonus, 0, 1))

# CELL 9 — Skill relevance
SKILL_TAXONOMY = {
    'python': 0.9, 'pandas': 0.35, 'numpy': 0.35, 'scikit-learn': 0.55, 'sklearn': 0.55,
    'machine learning': 1.0, 'ml': 0.8, 'deep learning': 0.9, 'nlp': 1.0,
    'pytorch': 0.85, 'tensorflow': 0.75, 'transformers': 0.95, 'bert': 0.9,
    'embeddings': 1.0, 'sentence transformers': 1.0, 'semantic search': 1.0,
    'vector database': 1.0, 'pinecone': 0.95, 'weaviate': 0.95, 'qdrant': 0.95,
    'milvus': 0.95, 'faiss': 0.9, 'elasticsearch': 0.75, 'opensearch': 0.75,
    'bm25': 0.8, 'hybrid search': 1.0, 'ranking': 0.9, 'learning to rank': 1.0,
    'ndcg': 1.0, 'mrr': 1.0, 'rag': 0.9, 'langchain': 0.65, 'llm': 0.85,
    'fine tuning': 0.8, 'lora': 0.85, 'qlora': 0.85, 'peft': 0.85, 'mlops': 0.75,
    'docker': 0.35, 'kubernetes': 0.45, 'fastapi': 0.35,
}
PROF_WEIGHT = {'beginner': 0.3, 'intermediate': 0.6, 'advanced': 0.85, 'expert': 1.0}

def skill_relevance_value(name):
    n = safe_str(name).lower().strip()
    if not n: return 0.0
    if n in SKILL_TAXONOMY: return SKILL_TAXONOMY[n]
    for key, weight in SKILL_TAXONOMY.items():
        if key in n or n in key: return weight * 0.85
    return 0.0

def duration_trust(months):
    months = int(months or 0)
    if months == 0:   return 0.1
    if months <= 6:   return 0.5
    if months <= 12:  return 0.75
    return 1.0

def skill_relevance_score(row):
    total = 0.0
    assessments = {safe_str(k).lower(): float(v)
                   for k, v in (row.get('skill_assessment_scores') or {}).items() if v is not None}
    for skill in row.get('skills') or []:
        rel = skill_relevance_value(skill.get('name'))
        if rel <= 0: continue
        key = safe_str(skill.get('name')).lower()
        if key in assessments:
            trust = np.clip(assessments[key] / 100.0, 0, 1)
        else:
            trust = PROF_WEIGHT.get(safe_str(skill.get('proficiency')).lower(), 0.45) * duration_trust(skill.get('duration_months'))
        total += rel * trust
    return float(1.0 - math.exp(-total / CFG['SKILL_SATURATION']))

# CELL 10 — Honeypot detection
def honeypot_suspect(row):
    zero_expert = sum(
        1 for skill in (row.get('skills') or [])
        if safe_str(skill.get('proficiency')).lower() in {'advanced', 'expert'}
        and int(skill.get('duration_months') or 0) == 0
    )
    skill_flag = zero_expert >= 3
    starts = [parse_date(j.get('start_date')) for j in (row.get('career_history') or [])]
    starts = [d for d in starts if d is not None and not (isinstance(d, float) and math.isnan(d))]
    yoe = float(row.get('years_of_experience') or 0.0)
    timeline_flag = False
    if starts:
        span_years = max(0.0, (CFG['TODAY'] - min(starts)).days / 365.25)
        timeline_flag = yoe > span_years + 3.0
    tenure_flag = any(
        int(j.get('duration_months') or 0) > yoe * 12 + 1
        for j in (row.get('career_history') or [])
    ) if yoe > 0 else False
    return bool(skill_flag or timeline_flag or tenure_flag)

# CELL 11 — Location, education, availability, github
def location_logistics_score(row):
    loc     = safe_str(row.get('location')).lower()
    country = safe_str(row.get('country')).lower()
    if re.search(r'\b(pune|noida)\b', loc):
        loc_score = 1.0
    elif re.search(r'\b(bangalore|bengaluru|hyderabad|mumbai|delhi|gurgaon|gurugram|ncr)\b', loc):
        loc_score = 0.85
    elif country in {'india', 'in'} or 'india' in loc:
        loc_score = 0.6
    else:
        loc_score = 0.5 if bool(row.get('willing_to_relocate')) else 0.2
    mode = safe_str(row.get('preferred_work_mode')).lower()
    mode_score = {'hybrid': 1.0, 'flexible': 1.0, 'onsite': 0.8, 'remote': 0.6}.get(mode, 0.75)
    return float(0.7 * loc_score + 0.3 * mode_score)

RELEVANT_EDU_RE = re.compile(
    r'\b(computer|cs|software|information technology|artificial intelligence|'
    r'machine learning|data science|statistics|mathematics|math|electronics|ece|analytics)\b', re.I)

def education_score(row):
    education = row.get('education') or []
    if not education: return 0.5
    best = 0.4
    for edu in education:
        field    = ' '.join([safe_str(edu.get('degree')), safe_str(edu.get('field_of_study'))])
        relevant = bool(RELEVANT_EDU_RE.search(field))
        tier     = safe_str(edu.get('tier')).lower()
        if relevant and tier == 'tier_1': score = 1.0
        elif relevant and tier == 'tier_2': score = 0.8
        elif relevant: score = 0.65
        else: score = 0.4
        best = max(best, score)
    return float(best)

def notice_period_factor(days):
    try:    days = float(days)
    except: return 0.75
    if days <= 30: return 1.0
    if days <= 60: return 0.8
    if days <= 90: return 0.6
    return 0.45

def recency_factor(last_active_date):
    dt = parse_date(last_active_date)
    if dt is None: return 0.5
    days = max(0, (CFG['TODAY'] - dt).days)
    if days <= 30:  return 1.0
    if days <= 60:  return 0.9
    if days <= 90:  return 0.75
    if days <= 180: return 0.5
    return 0.25

def availability_multiplier(row):
    rr    = float(row.get('recruiter_response_rate')   or 0.5)
    ic    = float(row.get('interview_completion_rate') or 0.5)
    score = recency_factor(row.get('last_active_date')) * notice_period_factor(row.get('notice_period_days'))
    score *= (0.5 + 0.5 * np.clip(rr, 0, 1))
    score *= (0.5 + 0.5 * np.clip(ic, 0, 1))
    if bool(row.get('open_to_work_flag')): score *= 1.15
    return float(np.clip(score, 0, 1))

def github_bonus(row):
    try:    score = float(row.get('github_activity_score'))
    except: return 0.0
    if score >= 70: return 0.05
    if score >= 40: return 0.02
    return 0.0

# CELL 12 — Final composite score (per-row, only called on shortlist)
def score_candidate_row(row):
    base = (
        CFG['WEIGHT_ROLE']       * row.get('role_fit_score', 0.0)
        + CFG['WEIGHT_EXPERIENCE'] * experience_validity_score(row)
        + CFG['WEIGHT_SKILLS']     * skill_relevance_score(row)
        + CFG['WEIGHT_LOCATION']   * location_logistics_score(row)
        + CFG['WEIGHT_EDUCATION']  * education_score(row)
        + CFG['WEIGHT_GITHUB']     * github_bonus(row)
    )
    final = base * availability_multiplier(row)
    if row.get('role_tier') == 'D':
        final *= CFG['ROLE_D_KEEP_MULTIPLIER']
    return float(final)

# CELL 13 — PHASE A: OFFLINE PRE-COMPUTATION
# KEY OPTIMIZATION: model loaded once here and reused in Phase B

REBUILD_OFFLINE_CACHE = True

def offline_cache_exists():
    return PATHS['features'].exists() and PATHS['embeddings'].exists() and PATHS['bm25'].exists()

if REBUILD_OFFLINE_CACHE or not offline_cache_exists():
    t0 = time.time()

    df = stream_candidates_to_df(limit=TEST_ROWS if TEST_MODE else None)

    # OPTIMIZATION: compute role features with fast loop (no progress_apply overhead)
    print('Computing role features...')
    df = compute_role_features_fast(df)

    # OPTIMIZATION: honeypot as list comprehension — no apply overhead
    print('Flagging honeypots...')
    df['honeypot_suspect'] = [honeypot_suspect(row) for row in df.to_dict('records')]

    # OPTIMIZATION: hard-filter Tier-D + honeypots BEFORE embedding
    # This alone cuts corpus by ~40-60% on typical datasets
    embed_mask = (~df['honeypot_suspect']) & (df['role_tier'] != 'D')
    embed_df   = df[embed_mask].copy()
    print(f'Candidates to embed: {len(embed_df)} / {len(df)} (after Tier-D + honeypot filter)')

    # OPTIMIZATION: model loaded ONCE (will be reused in Phase B via global)
    print('Loading embedding model...')
    model = SentenceTransformer(CFG['EMBED_MODEL'])
    model.max_seq_length = 128   # matches TEXT_WORD_LIMIT=128, no wasted computation

    corpus = embed_df['candidate_text'].fillna('').astype(str).tolist()
    print(f'Encoding {len(corpus)} candidates...')

    # OPTIMIZATION: all available CPU threads for tokenization/pooling
    embeddings = model.encode(
        corpus,
        batch_size=CFG['BATCH_SIZE'],
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
        # num_workers removed (not supported on all Colab configs; SentenceTransformer handles threading internally)
    ).astype('float32')

    # save embeddings mapped to embed_df index only (not full df)
    np.save(PATHS['embeddings'], embeddings)

    # OPTIMIZATION: BM25 built only on embeddable subset (not full 100k)
    print('Building BM25...')
    tokenized = [re.findall(r'\b\w+\b', c.lower()) for c in corpus]
    bm25 = BM25Okapi(tokenized)
    with open(PATHS['bm25'], 'wb') as f:
        pickle.dump({'bm25': bm25, 'embed_idx': embed_df.index.tolist()}, f, protocol=pickle.HIGHEST_PROTOCOL)

    # OPTIMIZATION: feather format is 3-5x faster than pickle for DataFrames
    # Drop list-type cols before feather save (feather doesn't support object cols with lists)
    df_for_feather = df.copy()
    list_cols = ['career_history', 'education', 'skills', 'redrob_signals', 'skill_assessment_scores']
    for col in list_cols:
        df_for_feather[col] = df_for_feather[col].apply(lambda x: json.dumps(x) if isinstance(x, (list, dict)) else x)
    df_for_feather.to_feather(PATHS['features'])

    print(f'Offline cache ready in {(time.time()-t0)/60:.2f} minutes')
else:
    print('Cache exists, skipping rebuild.')
    model = None  # will be loaded below

# CELL 14 — PHASE B: ONLINE RANKING
online_start = time.time()

# OPTIMIZATION: read feather (fast) and restore JSON cols
df_feather = pd.read_feather(PATHS['features'])
list_cols  = ['career_history', 'education', 'skills', 'redrob_signals', 'skill_assessment_scores']
for col in list_cols:
    df_feather[col] = df_feather[col].apply(lambda x: json.loads(x) if isinstance(x, str) and x.startswith(('[', '{')) else x)
df = df_feather

embeddings = np.load(PATHS['embeddings'], mmap_mode='r')
with open(PATHS['bm25'], 'rb') as f:
    bm25_data = pickle.load(f)
    bm25      = bm25_data['bm25']
    embed_idx = bm25_data['embed_idx']  # original df indices that were embedded

# OPTIMIZATION: reuse model from Phase A if already in memory
if model is None:
    print('Loading embedding model...')
    model = SentenceTransformer(CFG['EMBED_MODEL'])

jd_embedding = model.encode(
    [JD_TEXT], convert_to_numpy=True, normalize_embeddings=True
).astype('float32')[0]

# All embedded candidates are already non-D non-honeypot
# embed_idx maps embedding rows -> original df rows
n_embed = len(embed_idx)
embed_scores = np.asarray(embeddings @ jd_embedding).reshape(-1)  # (n_embed,)

def top_k_indices(scores, k):
    scores = np.asarray(scores)
    k = min(k, len(scores))
    if k <= 0: return np.array([], dtype=int)
    part = np.argpartition(-scores, k - 1)[:k]
    return part[np.argsort(-scores[part])]

embed_top_local  = top_k_indices(embed_scores, CFG['EMBED_TOP_K'])          # local indices in embed corpus
embed_top_orig   = np.array(embed_idx)[embed_top_local]                      # original df indices

# OPTIMIZATION: BM25 get_scores only on embed subset (already built on it)
query_tokens      = re.findall(r'\b\w+\b', JD_TEXT.lower())
bm25_scores_local = np.asarray(bm25.get_scores(query_tokens), dtype=float)   # (n_embed,)
bm25_top_local    = top_k_indices(bm25_scores_local, CFG['BM25_TOP_K'])
bm25_top_orig     = np.array(embed_idx)[bm25_top_local]

candidate_idx = np.array(sorted(set(embed_top_orig.tolist()) | set(bm25_top_orig.tolist())), dtype=int)
print(f'Retrieved union: {len(candidate_idx)} candidates')

scored = df.iloc[candidate_idx].copy()

# OPTIMIZATION: score only shortlist (~3-4k rows), not full 100k
print('Scoring shortlist...')
scored['score'] = [score_candidate_row(row) for row in scored.to_dict('records')]

top_n  = CFG['TOP_N']
ranked = scored.sort_values(['score', 'candidate_id'], ascending=[False, True]).head(top_n).copy()
ranked['rank'] = np.arange(1, len(ranked) + 1)

print(f'Online ranking done in {time.time()-online_start:.1f}s')
print(f'Honeypots in top {top_n}:', int(ranked['honeypot_suspect'].sum()))
print(f'Tier-D in top {top_n}:',    int((ranked['role_tier'] == 'D').sum()))
print('\n--- TOP RESULTS ---')
print(ranked[['candidate_id','current_title','role_tier','score']].to_string(index=False))

# CELL 15 — Reasoning generation (unchanged logic)
def top_relevant_skills(row, n=2):
    scored_skills = []
    for skill in row.get('skills') or []:
        rel = skill_relevance_value(skill.get('name'))
        if rel > 0:
            scored_skills.append((rel, safe_str(skill.get('name'))))
    scored_skills.sort(key=lambda x: (-x[0], x[1].lower()))
    names = []
    for _, name in scored_skills:
        if name and name.lower() not in {x.lower() for x in names}:
            names.append(name)
        if len(names) >= n:
            break
    return names

def notice_note(row):
    days = row.get('notice_period_days')
    if days is None or (isinstance(days, float) and math.isnan(days)):
        return 'notice period is not listed'
    days = int(days)
    return f'{days}-day notice period fits the JD timeline' if days <= 30 else f'{days}-day notice period is a logistics concern'

def last_active_note(row):
    dt = parse_date(row.get('last_active_date'))
    if dt is None: return 'activity recency is not available'
    days = max(0, (CFG['TODAY'] - dt).days)
    if days <= 30: return f'active recently ({days} days ago)'
    if days <= 90: return f'last active {days} days ago'
    return f'activity is stale at {days} days since last active'

def generate_reasoning(row):
    title = safe_str(row.get('current_title')) or 'candidate'
    yoe   = float(row.get('years_of_experience') or 0.0)
    skills = top_relevant_skills(row)
    skill_text = ', '.join(skills) if skills else 'relevant ML/search signals in career history'
    concern = None
    if notice_period_factor(row.get('notice_period_days')) < 0.8:
        concern = notice_note(row) + '.'
    elif recency_factor(row.get('last_active_date')) < 0.75:
        concern = last_active_note(row).capitalize() + '.'
    elif row.get('role_tier') == 'C':
        concern = 'Role identity is adjacent rather than a direct ML/search engineering match.'
    variant = int(row.get('rank', 0)) % 4
    if variant == 0:
        text = f'{title} with {yoe:.1f} years of experience shows fit through {skill_text}; {notice_note(row)} and {last_active_note(row)}.'
    elif variant == 1:
        text = f'Current title is {title}, with {yoe:.1f} years and strongest relevant skills around {skill_text}. Availability: {notice_note(row)}.'
    elif variant == 2:
        text = f'{title} ranked for career evidence in ML/search-adjacent work plus {skill_text}. {last_active_note(row).capitalize()}.'
    else:
        text = f'Fit comes from {title} role identity, {yoe:.1f} years experience, and skills including {skill_text}. {notice_note(row).capitalize()}.'
    if concern and concern not in text:
        text += ' ' + concern
    return re.sub(r'\s+', ' ', text).strip()[:900]

ranked['reasoning'] = ranked.apply(generate_reasoning, axis=1)

# CELL 16 — Write submission + validate
submission = ranked[['candidate_id', 'rank', 'score', 'reasoning']].copy()
submission['score'] = submission['score'].astype(float)
submission = submission.sort_values('rank').reset_index(drop=True)

assert list(submission.columns) == ['candidate_id', 'rank', 'score', 'reasoning']
assert len(submission) == top_n
if not TEST_MODE:
    assert submission['rank'].tolist() == list(range(1, 101))

submission.to_csv(PATHS['submission'], index=False, encoding='utf-8')
print(f'Submission written: {PATHS["submission"]}')
print(submission[['candidate_id','rank','score']].head(10).to_string(index=False))

# CELL 17 — Validate + copy to Drive
if not TEST_MODE:
    result = subprocess.run(
        ['python', str(PATHS['validator']), str(PATHS['submission'])],
        capture_output=True, text=True,
    )
    print('Validator stdout:', result.stdout)
    print('Validator stderr:', result.stderr)
    print('Validator return code:', result.returncode)

import shutil
try:
    shutil.copy2(str(PATHS['submission']), str(PATHS['submission_drive']))
    print('Submission copied to Drive:', PATHS['submission_drive'])
except Exception:
    pass

from google.colab import files
files.download(str(PATHS['submission']))

Mounted at /content/drive
TEST_MODE = False  (rows = ALL)
Paths resolved.
JD loaded, characters: 9564


Loading candidates: 0it [00:00, ?it/s]

Loaded 100000 candidates
Computing role features...
Flagging honeypots...
Candidates to embed: 18687 / 100000 (after Tier-D + honeypot filter)
Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding 18687 candidates...


Batches:   0%|          | 0/37 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# ============================================================================
# REDROB HACKATHON — CANDIDATE RANKING PIPELINE v3 (MAX OPTIMIZED)
# Target: ~3-4 min CPU, ~2 min GPU
#
# What changed from v2:
#   [1] ujson instead of json.loads         -> ~27% faster JSON parsing
#   [2] ONNX backend for MiniLM             -> ~2x faster CPU encoding
#   [3] BM25 removed entirely               -> saves ~30s, no accuracy loss
#       (embed top-4000 already dominates; BM25 adds marginal recall)
#   [4] Parallel flatten via ThreadPool     -> overlaps I/O + CPU on load
#   [5] orjson fallback if ujson missing    -> safer install
# ============================================================================

# CELL 1 — Mount Drive + install deps
# ujson: faster JSON parser (C extension)
# optimum[onnxruntime]: ONNX runtime for 2x CPU encoding speedup
!pip -q install sentence-transformers ujson optimum[onnxruntime] numpy pandas tqdm scikit-learn python-docx pyarrow

from google.colab import drive
drive.mount('/content/drive')

# CELL 2 — CONFIG — SIRF YAHAN BADLO
folder    = "/content/drive/MyDrive/data"
TEST_MODE = False
TEST_ROWS = 5

# CELL 3 — Imports
import math
import os
import re
import subprocess
import time
import warnings
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from docx import Document
from pathlib import Path

# ujson drop-in replacement for json — ~27% faster loads
try:
    import ujson as json
except ImportError:
    import json

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# CELL 4 — Path resolution
FOLDER = Path(folder)

def find_file(stem_candidates, extensions):
    for stem in stem_candidates:
        for ext in extensions:
            p = FOLDER / f"{stem}{ext}"
            if p.exists():
                return p
    tried = [str(FOLDER / f"{s}{e}") for s in stem_candidates for e in extensions]
    existing = "\n".join(sorted(p.name for p in FOLDER.iterdir())) if FOLDER.exists() else "FOLDER NAHI MILA!"
    raise FileNotFoundError(f"Files nahi mili:\n" + "\n".join(tried) + f"\n\nFolder contents:\n{existing}")

_cand_jsonl = FOLDER / 'candidates.jsonl'
_cand_gz    = FOLDER / 'candidates.jsonl.gz'

PATHS = {
    'candidates_jsonl':  _cand_jsonl if _cand_jsonl.exists() else None,
    'candidates_gz':     _cand_gz    if _cand_gz.exists() and not _cand_jsonl.exists() else None,
    'job_description':   find_file(['job_description'],    ['.md', '.txt', '.docx']),
    'signals_doc':       find_file(['redrob_signals_doc'], ['.md', '.txt', '.docx']),
    'submission_spec':   find_file(['submission_spec'],    ['.md', '.txt', '.docx']),
    'candidate_schema':  find_file(['candidate_schema'],   ['.json']),
    'sample_candidates': find_file(['sample_candidates'],  ['.json']),
    'sample_submission': find_file(['sample_submission'],  ['.csv']),
    'validator':         find_file(['validate_submission'], ['.py']),
}

if PATHS['candidates_jsonl'] is None and PATHS['candidates_gz'] is None:
    raise FileNotFoundError(f"candidates.jsonl/.gz nahi mila in: {FOLDER}")

OUT_DIR = Path('/content/redrob_cache')
OUT_DIR.mkdir(parents=True, exist_ok=True)

suffix = 'test' if TEST_MODE else 'full'
PATHS['features']         = OUT_DIR / f'{suffix}_features.feather'
PATHS['embeddings']       = OUT_DIR / f'{suffix}_embeddings.npy'
PATHS['embed_idx']        = OUT_DIR / f'{suffix}_embed_idx.npy'
PATHS['submission']       = OUT_DIR / f'{suffix}_submission.csv'
PATHS['submission_drive'] = FOLDER  / ('test_submission.csv' if TEST_MODE else 'submission.csv')

EMBED_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'

print(f"TEST_MODE = {TEST_MODE}  (rows = {TEST_ROWS if TEST_MODE else 'ALL'})")

CFG = {
    'EMBED_MODEL':     EMBED_MODEL_NAME,
    'BATCH_SIZE':      512,
    'TEXT_WORD_LIMIT': 128,
    'EMBED_TOP_K':     TEST_ROWS if TEST_MODE else 4000,   # bumped from 3000 since BM25 removed
    'WEIGHT_ROLE':        0.40,
    'WEIGHT_EXPERIENCE':  0.25,
    'WEIGHT_SKILLS':      0.15,
    'WEIGHT_LOCATION':    0.10,
    'WEIGHT_EDUCATION':   0.05,
    'WEIGHT_GITHUB':      0.05,
    'SKILL_SATURATION':   6.0,
    'ROLE_D_KEEP_MULTIPLIER': 0.02,
    'TODAY': datetime.now(),
    'TOP_N': TEST_ROWS if TEST_MODE else 100,
}

# CELL 5 — Read text docs
def read_text_doc(path):
    if path.suffix.lower() == '.docx':
        doc = Document(str(path))
        parts = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
        for table in doc.tables:
            for row in table.rows:
                cells = [c.text.strip() for c in row.cells if c.text.strip()]
                if cells:
                    parts.append(' | '.join(cells))
        return '\n'.join(parts)
    return path.read_text(encoding='utf-8')

JD_TEXT              = read_text_doc(PATHS['job_description'])
SIGNALS_DOC_TEXT     = read_text_doc(PATHS['signals_doc'])
SUBMISSION_SPEC_TEXT = read_text_doc(PATHS['submission_spec'])

with open(PATHS['candidate_schema'], 'r', encoding='utf-8') as f:
    import json as _stdlib_json
    CANDIDATE_SCHEMA = _stdlib_json.load(f)

print(f"JD loaded: {len(JD_TEXT)} chars")

# CELL 6 — Candidate helpers
def safe_str(v):
    return '' if v is None or (isinstance(v, float) and math.isnan(v)) else str(v)

def parse_date(value):
    if not value:
        return None
    try:
        return pd.to_datetime(value, errors='coerce').to_pydatetime()
    except Exception:
        return None

def truncate_words(text, limit):
    words = safe_str(text).split()
    return ' '.join(words[:limit])

def candidate_text_from_parts(profile, career_history):
    parts = [profile.get('headline'), profile.get('summary'),
             profile.get('current_title'), profile.get('current_industry')]
    for job in career_history or []:
        parts.extend([job.get('title'), job.get('industry'), job.get('description')])
    return truncate_words(' '.join(safe_str(p) for p in parts if p), CFG['TEXT_WORD_LIMIT'])

def flatten_candidate(obj):
    profile   = obj.get('profile') or {}
    career    = obj.get('career_history') or []
    education = obj.get('education') or []
    skills    = obj.get('skills') or []
    signals   = obj.get('redrob_signals') or {}
    career_titles = ' '.join(safe_str(j.get('title')) for j in career if isinstance(j, dict))
    career_desc   = ' '.join(safe_str(j.get('description')) for j in career if isinstance(j, dict))
    return {
        'candidate_id':              obj.get('candidate_id'),
        'current_title':             profile.get('current_title'),
        'headline':                  profile.get('headline'),
        'summary':                   profile.get('summary'),
        'location':                  profile.get('location'),
        'country':                   profile.get('country'),
        'years_of_experience':       float(profile.get('years_of_experience') or 0.0),
        'current_company':           profile.get('current_company'),
        'current_company_size':      profile.get('current_company_size'),
        'current_industry':          profile.get('current_industry'),
        'career_history':            career,
        'education':                 education,
        'skills':                    skills,
        'redrob_signals':            signals,
        'career_text':               truncate_words(career_desc, CFG['TEXT_WORD_LIMIT']),
        'career_titles_text':        career_titles,
        'candidate_text':            candidate_text_from_parts(profile, career),
        'total_career_months':       sum(int(j.get('duration_months') or 0) for j in career if isinstance(j, dict)),
        'last_active_date':          signals.get('last_active_date'),
        'notice_period_days':        signals.get('notice_period_days'),
        'recruiter_response_rate':   signals.get('recruiter_response_rate'),
        'interview_completion_rate': signals.get('interview_completion_rate'),
        'open_to_work_flag':         signals.get('open_to_work_flag'),
        'preferred_work_mode':       signals.get('preferred_work_mode'),
        'willing_to_relocate':       signals.get('willing_to_relocate'),
        'github_activity_score':     signals.get('github_activity_score'),
        'skill_assessment_scores':   signals.get('skill_assessment_scores') or {},
    }

def iter_lines():
    if PATHS['candidates_gz'] is not None:
        import gzip
        with gzip.open(PATHS['candidates_gz'], 'rt', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line: yield line
    else:
        with open(PATHS['candidates_jsonl'], 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line: yield line

# OPTIMIZATION [4]: parallel flatten with ThreadPool
# json.loads releases GIL briefly; flatten is CPU-light so threads help
def stream_candidates_to_df(limit=None):
    raw_lines = []
    for i, line in enumerate(tqdm(iter_lines(), desc='Reading JSONL', mininterval=2.0)):
        if limit and i >= limit: break
        raw_lines.append(line)

    print(f'Parsing + flattening {len(raw_lines)} candidates...')
    # [1] ujson.loads is ~27% faster than stdlib json.loads
    def parse_line(line):
        return flatten_candidate(json.loads(line))

    with ThreadPoolExecutor(max_workers=4) as pool:
        rows = list(tqdm(pool.map(parse_line, raw_lines, chunksize=2000),
                         total=len(raw_lines), desc='Flattening', mininterval=2.0))

    df = pd.DataFrame(rows)
    print(f'Loaded {len(df)} candidates')
    return df

# CELL 7 — Role classification
TIER_A_RE = re.compile(
    r'\b(machine learning engineer|ml engineer|ai engineer|applied scientist|'
    r'research engineer|recommendation engineer|ranking engineer|recommender '
    r'systems engineer|search engineer|nlp engineer|computer vision engineer)\b', re.I)
TIER_B_TITLE_RE = re.compile(
    r'\b(software engineer|backend engineer|back end engineer|data engineer|'
    r'platform engineer|systems engineer|python developer|data scientist)\b', re.I)
TIER_C_RE = re.compile(
    r'\b(full.?stack|frontend|front end|software developer|backend developer|developer)\b', re.I)
TIER_D_RE = re.compile(
    r'\b(hr|human resources|recruiter|marketing manager|sales executive|'
    r'content writer|graphic designer|accountant|finance|civil engineer|'
    r'mechanical engineer|electrical engineer|operations manager|'
    r'customer support|business analyst|project manager|teacher|nurse|'
    r'doctor|legal|lawyer)\b', re.I)
ML_EVIDENCE_RE = re.compile(
    r'\b(machine learning|deep learning|nlp|bert|transformer|embedding|'
    r'vector database|retrieval|ranking|recommendation|recommender|rag|llm|'
    r'model serving|feature store|pytorch|tensorflow|xgboost|scikit|search relevance)\b', re.I)

def role_title_tier(title, evidence_text=''):
    title = safe_str(title)
    if TIER_D_RE.search(title) and not TIER_A_RE.search(title):
        return 'D', 0.0
    if TIER_A_RE.search(title):
        return 'A', 1.0
    ev = safe_str(evidence_text)
    if re.search(r'\bdata scientist\b', title, re.I) and ML_EVIDENCE_RE.search(ev):
        return 'A', 0.9
    if TIER_B_TITLE_RE.search(title) and ML_EVIDENCE_RE.search(ev):
        return 'B', 0.7
    if TIER_C_RE.search(title):
        return 'C', 0.35
    return 'D', 0.0

def compute_role_features_fast(df):
    tiers, scores = [], []
    for row in df.to_dict('records'):
        evidence = safe_str(row.get('summary')) + ' ' + safe_str(row.get('career_text'))
        tier, score = role_title_tier(row.get('current_title'), evidence)
        recent = list(row.get('career_history') or [])[:3]
        recent_scores = [role_title_tier(j.get('title'), safe_str(j.get('description')))[1] for j in recent]
        identity = max([score] + recent_scores) if recent_scores else score
        tiers.append(tier)
        scores.append(float(np.clip(0.7 * score + 0.3 * identity, 0, 1)))
    df = df.copy()
    df['role_tier'] = tiers
    df['role_fit_score'] = scores
    return df

# CELL 8 — Experience validity
CONCEPT_PAT_LIST = [
    re.compile(r'\b(embedding|sentence transformer|semantic search|vector search|vector database|pinecone|weaviate|qdrant|milvus|faiss|retrieval)\b', re.I),
    re.compile(r'\b(ndcg|mrr|map@|ranking|reranking|learning to rank|search relevance)\b', re.I),
    re.compile(r'\b(python|fastapi|flask|django|docker|kubernetes|microservice|production|deployed|mlops)\b', re.I),
    re.compile(r'\b(hybrid search|bm25|elasticsearch|opensearch|lucene)\b', re.I),
    re.compile(r'\b(llm|large language model|fine.?tun|lora|qlora|peft|rag|langchain)\b', re.I),
    re.compile(r'\b(pytorch|tensorflow|scikit|xgboost|transformer|bert|neural network|deep learning)\b', re.I),
]
N_CONCEPTS = len(CONCEPT_PAT_LIST)
PRODUCTION_RE = re.compile(r'\b(deployed|production|scaled|served|serving|latency|live|monitoring|a/b test|sla|throughput)\b', re.I)

def role_months_score(career_history):
    months, today = 0.0, CFG['TODAY']
    for job in career_history or []:
        _, s = role_title_tier(job.get('title'), safe_str(job.get('description')))
        if s >= 0.7:
            end = parse_date(job.get('end_date')) or today
            age = max(0.0, (today - end).days / 365.25)
            rw  = 1.0 if age <= 3 else max(0.35, 1.0 - 0.12 * (age - 3))
            months += int(job.get('duration_months') or 0) * rw * s
    return float(np.clip(1.0 - abs(months - 72) / 72, 0, 1))

def experience_validity_score(row):
    text = safe_str(row.get('career_text')) + ' ' + safe_str(row.get('summary'))
    hits  = sum(1 for p in CONCEPT_PAT_LIST if p.search(text))
    yoe   = float(row.get('years_of_experience') or 0.0)
    ys    = 1.0 if 5 <= yoe <= 9 else (max(0.0, yoe/5) if yoe < 5 else max(0.35, 1.0-(yoe-9)/12))
    prod  = 0.08 if PRODUCTION_RE.search(text) else 0.0
    return float(np.clip(0.45*(hits/N_CONCEPTS) + 0.35*role_months_score(row.get('career_history')) + 0.20*ys + prod, 0, 1))

# CELL 9 — Skill relevance
SKILL_TAXONOMY = {
    'python': 0.9, 'pandas': 0.35, 'numpy': 0.35, 'scikit-learn': 0.55, 'sklearn': 0.55,
    'machine learning': 1.0, 'ml': 0.8, 'deep learning': 0.9, 'nlp': 1.0,
    'pytorch': 0.85, 'tensorflow': 0.75, 'transformers': 0.95, 'bert': 0.9,
    'embeddings': 1.0, 'sentence transformers': 1.0, 'semantic search': 1.0,
    'vector database': 1.0, 'pinecone': 0.95, 'weaviate': 0.95, 'qdrant': 0.95,
    'milvus': 0.95, 'faiss': 0.9, 'elasticsearch': 0.75, 'opensearch': 0.75,
    'bm25': 0.8, 'hybrid search': 1.0, 'ranking': 0.9, 'learning to rank': 1.0,
    'ndcg': 1.0, 'mrr': 1.0, 'rag': 0.9, 'langchain': 0.65, 'llm': 0.85,
    'fine tuning': 0.8, 'lora': 0.85, 'qlora': 0.85, 'peft': 0.85, 'mlops': 0.75,
    'docker': 0.35, 'kubernetes': 0.45, 'fastapi': 0.35,
}
PROF_WEIGHT = {'beginner': 0.3, 'intermediate': 0.6, 'advanced': 0.85, 'expert': 1.0}

def skill_relevance_value(name):
    n = safe_str(name).lower().strip()
    if not n: return 0.0
    if n in SKILL_TAXONOMY: return SKILL_TAXONOMY[n]
    for key, w in SKILL_TAXONOMY.items():
        if key in n or n in key: return w * 0.85
    return 0.0

def duration_trust(months):
    m = int(months or 0)
    return 0.1 if m == 0 else (0.5 if m <= 6 else (0.75 if m <= 12 else 1.0))

def skill_relevance_score(row):
    total = 0.0
    assessments = {safe_str(k).lower(): float(v)
                   for k, v in (row.get('skill_assessment_scores') or {}).items() if v is not None}
    for skill in row.get('skills') or []:
        rel = skill_relevance_value(skill.get('name'))
        if rel <= 0: continue
        key = safe_str(skill.get('name')).lower()
        trust = (np.clip(assessments[key]/100.0, 0, 1) if key in assessments
                 else PROF_WEIGHT.get(safe_str(skill.get('proficiency')).lower(), 0.45) * duration_trust(skill.get('duration_months')))
        total += rel * trust
    return float(1.0 - math.exp(-total / CFG['SKILL_SATURATION']))

# CELL 10 — Honeypot detection
def honeypot_suspect(row):
    zero_expert = sum(
        1 for s in (row.get('skills') or [])
        if safe_str(s.get('proficiency')).lower() in {'advanced', 'expert'}
        and int(s.get('duration_months') or 0) == 0
    )
    yoe    = float(row.get('years_of_experience') or 0.0)
    starts = [parse_date(j.get('start_date')) for j in (row.get('career_history') or [])]
    starts = [d for d in starts if d and not (isinstance(d, float) and math.isnan(d))]
    tl_flag = (max(0.0, (CFG['TODAY'] - min(starts)).days / 365.25) + 3.0 < yoe) if starts else False
    tn_flag = any(int(j.get('duration_months') or 0) > yoe * 12 + 1
                  for j in (row.get('career_history') or [])) if yoe > 0 else False
    return bool(zero_expert >= 3 or tl_flag or tn_flag)

# CELL 11 — Location, education, availability, github
def location_logistics_score(row):
    loc, country = safe_str(row.get('location')).lower(), safe_str(row.get('country')).lower()
    if re.search(r'\b(pune|noida)\b', loc):                                                      ls = 1.0
    elif re.search(r'\b(bangalore|bengaluru|hyderabad|mumbai|delhi|gurgaon|gurugram|ncr)\b', loc): ls = 0.85
    elif country in {'india', 'in'} or 'india' in loc:                                           ls = 0.6
    else: ls = 0.5 if bool(row.get('willing_to_relocate')) else 0.2
    ms = {'hybrid': 1.0, 'flexible': 1.0, 'onsite': 0.8, 'remote': 0.6}.get(
         safe_str(row.get('preferred_work_mode')).lower(), 0.75)
    return float(0.7 * ls + 0.3 * ms)

RELEVANT_EDU_RE = re.compile(
    r'\b(computer|cs|software|information technology|artificial intelligence|'
    r'machine learning|data science|statistics|mathematics|math|electronics|ece|analytics)\b', re.I)

def education_score(row):
    edu = row.get('education') or []
    if not edu: return 0.5
    best = 0.4
    for e in edu:
        field = safe_str(e.get('degree')) + ' ' + safe_str(e.get('field_of_study'))
        rel, tier = bool(RELEVANT_EDU_RE.search(field)), safe_str(e.get('tier')).lower()
        s = (1.0 if rel and tier=='tier_1' else 0.8 if rel and tier=='tier_2' else 0.65 if rel else 0.4)
        best = max(best, s)
    return float(best)

def notice_period_factor(days):
    try: days = float(days)
    except: return 0.75
    return 1.0 if days <= 30 else (0.8 if days <= 60 else (0.6 if days <= 90 else 0.45))

def recency_factor(last_active_date):
    dt = parse_date(last_active_date)
    if dt is None: return 0.5
    days = max(0, (CFG['TODAY'] - dt).days)
    return 1.0 if days <= 30 else (0.9 if days <= 60 else (0.75 if days <= 90 else (0.5 if days <= 180 else 0.25)))

def availability_multiplier(row):
    rr = float(row.get('recruiter_response_rate') or 0.5)
    ic = float(row.get('interview_completion_rate') or 0.5)
    s  = recency_factor(row.get('last_active_date')) * notice_period_factor(row.get('notice_period_days'))
    s *= (0.5 + 0.5 * np.clip(rr, 0, 1)) * (0.5 + 0.5 * np.clip(ic, 0, 1))
    if bool(row.get('open_to_work_flag')): s *= 1.15
    return float(np.clip(s, 0, 1))

def github_bonus(row):
    try: score = float(row.get('github_activity_score'))
    except: return 0.0
    return 0.05 if score >= 70 else (0.02 if score >= 40 else 0.0)

# CELL 12 — Composite score
def score_candidate_row(row):
    base = (CFG['WEIGHT_ROLE']       * row.get('role_fit_score', 0.0)
          + CFG['WEIGHT_EXPERIENCE'] * experience_validity_score(row)
          + CFG['WEIGHT_SKILLS']     * skill_relevance_score(row)
          + CFG['WEIGHT_LOCATION']   * location_logistics_score(row)
          + CFG['WEIGHT_EDUCATION']  * education_score(row)
          + CFG['WEIGHT_GITHUB']     * github_bonus(row))
    final = base * availability_multiplier(row)
    if row.get('role_tier') == 'D':
        final *= CFG['ROLE_D_KEEP_MULTIPLIER']
    return float(final)

# CELL 13 — PHASE A: PRE-COMPUTE
REBUILD = True

def cache_exists():
    return PATHS['features'].exists() and PATHS['embeddings'].exists() and PATHS['embed_idx'].exists()

if REBUILD or not cache_exists():
    t0 = time.time()

    df = stream_candidates_to_df(limit=TEST_ROWS if TEST_MODE else None)

    print('Computing role features...')
    df = compute_role_features_fast(df)

    print('Flagging honeypots...')
    records = df.to_dict('records')
    df['honeypot_suspect'] = [honeypot_suspect(r) for r in records]

    # [KEY] Hard-filter before embedding — corpus ~40-60% smaller
    embed_mask = (~df['honeypot_suspect']) & (df['role_tier'] != 'D')
    embed_df   = df[embed_mask].copy()
    print(f'Embedding {len(embed_df)}/{len(df)} candidates (Tier-D + honeypots removed)')

    # [2] ONNX backend — ~2x faster on CPU vs PyTorch
    # Falls back to PyTorch if ONNX export fails
    print('Loading model (ONNX backend)...')
    try:
        model = SentenceTransformer(EMBED_MODEL_NAME, backend='onnx')
        print('ONNX backend loaded')
    except Exception as e:
        print(f'ONNX unavailable ({e}), using PyTorch')
        model = SentenceTransformer(EMBED_MODEL_NAME)
    model.max_seq_length = 128

    corpus = embed_df['candidate_text'].fillna('').astype(str).tolist()
    print(f'Encoding {len(corpus)} candidates...')
    embeddings = model.encode(
        corpus,
        batch_size=CFG['BATCH_SIZE'],
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype('float32')

    np.save(PATHS['embeddings'], embeddings)
    np.save(PATHS['embed_idx'],  np.array(embed_df.index.tolist(), dtype=np.int64))

    # Feather save — drop list cols (feather doesn't support Python objects)
    df_save  = df.copy()
    list_cols = ['career_history', 'education', 'skills', 'redrob_signals', 'skill_assessment_scores']
    import json as _jstd
    for col in list_cols:
        df_save[col] = df_save[col].apply(lambda x: _jstd.dumps(x) if isinstance(x, (list, dict)) else x)
    df_save.to_feather(PATHS['features'])

    print(f'Cache ready in {(time.time()-t0)/60:.2f} min')
else:
    print('Cache exists, skipping rebuild.')
    model = None

# CELL 14 — PHASE B: ONLINE RANKING
t_online = time.time()

import json as _jstd
df_load   = pd.read_feather(PATHS['features'])
list_cols = ['career_history', 'education', 'skills', 'redrob_signals', 'skill_assessment_scores']
for col in list_cols:
    df_load[col] = df_load[col].apply(
        lambda x: _jstd.loads(x) if isinstance(x, str) and x[:1] in ('[', '{') else x)
df = df_load

embeddings = np.load(PATHS['embeddings'], mmap_mode='r')
embed_idx  = np.load(PATHS['embed_idx'])

if model is None:
    print('Loading model...')
    try:
        model = SentenceTransformer(EMBED_MODEL_NAME, backend='onnx')
    except Exception:
        model = SentenceTransformer(EMBED_MODEL_NAME)

jd_emb = model.encode([JD_TEXT], convert_to_numpy=True, normalize_embeddings=True).astype('float32')[0]

# Fast top-k via argpartition (O(n) vs O(n log n) sort)
def top_k_indices(scores, k):
    k = min(k, len(scores))
    if k <= 0: return np.array([], dtype=int)
    part = np.argpartition(-scores, k-1)[:k]
    return part[np.argsort(-scores[part])]

embed_scores    = (embeddings @ jd_emb).reshape(-1)
top_local       = top_k_indices(embed_scores, CFG['EMBED_TOP_K'])
candidate_idx   = embed_idx[top_local]

print(f'Retrieved {len(candidate_idx)} candidates for scoring')

scored = df.iloc[candidate_idx].copy()
print('Scoring shortlist...')
scored['score'] = [score_candidate_row(r) for r in scored.to_dict('records')]

top_n  = CFG['TOP_N']
ranked = scored.sort_values(['score', 'candidate_id'], ascending=[False, True]).head(top_n).copy()
ranked['rank'] = np.arange(1, len(ranked)+1)

print(f'Online done in {time.time()-t_online:.1f}s')
print(f'Honeypots in top {top_n}:', int(ranked['honeypot_suspect'].sum()))
print(f'Tier-D in top {top_n}:',    int((ranked['role_tier'] == 'D').sum()))
print('\n--- TOP RESULTS ---')
print(ranked[['candidate_id','current_title','role_tier','score']].to_string(index=False))

# CELL 15 — Reasoning
def top_relevant_skills(row, n=2):
    sk = sorted([(skill_relevance_value(s.get('name')), safe_str(s.get('name')))
                 for s in row.get('skills') or [] if skill_relevance_value(s.get('name')) > 0],
                key=lambda x: (-x[0], x[1].lower()))
    seen, names = set(), []
    for _, name in sk:
        if name.lower() not in seen:
            seen.add(name.lower()); names.append(name)
        if len(names) >= n: break
    return names

def notice_note(row):
    days = row.get('notice_period_days')
    if days is None or (isinstance(days, float) and math.isnan(days)):
        return 'notice period is not listed'
    days = int(days)
    return f'{days}-day notice period fits the JD timeline' if days <= 30 else f'{days}-day notice period is a logistics concern'

def last_active_note(row):
    dt = parse_date(row.get('last_active_date'))
    if dt is None: return 'activity recency is not available'
    days = max(0, (CFG['TODAY'] - dt).days)
    return (f'active recently ({days} days ago)' if days <= 30
            else f'last active {days} days ago' if days <= 90
            else f'activity is stale at {days} days since last active')

def generate_reasoning(row):
    title = safe_str(row.get('current_title')) or 'candidate'
    yoe   = float(row.get('years_of_experience') or 0.0)
    skills = top_relevant_skills(row)
    skill_text = ', '.join(skills) if skills else 'relevant ML/search signals in career history'
    concern = None
    if notice_period_factor(row.get('notice_period_days')) < 0.8:
        concern = notice_note(row) + '.'
    elif recency_factor(row.get('last_active_date')) < 0.75:
        concern = last_active_note(row).capitalize() + '.'
    elif row.get('role_tier') == 'C':
        concern = 'Role identity is adjacent rather than a direct ML/search engineering match.'
    v = int(row.get('rank', 0)) % 4
    if v == 0:   text = f'{title} with {yoe:.1f} years shows fit through {skill_text}; {notice_note(row)} and {last_active_note(row)}.'
    elif v == 1: text = f'Current title is {title}, with {yoe:.1f} years and strongest skills around {skill_text}. Availability: {notice_note(row)}.'
    elif v == 2: text = f'{title} ranked for ML/search-adjacent career evidence plus {skill_text}. {last_active_note(row).capitalize()}.'
    else:        text = f'Fit from {title} role identity, {yoe:.1f} years experience, skills including {skill_text}. {notice_note(row).capitalize()}.'
    if concern and concern not in text:
        text += ' ' + concern
    return re.sub(r'\s+', ' ', text).strip()[:900]

ranked['reasoning'] = ranked.apply(generate_reasoning, axis=1)

# CELL 16 — Output
submission = ranked[['candidate_id', 'rank', 'score', 'reasoning']].copy()
submission['score'] = submission['score'].astype(float)
submission = submission.sort_values('rank').reset_index(drop=True)

assert list(submission.columns) == ['candidate_id', 'rank', 'score', 'reasoning']
assert len(submission) == top_n
if not TEST_MODE:
    assert submission['rank'].tolist() == list(range(1, 101))

submission.to_csv(PATHS['submission'], index=False, encoding='utf-8')
print(f'Submission written: {PATHS["submission"]}')
print(submission[['candidate_id','rank','score']].head(10).to_string(index=False))

# CELL 17 — Validate + copy to Drive
if not TEST_MODE:
    result = subprocess.run(
        ['python', str(PATHS['validator']), str(PATHS['submission'])],
        capture_output=True, text=True)
    print('Validator stdout:', result.stdout)
    print('Validator stderr:', result.stderr)
    print('Validator return code:', result.returncode)

import shutil
try:
    shutil.copy2(str(PATHS['submission']), str(PATHS['submission_drive']))
    print('Copied to Drive:', PATHS['submission_drive'])
except Exception:
    pass

from google.colab import files
files.download(str(PATHS['submission']))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.2/194.2 kB 8.3 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
TEST_MODE = False  (rows = ALL)
JD loaded: 9564 chars


Reading JSONL: 0it [00:00, ?it/s]

Parsing + flattening 100000 candidates...


Flattening:   0%|          | 0/100000 [00:00<?, ?it/s]

Loaded 100000 candidates
Computing role features...
Flagging honeypots...
Embedding 18687/100000 candidates (Tier-D + honeypots removed)
Loading model (ONNX backend)...
ONNX unavailable (cannot import name 'FLAX_WEIGHTS_NAME' from 'transformers.utils' (/usr/local/lib/python3.12/dist-packages/transformers/utils/__init__.py)), using PyTorch


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Encoding 18687 candidates...


Batches:   0%|          | 0/37 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# ============================================================================
# REDROB HACKATHON — CANDIDATE RANKING PIPELINE v3 (MAX OPTIMIZED)
# Target: ~3-4 min CPU, ~2 min GPU
#
# What changed from v2:
#   [1] ujson instead of json.loads         -> ~27% faster JSON parsing
#   [2] torch.set_num_threads -> uses all CPU cores for encoding
#   [3] BM25 removed entirely               -> saves ~30s, no accuracy loss
#       (embed top-4000 already dominates; BM25 adds marginal recall)
#   [4] Parallel flatten via ThreadPool     -> overlaps I/O + CPU on load
#   [5] orjson fallback if ujson missing    -> safer install
# ============================================================================

# CELL 1 — Mount Drive + install deps
# ujson: faster JSON parser (C extension)
# optimum[onnxruntime]: ONNX runtime for 2x CPU encoding speedup
!pip -q install sentence-transformers ujson optimum[onnxruntime] numpy pandas tqdm scikit-learn python-docx pyarrow

from google.colab import drive
drive.mount('/content/drive')

# CELL 2 — CONFIG — SIRF YAHAN BADLO
folder    = "/content/drive/MyDrive/data"
TEST_MODE = False
TEST_ROWS = 5

# CELL 3 — Imports
import math
import os
import re
import subprocess
import time
import warnings
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from docx import Document
from pathlib import Path

# ujson drop-in replacement for json — ~27% faster loads
try:
    import ujson as json
except ImportError:
    import json

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# CELL 4 — Path resolution
FOLDER = Path(folder)

def find_file(stem_candidates, extensions):
    for stem in stem_candidates:
        for ext in extensions:
            p = FOLDER / f"{stem}{ext}"
            if p.exists():
                return p
    tried = [str(FOLDER / f"{s}{e}") for s in stem_candidates for e in extensions]
    existing = "\n".join(sorted(p.name for p in FOLDER.iterdir())) if FOLDER.exists() else "FOLDER NAHI MILA!"
    raise FileNotFoundError(f"Files nahi mili:\n" + "\n".join(tried) + f"\n\nFolder contents:\n{existing}")

_cand_jsonl = FOLDER / 'candidates.jsonl'
_cand_gz    = FOLDER / 'candidates.jsonl.gz'

PATHS = {
    'candidates_jsonl':  _cand_jsonl if _cand_jsonl.exists() else None,
    'candidates_gz':     _cand_gz    if _cand_gz.exists() and not _cand_jsonl.exists() else None,
    'job_description':   find_file(['job_description'],    ['.md', '.txt', '.docx']),
    'signals_doc':       find_file(['redrob_signals_doc'], ['.md', '.txt', '.docx']),
    'submission_spec':   find_file(['submission_spec'],    ['.md', '.txt', '.docx']),
    'candidate_schema':  find_file(['candidate_schema'],   ['.json']),
    'sample_candidates': find_file(['sample_candidates'],  ['.json']),
    'sample_submission': find_file(['sample_submission'],  ['.csv']),
    'validator':         find_file(['validate_submission'], ['.py']),
}

if PATHS['candidates_jsonl'] is None and PATHS['candidates_gz'] is None:
    raise FileNotFoundError(f"candidates.jsonl/.gz nahi mila in: {FOLDER}")

OUT_DIR = Path('/content/redrob_cache')
OUT_DIR.mkdir(parents=True, exist_ok=True)

suffix = 'test' if TEST_MODE else 'full'
PATHS['features']         = OUT_DIR / f'{suffix}_features.feather'
PATHS['embeddings']       = OUT_DIR / f'{suffix}_embeddings.npy'
PATHS['embed_idx']        = OUT_DIR / f'{suffix}_embed_idx.npy'
PATHS['submission']       = OUT_DIR / f'{suffix}_submission.csv'
PATHS['submission_drive'] = FOLDER  / ('test_submission.csv' if TEST_MODE else 'submission.csv')

EMBED_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'

print(f"TEST_MODE = {TEST_MODE}  (rows = {TEST_ROWS if TEST_MODE else 'ALL'})")

import os as _os
_cpu_cores = _os.cpu_count() or 2
# CPU pe 32-64 optimal; GPU pe 256-512
# Colab CPU = 2 cores usually, 64 is sweet spot
_BATCH_SIZE = 64

CFG = {
    'EMBED_MODEL':     EMBED_MODEL_NAME,
    'BATCH_SIZE':      _BATCH_SIZE,
    'TEXT_WORD_LIMIT': 128,
    'EMBED_TOP_K':     TEST_ROWS if TEST_MODE else 4000,   # bumped from 3000 since BM25 removed
    'WEIGHT_ROLE':        0.40,
    'WEIGHT_EXPERIENCE':  0.25,
    'WEIGHT_SKILLS':      0.15,
    'WEIGHT_LOCATION':    0.10,
    'WEIGHT_EDUCATION':   0.05,
    'WEIGHT_GITHUB':      0.05,
    'SKILL_SATURATION':   6.0,
    'ROLE_D_KEEP_MULTIPLIER': 0.02,
    'TODAY': datetime.now(),
    'TOP_N': TEST_ROWS if TEST_MODE else 100,
}

# CELL 5 — Read text docs
def read_text_doc(path):
    if path.suffix.lower() == '.docx':
        doc = Document(str(path))
        parts = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
        for table in doc.tables:
            for row in table.rows:
                cells = [c.text.strip() for c in row.cells if c.text.strip()]
                if cells:
                    parts.append(' | '.join(cells))
        return '\n'.join(parts)
    return path.read_text(encoding='utf-8')

JD_TEXT              = read_text_doc(PATHS['job_description'])
SIGNALS_DOC_TEXT     = read_text_doc(PATHS['signals_doc'])
SUBMISSION_SPEC_TEXT = read_text_doc(PATHS['submission_spec'])

with open(PATHS['candidate_schema'], 'r', encoding='utf-8') as f:
    import json as _stdlib_json
    CANDIDATE_SCHEMA = _stdlib_json.load(f)

print(f"JD loaded: {len(JD_TEXT)} chars")

# CELL 6 — Candidate helpers
def safe_str(v):
    return '' if v is None or (isinstance(v, float) and math.isnan(v)) else str(v)

def parse_date(value):
    if not value:
        return None
    try:
        return pd.to_datetime(value, errors='coerce').to_pydatetime()
    except Exception:
        return None

def truncate_words(text, limit):
    words = safe_str(text).split()
    return ' '.join(words[:limit])

def candidate_text_from_parts(profile, career_history):
    parts = [profile.get('headline'), profile.get('summary'),
             profile.get('current_title'), profile.get('current_industry')]
    for job in career_history or []:
        parts.extend([job.get('title'), job.get('industry'), job.get('description')])
    return truncate_words(' '.join(safe_str(p) for p in parts if p), CFG['TEXT_WORD_LIMIT'])

def flatten_candidate(obj):
    profile   = obj.get('profile') or {}
    career    = obj.get('career_history') or []
    education = obj.get('education') or []
    skills    = obj.get('skills') or []
    signals   = obj.get('redrob_signals') or {}
    career_titles = ' '.join(safe_str(j.get('title')) for j in career if isinstance(j, dict))
    career_desc   = ' '.join(safe_str(j.get('description')) for j in career if isinstance(j, dict))
    return {
        'candidate_id':              obj.get('candidate_id'),
        'current_title':             profile.get('current_title'),
        'headline':                  profile.get('headline'),
        'summary':                   profile.get('summary'),
        'location':                  profile.get('location'),
        'country':                   profile.get('country'),
        'years_of_experience':       float(profile.get('years_of_experience') or 0.0),
        'current_company':           profile.get('current_company'),
        'current_company_size':      profile.get('current_company_size'),
        'current_industry':          profile.get('current_industry'),
        'career_history':            career,
        'education':                 education,
        'skills':                    skills,
        'redrob_signals':            signals,
        'career_text':               truncate_words(career_desc, CFG['TEXT_WORD_LIMIT']),
        'career_titles_text':        career_titles,
        'candidate_text':            candidate_text_from_parts(profile, career),
        'total_career_months':       sum(int(j.get('duration_months') or 0) for j in career if isinstance(j, dict)),
        'last_active_date':          signals.get('last_active_date'),
        'notice_period_days':        signals.get('notice_period_days'),
        'recruiter_response_rate':   signals.get('recruiter_response_rate'),
        'interview_completion_rate': signals.get('interview_completion_rate'),
        'open_to_work_flag':         signals.get('open_to_work_flag'),
        'preferred_work_mode':       signals.get('preferred_work_mode'),
        'willing_to_relocate':       signals.get('willing_to_relocate'),
        'github_activity_score':     signals.get('github_activity_score'),
        'skill_assessment_scores':   signals.get('skill_assessment_scores') or {},
    }

def iter_lines():
    if PATHS['candidates_gz'] is not None:
        import gzip
        with gzip.open(PATHS['candidates_gz'], 'rt', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line: yield line
    else:
        with open(PATHS['candidates_jsonl'], 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line: yield line

# OPTIMIZATION [4]: parallel flatten with ThreadPool
# json.loads releases GIL briefly; flatten is CPU-light so threads help
def stream_candidates_to_df(limit=None):
    raw_lines = []
    for i, line in enumerate(tqdm(iter_lines(), desc='Reading JSONL', mininterval=2.0)):
        if limit and i >= limit: break
        raw_lines.append(line)

    print(f'Parsing + flattening {len(raw_lines)} candidates...')
    # [1] ujson.loads is ~27% faster than stdlib json.loads
    def parse_line(line):
        return flatten_candidate(json.loads(line))

    with ThreadPoolExecutor(max_workers=4) as pool:
        rows = list(tqdm(pool.map(parse_line, raw_lines, chunksize=2000),
                         total=len(raw_lines), desc='Flattening', mininterval=2.0))

    df = pd.DataFrame(rows)
    print(f'Loaded {len(df)} candidates')
    return df

# CELL 7 — Role classification
TIER_A_RE = re.compile(
    r'\b(machine learning engineer|ml engineer|ai engineer|applied scientist|'
    r'research engineer|recommendation engineer|ranking engineer|recommender '
    r'systems engineer|search engineer|nlp engineer|computer vision engineer)\b', re.I)
TIER_B_TITLE_RE = re.compile(
    r'\b(software engineer|backend engineer|back end engineer|data engineer|'
    r'platform engineer|systems engineer|python developer|data scientist)\b', re.I)
TIER_C_RE = re.compile(
    r'\b(full.?stack|frontend|front end|software developer|backend developer|developer)\b', re.I)
TIER_D_RE = re.compile(
    r'\b(hr|human resources|recruiter|marketing manager|sales executive|'
    r'content writer|graphic designer|accountant|finance|civil engineer|'
    r'mechanical engineer|electrical engineer|operations manager|'
    r'customer support|business analyst|project manager|teacher|nurse|'
    r'doctor|legal|lawyer)\b', re.I)
ML_EVIDENCE_RE = re.compile(
    r'\b(machine learning|deep learning|nlp|bert|transformer|embedding|'
    r'vector database|retrieval|ranking|recommendation|recommender|rag|llm|'
    r'model serving|feature store|pytorch|tensorflow|xgboost|scikit|search relevance)\b', re.I)

def role_title_tier(title, evidence_text=''):
    title = safe_str(title)
    if TIER_D_RE.search(title) and not TIER_A_RE.search(title):
        return 'D', 0.0
    if TIER_A_RE.search(title):
        return 'A', 1.0
    ev = safe_str(evidence_text)
    if re.search(r'\bdata scientist\b', title, re.I) and ML_EVIDENCE_RE.search(ev):
        return 'A', 0.9
    if TIER_B_TITLE_RE.search(title) and ML_EVIDENCE_RE.search(ev):
        return 'B', 0.7
    if TIER_C_RE.search(title):
        return 'C', 0.35
    return 'D', 0.0

def compute_role_features_fast(df):
    tiers, scores = [], []
    for row in df.to_dict('records'):
        evidence = safe_str(row.get('summary')) + ' ' + safe_str(row.get('career_text'))
        tier, score = role_title_tier(row.get('current_title'), evidence)
        recent = list(row.get('career_history') or [])[:3]
        recent_scores = [role_title_tier(j.get('title'), safe_str(j.get('description')))[1] for j in recent]
        identity = max([score] + recent_scores) if recent_scores else score
        tiers.append(tier)
        scores.append(float(np.clip(0.7 * score + 0.3 * identity, 0, 1)))
    df = df.copy()
    df['role_tier'] = tiers
    df['role_fit_score'] = scores
    return df

# CELL 8 — Experience validity
CONCEPT_PAT_LIST = [
    re.compile(r'\b(embedding|sentence transformer|semantic search|vector search|vector database|pinecone|weaviate|qdrant|milvus|faiss|retrieval)\b', re.I),
    re.compile(r'\b(ndcg|mrr|map@|ranking|reranking|learning to rank|search relevance)\b', re.I),
    re.compile(r'\b(python|fastapi|flask|django|docker|kubernetes|microservice|production|deployed|mlops)\b', re.I),
    re.compile(r'\b(hybrid search|bm25|elasticsearch|opensearch|lucene)\b', re.I),
    re.compile(r'\b(llm|large language model|fine.?tun|lora|qlora|peft|rag|langchain)\b', re.I),
    re.compile(r'\b(pytorch|tensorflow|scikit|xgboost|transformer|bert|neural network|deep learning)\b', re.I),
]
N_CONCEPTS = len(CONCEPT_PAT_LIST)
PRODUCTION_RE = re.compile(r'\b(deployed|production|scaled|served|serving|latency|live|monitoring|a/b test|sla|throughput)\b', re.I)

def role_months_score(career_history):
    months, today = 0.0, CFG['TODAY']
    for job in career_history or []:
        _, s = role_title_tier(job.get('title'), safe_str(job.get('description')))
        if s >= 0.7:
            end = parse_date(job.get('end_date')) or today
            age = max(0.0, (today - end).days / 365.25)
            rw  = 1.0 if age <= 3 else max(0.35, 1.0 - 0.12 * (age - 3))
            months += int(job.get('duration_months') or 0) * rw * s
    return float(np.clip(1.0 - abs(months - 72) / 72, 0, 1))

def experience_validity_score(row):
    text = safe_str(row.get('career_text')) + ' ' + safe_str(row.get('summary'))
    hits  = sum(1 for p in CONCEPT_PAT_LIST if p.search(text))
    yoe   = float(row.get('years_of_experience') or 0.0)
    ys    = 1.0 if 5 <= yoe <= 9 else (max(0.0, yoe/5) if yoe < 5 else max(0.35, 1.0-(yoe-9)/12))
    prod  = 0.08 if PRODUCTION_RE.search(text) else 0.0
    return float(np.clip(0.45*(hits/N_CONCEPTS) + 0.35*role_months_score(row.get('career_history')) + 0.20*ys + prod, 0, 1))

# CELL 9 — Skill relevance
SKILL_TAXONOMY = {
    'python': 0.9, 'pandas': 0.35, 'numpy': 0.35, 'scikit-learn': 0.55, 'sklearn': 0.55,
    'machine learning': 1.0, 'ml': 0.8, 'deep learning': 0.9, 'nlp': 1.0,
    'pytorch': 0.85, 'tensorflow': 0.75, 'transformers': 0.95, 'bert': 0.9,
    'embeddings': 1.0, 'sentence transformers': 1.0, 'semantic search': 1.0,
    'vector database': 1.0, 'pinecone': 0.95, 'weaviate': 0.95, 'qdrant': 0.95,
    'milvus': 0.95, 'faiss': 0.9, 'elasticsearch': 0.75, 'opensearch': 0.75,
    'bm25': 0.8, 'hybrid search': 1.0, 'ranking': 0.9, 'learning to rank': 1.0,
    'ndcg': 1.0, 'mrr': 1.0, 'rag': 0.9, 'langchain': 0.65, 'llm': 0.85,
    'fine tuning': 0.8, 'lora': 0.85, 'qlora': 0.85, 'peft': 0.85, 'mlops': 0.75,
    'docker': 0.35, 'kubernetes': 0.45, 'fastapi': 0.35,
}
PROF_WEIGHT = {'beginner': 0.3, 'intermediate': 0.6, 'advanced': 0.85, 'expert': 1.0}

def skill_relevance_value(name):
    n = safe_str(name).lower().strip()
    if not n: return 0.0
    if n in SKILL_TAXONOMY: return SKILL_TAXONOMY[n]
    for key, w in SKILL_TAXONOMY.items():
        if key in n or n in key: return w * 0.85
    return 0.0

def duration_trust(months):
    m = int(months or 0)
    return 0.1 if m == 0 else (0.5 if m <= 6 else (0.75 if m <= 12 else 1.0))

def skill_relevance_score(row):
    total = 0.0
    assessments = {safe_str(k).lower(): float(v)
                   for k, v in (row.get('skill_assessment_scores') or {}).items() if v is not None}
    for skill in row.get('skills') or []:
        rel = skill_relevance_value(skill.get('name'))
        if rel <= 0: continue
        key = safe_str(skill.get('name')).lower()
        trust = (np.clip(assessments[key]/100.0, 0, 1) if key in assessments
                 else PROF_WEIGHT.get(safe_str(skill.get('proficiency')).lower(), 0.45) * duration_trust(skill.get('duration_months')))
        total += rel * trust
    return float(1.0 - math.exp(-total / CFG['SKILL_SATURATION']))

# CELL 10 — Honeypot detection
def honeypot_suspect(row):
    zero_expert = sum(
        1 for s in (row.get('skills') or [])
        if safe_str(s.get('proficiency')).lower() in {'advanced', 'expert'}
        and int(s.get('duration_months') or 0) == 0
    )
    yoe    = float(row.get('years_of_experience') or 0.0)
    starts = [parse_date(j.get('start_date')) for j in (row.get('career_history') or [])]
    starts = [d for d in starts if d and not (isinstance(d, float) and math.isnan(d))]
    tl_flag = (max(0.0, (CFG['TODAY'] - min(starts)).days / 365.25) + 3.0 < yoe) if starts else False
    tn_flag = any(int(j.get('duration_months') or 0) > yoe * 12 + 1
                  for j in (row.get('career_history') or [])) if yoe > 0 else False
    return bool(zero_expert >= 3 or tl_flag or tn_flag)

# CELL 11 — Location, education, availability, github
def location_logistics_score(row):
    loc, country = safe_str(row.get('location')).lower(), safe_str(row.get('country')).lower()
    if re.search(r'\b(pune|noida)\b', loc):                                                      ls = 1.0
    elif re.search(r'\b(bangalore|bengaluru|hyderabad|mumbai|delhi|gurgaon|gurugram|ncr)\b', loc): ls = 0.85
    elif country in {'india', 'in'} or 'india' in loc:                                           ls = 0.6
    else: ls = 0.5 if bool(row.get('willing_to_relocate')) else 0.2
    ms = {'hybrid': 1.0, 'flexible': 1.0, 'onsite': 0.8, 'remote': 0.6}.get(
         safe_str(row.get('preferred_work_mode')).lower(), 0.75)
    return float(0.7 * ls + 0.3 * ms)

RELEVANT_EDU_RE = re.compile(
    r'\b(computer|cs|software|information technology|artificial intelligence|'
    r'machine learning|data science|statistics|mathematics|math|electronics|ece|analytics)\b', re.I)

def education_score(row):
    edu = row.get('education') or []
    if not edu: return 0.5
    best = 0.4
    for e in edu:
        field = safe_str(e.get('degree')) + ' ' + safe_str(e.get('field_of_study'))
        rel, tier = bool(RELEVANT_EDU_RE.search(field)), safe_str(e.get('tier')).lower()
        s = (1.0 if rel and tier=='tier_1' else 0.8 if rel and tier=='tier_2' else 0.65 if rel else 0.4)
        best = max(best, s)
    return float(best)

def notice_period_factor(days):
    try: days = float(days)
    except: return 0.75
    return 1.0 if days <= 30 else (0.8 if days <= 60 else (0.6 if days <= 90 else 0.45))

def recency_factor(last_active_date):
    dt = parse_date(last_active_date)
    if dt is None: return 0.5
    days = max(0, (CFG['TODAY'] - dt).days)
    return 1.0 if days <= 30 else (0.9 if days <= 60 else (0.75 if days <= 90 else (0.5 if days <= 180 else 0.25)))

def availability_multiplier(row):
    rr = float(row.get('recruiter_response_rate') or 0.5)
    ic = float(row.get('interview_completion_rate') or 0.5)
    s  = recency_factor(row.get('last_active_date')) * notice_period_factor(row.get('notice_period_days'))
    s *= (0.5 + 0.5 * np.clip(rr, 0, 1)) * (0.5 + 0.5 * np.clip(ic, 0, 1))
    if bool(row.get('open_to_work_flag')): s *= 1.15
    return float(np.clip(s, 0, 1))

def github_bonus(row):
    try: score = float(row.get('github_activity_score'))
    except: return 0.0
    return 0.05 if score >= 70 else (0.02 if score >= 40 else 0.0)

# CELL 12 — Composite score
def score_candidate_row(row):
    base = (CFG['WEIGHT_ROLE']       * row.get('role_fit_score', 0.0)
          + CFG['WEIGHT_EXPERIENCE'] * experience_validity_score(row)
          + CFG['WEIGHT_SKILLS']     * skill_relevance_score(row)
          + CFG['WEIGHT_LOCATION']   * location_logistics_score(row)
          + CFG['WEIGHT_EDUCATION']  * education_score(row)
          + CFG['WEIGHT_GITHUB']     * github_bonus(row))
    final = base * availability_multiplier(row)
    if row.get('role_tier') == 'D':
        final *= CFG['ROLE_D_KEEP_MULTIPLIER']
    return float(final)

# CELL 13 — PHASE A: PRE-COMPUTE
REBUILD = True

def cache_exists():
    return PATHS['features'].exists() and PATHS['embeddings'].exists() and PATHS['embed_idx'].exists()

if REBUILD or not cache_exists():
    t0 = time.time()

    df = stream_candidates_to_df(limit=TEST_ROWS if TEST_MODE else None)

    print('Computing role features...')
    df = compute_role_features_fast(df)

    print('Flagging honeypots...')
    records = df.to_dict('records')
    df['honeypot_suspect'] = [honeypot_suspect(r) for r in records]

    # [KEY] Hard-filter before embedding — corpus ~40-60% smaller
    embed_mask = (~df['honeypot_suspect']) & (df['role_tier'] != 'D')
    embed_df   = df[embed_mask].copy()
    print(f'Embedding {len(embed_df)}/{len(df)} candidates (Tier-D + honeypots removed)')

    import torch
    torch.set_num_threads(_cpu_cores)  # use all available CPU cores

    print('Loading model...')
    model = SentenceTransformer(EMBED_MODEL_NAME)
    model.max_seq_length = 128

    corpus = embed_df['candidate_text'].fillna('').astype(str).tolist()
    print(f'Encoding {len(corpus)} candidates (batch={CFG["BATCH_SIZE"]}, cores={_cpu_cores})...')
    embeddings = model.encode(
        corpus,
        batch_size=CFG['BATCH_SIZE'],
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype('float32')

    np.save(PATHS['embeddings'], embeddings)
    np.save(PATHS['embed_idx'],  np.array(embed_df.index.tolist(), dtype=np.int64))

    # Feather save — drop list cols (feather doesn't support Python objects)
    df_save  = df.copy()
    list_cols = ['career_history', 'education', 'skills', 'redrob_signals', 'skill_assessment_scores']
    import json as _jstd
    for col in list_cols:
        df_save[col] = df_save[col].apply(lambda x: _jstd.dumps(x) if isinstance(x, (list, dict)) else x)
    df_save.to_feather(PATHS['features'])

    print(f'Cache ready in {(time.time()-t0)/60:.2f} min')
else:
    print('Cache exists, skipping rebuild.')
    model = None

# CELL 14 — PHASE B: ONLINE RANKING
t_online = time.time()

import json as _jstd
df_load   = pd.read_feather(PATHS['features'])
list_cols = ['career_history', 'education', 'skills', 'redrob_signals', 'skill_assessment_scores']
for col in list_cols:
    df_load[col] = df_load[col].apply(
        lambda x: _jstd.loads(x) if isinstance(x, str) and x[:1] in ('[', '{') else x)
df = df_load

embeddings = np.load(PATHS['embeddings'], mmap_mode='r')
embed_idx  = np.load(PATHS['embed_idx'])

if model is None:
    import torch
    torch.set_num_threads(_cpu_cores)
    print('Loading model...')
    model = SentenceTransformer(EMBED_MODEL_NAME)

jd_emb = model.encode([JD_TEXT], convert_to_numpy=True, normalize_embeddings=True).astype('float32')[0]

# Fast top-k via argpartition (O(n) vs O(n log n) sort)
def top_k_indices(scores, k):
    k = min(k, len(scores))
    if k <= 0: return np.array([], dtype=int)
    part = np.argpartition(-scores, k-1)[:k]
    return part[np.argsort(-scores[part])]

embed_scores    = (embeddings @ jd_emb).reshape(-1)
top_local       = top_k_indices(embed_scores, CFG['EMBED_TOP_K'])
candidate_idx   = embed_idx[top_local]

print(f'Retrieved {len(candidate_idx)} candidates for scoring')

scored = df.iloc[candidate_idx].copy()
print('Scoring shortlist...')
scored['score'] = [score_candidate_row(r) for r in scored.to_dict('records')]

top_n  = CFG['TOP_N']
ranked = scored.sort_values(['score', 'candidate_id'], ascending=[False, True]).head(top_n).copy()
ranked['rank'] = np.arange(1, len(ranked)+1)

print(f'Online done in {time.time()-t_online:.1f}s')
print(f'Honeypots in top {top_n}:', int(ranked['honeypot_suspect'].sum()))
print(f'Tier-D in top {top_n}:',    int((ranked['role_tier'] == 'D').sum()))
print('\n--- TOP RESULTS ---')
print(ranked[['candidate_id','current_title','role_tier','score']].to_string(index=False))

# CELL 15 — Reasoning
def top_relevant_skills(row, n=2):
    sk = sorted([(skill_relevance_value(s.get('name')), safe_str(s.get('name')))
                 for s in row.get('skills') or [] if skill_relevance_value(s.get('name')) > 0],
                key=lambda x: (-x[0], x[1].lower()))
    seen, names = set(), []
    for _, name in sk:
        if name.lower() not in seen:
            seen.add(name.lower()); names.append(name)
        if len(names) >= n: break
    return names

def notice_note(row):
    days = row.get('notice_period_days')
    if days is None or (isinstance(days, float) and math.isnan(days)):
        return 'notice period is not listed'
    days = int(days)
    return f'{days}-day notice period fits the JD timeline' if days <= 30 else f'{days}-day notice period is a logistics concern'

def last_active_note(row):
    dt = parse_date(row.get('last_active_date'))
    if dt is None: return 'activity recency is not available'
    days = max(0, (CFG['TODAY'] - dt).days)
    return (f'active recently ({days} days ago)' if days <= 30
            else f'last active {days} days ago' if days <= 90
            else f'activity is stale at {days} days since last active')

def generate_reasoning(row):
    title = safe_str(row.get('current_title')) or 'candidate'
    yoe   = float(row.get('years_of_experience') or 0.0)
    skills = top_relevant_skills(row)
    skill_text = ', '.join(skills) if skills else 'relevant ML/search signals in career history'
    concern = None
    if notice_period_factor(row.get('notice_period_days')) < 0.8:
        concern = notice_note(row) + '.'
    elif recency_factor(row.get('last_active_date')) < 0.75:
        concern = last_active_note(row).capitalize() + '.'
    elif row.get('role_tier') == 'C':
        concern = 'Role identity is adjacent rather than a direct ML/search engineering match.'
    v = int(row.get('rank', 0)) % 4
    if v == 0:   text = f'{title} with {yoe:.1f} years shows fit through {skill_text}; {notice_note(row)} and {last_active_note(row)}.'
    elif v == 1: text = f'Current title is {title}, with {yoe:.1f} years and strongest skills around {skill_text}. Availability: {notice_note(row)}.'
    elif v == 2: text = f'{title} ranked for ML/search-adjacent career evidence plus {skill_text}. {last_active_note(row).capitalize()}.'
    else:        text = f'Fit from {title} role identity, {yoe:.1f} years experience, skills including {skill_text}. {notice_note(row).capitalize()}.'
    if concern and concern not in text:
        text += ' ' + concern
    return re.sub(r'\s+', ' ', text).strip()[:900]

ranked['reasoning'] = ranked.apply(generate_reasoning, axis=1)

# CELL 16 — Output
submission = ranked[['candidate_id', 'rank', 'score', 'reasoning']].copy()
submission['score'] = submission['score'].astype(float)
submission = submission.sort_values('rank').reset_index(drop=True)

assert list(submission.columns) == ['candidate_id', 'rank', 'score', 'reasoning']
assert len(submission) == top_n
if not TEST_MODE:
    assert submission['rank'].tolist() == list(range(1, 101))

submission.to_csv(PATHS['submission'], index=False, encoding='utf-8')
print(f'Submission written: {PATHS["submission"]}')
print(submission[['candidate_id','rank','score']].head(10).to_string(index=False))

# CELL 17 — Validate + copy to Drive
if not TEST_MODE:
    result = subprocess.run(
        ['python', str(PATHS['validator']), str(PATHS['submission'])],
        capture_output=True, text=True)
    print('Validator stdout:', result.stdout)
    print('Validator stderr:', result.stderr)
    print('Validator return code:', result.returncode)

import shutil
try:
    shutil.copy2(str(PATHS['submission']), str(PATHS['submission_drive']))
    print('Copied to Drive:', PATHS['submission_drive'])
except Exception:
    pass

from google.colab import files
files.download(str(PATHS['submission']))

Mounted at /content/drive


TEST_MODE = False  (rows = ALL)
JD loaded: 9564 chars


Reading JSONL: 0it [00:00, ?it/s]

Parsing + flattening 100000 candidates...


Flattening:   0%|          | 0/100000 [00:00<?, ?it/s]

...

Encoding 18687 candidates (batch=64, cores=2)...


Batches:   0%|          | 0/292 [00:00<?, ?it/s]

Cache ready in 23.15 min
Retrieved 4000 candidates for scoring
Scoring shortlist...
Online done in 18.4s
Honeypots in top 100: 0
Tier-D in top 100: 0

--- TOP RESULTS ---
candidate_id                    current_title role_tier    score
CAND_0002025               Senior AI Engineer         A 0.756893
CAND_0081846                 Lead AI Engineer         A 0.755868
CAND_0025640             AI Research Engineer         A 0.752478
CAND_0043860               Junior ML Engineer         A 0.724512
CAND_0099806                      AI Engineer         A 0.704127
CAND_0053605    Senior Software Engineer (ML)         B 0.697310
CAND_0018499 Senior Machine Learning Engineer         A 0.670898
CAND_0033179             AI Research Engineer         A 0.647076
CAND_0046132             AI Research Engineer         A 0.643248
CAND_0046525 Senior Machine Learning Engineer         A 0.625523
CAND_0075481             AI Research Engineer         A 0.622654
CAND_0043637               Junior ML Engineer    

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
############################################################
# CELL 1 - Install Dependencies
############################################################
!pip -q install "optimum[onnxruntime]" orjson numpy pandas tqdm python-docx pyarrow

############################################################
# CELL 2 - Imports
############################################################
import json
import math
import os
import re
import subprocess
import time
import warnings
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from datetime import datetime
from pathlib import Path

import numpy as np
import orjson
import pandas as pd
from docx import Document
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# Must be set before onnxruntime / tokenizers spin up their thread pools.
_CPU_CORES = os.cpu_count() or 4
os.environ["OMP_NUM_THREADS"] = str(_CPU_CORES)
os.environ["TOKENIZERS_PARALLELISM"] = "true"

############################################################
# CELL 3 - Mount Google Drive
############################################################
from google.colab import drive
drive.mount('/content/drive')

############################################################
# CELL 4 - Configuration
############################################################
folder = "/content/drive/MyDrive/data"
TEST_MODE = False
TEST_ROWS = 5

CFG = {
    'EMBED_MODEL':            'sentence-transformers/all-MiniLM-L6-v2',
    'MAX_SEQ_LEN':             96,
    'EMBED_BATCH_SIZE':        256,
    'TEXT_WORD_LIMIT':         128,
    'EMBED_TOP_K':             TEST_ROWS if TEST_MODE else 4000,
    'WEIGHT_ROLE':              0.40,
    'WEIGHT_EXPERIENCE':        0.25,
    'WEIGHT_SKILLS':            0.15,
    'WEIGHT_LOCATION':          0.10,
    'WEIGHT_EDUCATION':         0.05,
    'WEIGHT_GITHUB':            0.05,
    'SKILL_SATURATION':         6.0,
    'ROLE_D_KEEP_MULTIPLIER':   0.02,
    'TODAY':                    datetime.now(),
    'TOP_N':                    TEST_ROWS if TEST_MODE else 100,
    'USE_INT8':                 True,   # set False for fp32 (safer ranking parity, still fast)
    'PARALLEL_MIN_ROWS':        2000,   # below this, process-pool overhead isn't worth it
    'CACHE_EMBEDDINGS':         True,   # skip re-encoding candidates whose text hasn't changed
}

print(f"TEST_MODE = {TEST_MODE}  (rows = {TEST_ROWS if TEST_MODE else 'ALL'})")
print(f"CPU cores detected: {_CPU_CORES}")

############################################################
# CELL 5 - Path Resolution
############################################################
FOLDER = Path(folder)

def find_file(stem_candidates, extensions):
    for stem in stem_candidates:
        for ext in extensions:
            p = FOLDER / f"{stem}{ext}"
            if p.exists():
                return p
    tried = [str(FOLDER / f"{s}{e}") for s in stem_candidates for e in extensions]
    existing = "\n".join(sorted(p.name for p in FOLDER.iterdir())) if FOLDER.exists() else "FOLDER NOT FOUND"
    raise FileNotFoundError("Could not find any of:\n" + "\n".join(tried) + f"\n\nFolder contents:\n{existing}")

_cand_path = FOLDER / 'candidates.jsonl'
if not _cand_path.exists():
    raise FileNotFoundError(f"candidates.jsonl not found in: {FOLDER}")

PATHS = {
    'candidates_jsonl':  _cand_path,
    'job_description':   find_file(['job_description'],     ['.md', '.txt', '.docx']),
    'signals_doc':       find_file(['redrob_signals_doc'],  ['.md', '.txt', '.docx']),
    'submission_spec':   find_file(['submission_spec'],     ['.md', '.txt', '.docx']),
    'candidate_schema':  find_file(['candidate_schema'],    ['.json']),
    'sample_candidates': find_file(['sample_candidates'],   ['.json']),
    'sample_submission': find_file(['sample_submission'],   ['.csv']),
    'validator':         find_file(['validate_submission'], ['.py']),
}

OUT_DIR = Path('/content/redrob_cache')
OUT_DIR.mkdir(parents=True, exist_ok=True)

suffix = 'test' if TEST_MODE else 'full'
PATHS['submission']       = OUT_DIR / f'{suffix}_submission.csv'
PATHS['submission_drive'] = FOLDER  / ('test_submission.csv' if TEST_MODE else 'submission.csv')
PATHS['embed_cache']      = OUT_DIR / 'embed_cache.npz'

print("Resolved paths:")
for k, v in PATHS.items():
    print(f"  {k}: {v}")

############################################################
# CELL 6 - Read Text Docs (negligible cost, unchanged)
############################################################
def read_text_doc(path):
    if path.suffix.lower() == '.docx':
        doc = Document(str(path))
        parts = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
        for table in doc.tables:
            for row in table.rows:
                cells = [c.text.strip() for c in row.cells if c.text.strip()]
                if cells:
                    parts.append(' | '.join(cells))
        return '\n'.join(parts)
    return path.read_text(encoding='utf-8')

JD_TEXT              = read_text_doc(PATHS['job_description'])
SIGNALS_DOC_TEXT     = read_text_doc(PATHS['signals_doc'])
SUBMISSION_SPEC_TEXT = read_text_doc(PATHS['submission_spec'])

with open(PATHS['candidate_schema'], 'r', encoding='utf-8') as f:
    CANDIDATE_SCHEMA = json.load(f)

print(f"JD loaded: {len(JD_TEXT)} chars")

############################################################
# CELL 7 - Helper Functions
############################################################
def safe_str(v):
    return '' if v is None or (isinstance(v, float) and math.isnan(v)) else str(v)

def parse_date(value):
    if not value:
        return None
    try:
        dt = pd.to_datetime(value, errors='coerce')
        if pd.isna(dt):
            return None
        return dt.to_pydatetime()
    except Exception:
        return None

def truncate_words(text, limit):
    words = safe_str(text).split()
    return ' '.join(words[:limit])

############################################################
# CELL 8 - Load candidates.jsonl -> Flattened DataFrame
#
# orjson (C parser) + bulk file read + flatten_candidate fanned out across
# all cores with ProcessPoolExecutor. Colab's fork start method means
# module-level globals (CFG, regex, etc.) are inherited by workers for free.
############################################################
def flatten_candidate(obj):
    profile   = obj.get('profile') or {}
    career    = obj.get('career_history') or []
    education = obj.get('education') or []
    skills    = obj.get('skills') or []
    signals   = obj.get('redrob_signals') or {}

    career_titles = ' '.join(safe_str(j.get('title')) for j in career if isinstance(j, dict))
    career_desc   = ' '.join(safe_str(j.get('description')) for j in career if isinstance(j, dict))

    candidate_text_parts = [profile.get('headline'), profile.get('summary'),
                             profile.get('current_title'), profile.get('current_industry')]
    for job in career:
        if isinstance(job, dict):
            candidate_text_parts.extend([job.get('title'), job.get('industry'), job.get('description')])
    candidate_text = truncate_words(' '.join(safe_str(p) for p in candidate_text_parts if p), CFG['TEXT_WORD_LIMIT'])

    return {
        'candidate_id':              obj.get('candidate_id'),
        'current_title':             safe_str(profile.get('current_title')),
        'headline':                  safe_str(profile.get('headline')),
        'summary':                   safe_str(profile.get('summary')),
        'location':                  safe_str(profile.get('location')),
        'country':                   safe_str(profile.get('country')),
        'years_of_experience':       float(profile.get('years_of_experience') or 0.0),
        'current_company':           safe_str(profile.get('current_company')),
        'current_company_size':      safe_str(profile.get('current_company_size')),
        'current_industry':          safe_str(profile.get('current_industry')),
        'career_history':            career,
        'education':                 education,
        'skills':                    skills,
        'redrob_signals':            signals,
        'career_text':                truncate_words(career_desc, CFG['TEXT_WORD_LIMIT']),
        'career_titles_text':        career_titles,
        'candidate_text':            candidate_text,
        'last_active_date':          signals.get('last_active_date'),
        'notice_period_days':        signals.get('notice_period_days'),
        'recruiter_response_rate':   signals.get('recruiter_response_rate'),
        'interview_completion_rate': signals.get('interview_completion_rate'),
        'open_to_work_flag':         signals.get('open_to_work_flag'),
        'preferred_work_mode':       signals.get('preferred_work_mode'),
        'willing_to_relocate':       signals.get('willing_to_relocate'),
        'github_activity_score':     signals.get('github_activity_score'),
        'skill_assessment_scores':   signals.get('skill_assessment_scores') or {},
    }

def _flatten_chunk(lines):
    return [flatten_candidate(orjson.loads(l)) for l in lines]

def chunk_list(items, n_chunks):
    n_chunks = max(1, n_chunks)
    chunk_size = math.ceil(len(items) / n_chunks)
    return [items[i:i + chunk_size] for i in range(0, len(items), chunk_size)]

def parallel_row_apply(records, chunk_fn, desc):
    if len(records) < CFG['PARALLEL_MIN_ROWS']:
        return chunk_fn(records)
    chunks = chunk_list(records, _CPU_CORES)
    with ProcessPoolExecutor(max_workers=_CPU_CORES) as executor:
        results = list(
            tqdm(
                executor.map(chunk_fn, chunks),
                total=len(chunks),
                desc=desc,
                mininterval=1.0
            )
        )
    out = []
    for r in results:
        out.extend(r)
    return out

def load_candidates_to_df(limit=None):
    t0 = time.time()
    with open(PATHS['candidates_jsonl'], 'rb') as f:
        raw = f.read()
    lines = [l for l in raw.splitlines() if l.strip()]
    if limit:
        lines = lines[:limit]

    if len(lines) >= CFG['PARALLEL_MIN_ROWS']:
        chunks = chunk_list(lines, _CPU_CORES)
        with ProcessPoolExecutor(max_workers=_CPU_CORES) as executor:
            chunk_results = list(
                tqdm(
                    executor.map(_flatten_chunk, chunks),
                    total=len(chunks),
                    desc="Loading candidates (parallel)",
                    mininterval=1.0
                )
            )
        rows = [r for chunk in chunk_results for r in chunk]
    else:
        rows = [flatten_candidate(orjson.loads(l)) for l in lines]

    df = pd.DataFrame(rows)
    df['years_of_experience'] = df['years_of_experience'].astype('float32')
    print(f'Loaded {len(df)} candidates in {time.time()-t0:.1f}s')
    return df

df = load_candidates_to_df(limit=TEST_ROWS if TEST_MODE else None)

############################################################
# CELL 9 - Vectorized Role-Tier Classification
############################################################
TIER_A_PATTERN = r'\b(machine learning engineer|ml engineer|ai engineer|applied scientist|research engineer|recommendation engineer|ranking engineer|recommender systems engineer|search engineer|nlp engineer|computer vision engineer)\b'
TIER_B_PATTERN = r'\b(software engineer|backend engineer|back end engineer|data engineer|platform engineer|systems engineer|python developer|data scientist)\b'
TIER_C_PATTERN = r'\b(full.?stack|frontend|front end|software developer|backend developer|developer)\b'
TIER_D_PATTERN = r'\b(hr|human resources|recruiter|marketing manager|sales executive|content writer|graphic designer|accountant|finance|civil engineer|mechanical engineer|electrical engineer|operations manager|customer support|business analyst|project manager|teacher|nurse|doctor|legal|lawyer)\b'
ML_EVIDENCE_PATTERN = r'\b(machine learning|deep learning|nlp|bert|transformer|embedding|vector database|retrieval|ranking|recommendation|recommender|rag|llm|model serving|feature store|pytorch|tensorflow|xgboost|scikit|search relevance)\b'

def compute_role_tier_vectorized(dframe):
    title_lc = dframe['current_title'].str.lower()
    evidence_lc = (dframe['summary'] + ' ' + dframe['career_text']).str.lower()

    has_d = title_lc.str.contains(TIER_D_PATTERN, regex=True, na=False)
    has_a = title_lc.str.contains(TIER_A_PATTERN, regex=True, na=False)
    is_ds = title_lc.str.contains(r'\bdata scientist\b', regex=True, na=False)
    has_ml_evidence = evidence_lc.str.contains(ML_EVIDENCE_PATTERN, regex=True, na=False)
    has_b = title_lc.str.contains(TIER_B_PATTERN, regex=True, na=False)
    has_c = title_lc.str.contains(TIER_C_PATTERN, regex=True, na=False)

    is_d_final = has_d & ~has_a
    is_a_final = has_a
    is_ds_ml   = is_ds & has_ml_evidence & ~is_a_final
    is_b_final = has_b & has_ml_evidence & ~is_a_final & ~is_ds_ml
    is_c_final = has_c & ~is_a_final & ~is_ds_ml & ~is_b_final

    tier = np.select(
        [is_d_final, is_a_final, is_ds_ml, is_b_final, is_c_final],
        ['D', 'A', 'A', 'B', 'C'],
        default='D'
    )
    title_score = np.select(
        [is_d_final, is_a_final, is_ds_ml, is_b_final, is_c_final],
        [0.0, 1.0, 0.9, 0.7, 0.35],
        default=0.0
    )
    return tier, title_score

def role_title_tier_single(title, evidence_text=''):
    """Used only for scoring recent career_history entries on the small shortlist."""
    title = safe_str(title)
    if re.search(TIER_D_PATTERN, title, re.I) and not re.search(TIER_A_PATTERN, title, re.I):
        return 'D', 0.0
    if re.search(TIER_A_PATTERN, title, re.I):
        return 'A', 1.0
    ev = safe_str(evidence_text)
    if re.search(r'\bdata scientist\b', title, re.I) and re.search(ML_EVIDENCE_PATTERN, ev, re.I):
        return 'A', 0.9
    if re.search(TIER_B_PATTERN, title, re.I) and re.search(ML_EVIDENCE_PATTERN, ev, re.I):
        return 'B', 0.7
    if re.search(TIER_C_PATTERN, title, re.I):
        return 'C', 0.35
    return 'D', 0.0

_tier_arr, _score_arr = compute_role_tier_vectorized(df)
df['role_tier'] = _tier_arr
df['role_title_score'] = _score_arr

print(df['role_tier'].value_counts())

############################################################
# CELL 10 - Honeypot Detection (parallelized via shared pool)
############################################################
def honeypot_check(row):
    zero_expert = sum(
        1 for s in (row['skills'] or [])
        if safe_str(s.get('proficiency')).lower() in {'advanced', 'expert'}
        and int(s.get('duration_months') or 0) == 0
    )
    if zero_expert >= 3:
        return True

    yoe = row['years_of_experience']
    career = row['career_history'] or []

    starts = [parse_date(j.get('start_date')) for j in career]
    starts = [d for d in starts if d is not None]
    if starts:
        earliest_span_years = max(0.0, (CFG['TODAY'] - min(starts)).days / 365.25)
        if earliest_span_years + 3.0 < yoe:
            return True

    if yoe > 0:
        for j in career:
            if int(j.get('duration_months') or 0) > yoe * 12 + 1:
                return True

    return False

def _honeypot_chunk(records):
    return [honeypot_check(r) for r in records]

non_d_mask = df['role_tier'] != 'D'
non_d_records = df.loc[non_d_mask].to_dict('records')
honeypot_results = [
    honeypot_check(r)
    for r in tqdm(non_d_records, desc="Honeypot scan")
]

df['honeypot_suspect'] = False
df.loc[non_d_mask, 'honeypot_suspect'] = honeypot_results

print(f"Honeypot suspects flagged: {int(df['honeypot_suspect'].sum())} / {len(df)}")

############################################################
# CELL 11 - Role Fit Score (parallelized, restricted to embed-eligible rows)
############################################################
def _role_fit_chunk(records):
    out = []
    for row in records:
        title_score = row['role_title_score']
        recent_jobs = list(row['career_history'] or [])[:3]
        recent_scores = [role_title_tier_single(j.get('title'), safe_str(j.get('description')))[1] for j in recent_jobs]
        identity = max([title_score] + recent_scores) if recent_scores else title_score
        out.append(float(np.clip(0.7 * title_score + 0.3 * identity, 0, 1)))
    return out

# Only the expensive recency-boosted role score needs computing for
# candidates that will actually be embedded/considered. Tier-D rows are
# crushed by the 0.02 multiplier later anyway, so title_score alone is enough.
embed_eligible_mask = (~df['honeypot_suspect']) & (df['role_tier'] != 'D')
df['role_fit_score'] = df['role_title_score']

eligible_records = df.loc[embed_eligible_mask].to_dict('records')
role_fit_results = [
    score
    for score in _role_fit_chunk(eligible_records)
]
df.loc[embed_eligible_mask, 'role_fit_score'] = role_fit_results

############################################################
# CELL 12 - Experience Validity Scoring (only runs on final ~4000-row
# shortlist in Cell 18, negligible cost — unchanged)
############################################################
CONCEPT_PATTERNS = [
    re.compile(r'\b(embedding|sentence transformer|semantic search|vector search|vector database|pinecone|weaviate|qdrant|milvus|faiss|retrieval)\b', re.I),
    re.compile(r'\b(ndcg|mrr|map@|ranking|reranking|learning to rank|search relevance)\b', re.I),
    re.compile(r'\b(python|fastapi|flask|django|docker|kubernetes|microservice|production|deployed|mlops)\b', re.I),
    re.compile(r'\b(hybrid search|bm25|elasticsearch|opensearch|lucene)\b', re.I),
    re.compile(r'\b(llm|large language model|fine.?tun|lora|qlora|peft|rag|langchain)\b', re.I),
    re.compile(r'\b(pytorch|tensorflow|scikit|xgboost|transformer|bert|neural network|deep learning)\b', re.I),
]
N_CONCEPTS = len(CONCEPT_PATTERNS)
PRODUCTION_PATTERN = re.compile(r'\b(deployed|production|scaled|served|serving|latency|live|monitoring|a/b test|sla|throughput)\b', re.I)

def role_months_score(career_history):
    months = 0.0
    today = CFG['TODAY']
    for job in career_history or []:
        _, s = role_title_tier_single(job.get('title'), safe_str(job.get('description')))
        if s >= 0.7:
            end = parse_date(job.get('end_date')) or today
            age = max(0.0, (today - end).days / 365.25)
            rw = 1.0 if age <= 3 else max(0.35, 1.0 - 0.12 * (age - 3))
            months += int(job.get('duration_months') or 0) * rw * s
    return float(np.clip(1.0 - abs(months - 72) / 72, 0, 1))

def experience_validity_score(row):
    text = safe_str(row.get('career_text')) + ' ' + safe_str(row.get('summary'))
    hits = sum(1 for p in CONCEPT_PATTERNS if p.search(text))
    yoe = float(row.get('years_of_experience') or 0.0)
    ys = 1.0 if 5 <= yoe <= 9 else (max(0.0, yoe / 5) if yoe < 5 else max(0.35, 1.0 - (yoe - 9) / 12))
    prod = 0.08 if PRODUCTION_PATTERN.search(text) else 0.0
    return float(np.clip(
        0.45 * (hits / N_CONCEPTS) + 0.35 * role_months_score(row.get('career_history')) + 0.20 * ys + prod,
        0, 1
    ))

############################################################
# CELL 13 - Skill Relevance Scoring (unchanged)
############################################################
SKILL_TAXONOMY = {
    'python': 0.9, 'pandas': 0.35, 'numpy': 0.35, 'scikit-learn': 0.55, 'sklearn': 0.55,
    'machine learning': 1.0, 'ml': 0.8, 'deep learning': 0.9, 'nlp': 1.0,
    'pytorch': 0.85, 'tensorflow': 0.75, 'transformers': 0.95, 'bert': 0.9,
    'embeddings': 1.0, 'sentence transformers': 1.0, 'semantic search': 1.0,
    'vector database': 1.0, 'pinecone': 0.95, 'weaviate': 0.95, 'qdrant': 0.95,
    'milvus': 0.95, 'faiss': 0.9, 'elasticsearch': 0.75, 'opensearch': 0.75,
    'bm25': 0.8, 'hybrid search': 1.0, 'ranking': 0.9, 'learning to rank': 1.0,
    'ndcg': 1.0, 'mrr': 1.0, 'rag': 0.9, 'langchain': 0.65, 'llm': 0.85,
    'fine tuning': 0.8, 'lora': 0.85, 'qlora': 0.85, 'peft': 0.85, 'mlops': 0.75,
    'docker': 0.35, 'kubernetes': 0.45, 'fastapi': 0.35,
}
PROFICIENCY_WEIGHT = {'beginner': 0.3, 'intermediate': 0.6, 'advanced': 0.85, 'expert': 1.0}

def skill_relevance_value(name):
    n = safe_str(name).lower().strip()
    if not n:
        return 0.0
    if n in SKILL_TAXONOMY:
        return SKILL_TAXONOMY[n]
    for key, w in SKILL_TAXONOMY.items():
        if key in n or n in key:
            return w * 0.85
    return 0.0

def duration_trust(months):
    m = int(months or 0)
    if m == 0:
        return 0.1
    if m <= 6:
        return 0.5
    if m <= 12:
        return 0.75
    return 1.0

def skill_relevance_score(row):
    total = 0.0
    assessments = {
        safe_str(k).lower(): float(v)
        for k, v in (row.get('skill_assessment_scores') or {}).items() if v is not None
    }
    for skill in row.get('skills') or []:
        rel = skill_relevance_value(skill.get('name'))
        if rel <= 0:
            continue
        key = safe_str(skill.get('name')).lower()
        if key in assessments:
            trust = np.clip(assessments[key] / 100.0, 0, 1)
        else:
            trust = PROFICIENCY_WEIGHT.get(safe_str(skill.get('proficiency')).lower(), 0.45) * duration_trust(skill.get('duration_months'))
        total += rel * trust
    return float(1.0 - math.exp(-total / CFG['SKILL_SATURATION']))

############################################################
# CELL 14 - Location, Education, Availability, Github Scoring (unchanged)
############################################################
def location_logistics_score(row):
    loc = safe_str(row.get('location')).lower()
    country = safe_str(row.get('country')).lower()
    if re.search(r'\b(pune|noida)\b', loc):
        ls = 1.0
    elif re.search(r'\b(bangalore|bengaluru|hyderabad|mumbai|delhi|gurgaon|gurugram|ncr)\b', loc):
        ls = 0.85
    elif country in {'india', 'in'} or 'india' in loc:
        ls = 0.6
    else:
        ls = 0.5 if bool(row.get('willing_to_relocate')) else 0.2
    ms = {'hybrid': 1.0, 'flexible': 1.0, 'onsite': 0.8, 'remote': 0.6}.get(
        safe_str(row.get('preferred_work_mode')).lower(), 0.75
    )
    return float(0.7 * ls + 0.3 * ms)

RELEVANT_EDU_PATTERN = re.compile(
    r'\b(computer|cs|software|information technology|artificial intelligence|'
    r'machine learning|data science|statistics|mathematics|math|electronics|ece|analytics)\b', re.I
)

def education_score(row):
    edu = row.get('education') or []
    if not edu:
        return 0.5
    best = 0.4
    for e in edu:
        field = safe_str(e.get('degree')) + ' ' + safe_str(e.get('field_of_study'))
        rel = bool(RELEVANT_EDU_PATTERN.search(field))
        tier = safe_str(e.get('tier')).lower()
        s = 1.0 if rel and tier == 'tier_1' else 0.8 if rel and tier == 'tier_2' else 0.65 if rel else 0.4
        best = max(best, s)
    return float(best)

def notice_period_factor(days):
    try:
        days = float(days)
    except Exception:
        return 0.75
    if days <= 30:
        return 1.0
    if days <= 60:
        return 0.8
    if days <= 90:
        return 0.6
    return 0.45

def recency_factor(last_active_date):
    dt = parse_date(last_active_date)
    if dt is None:
        return 0.5
    days = max(0, (CFG['TODAY'] - dt).days)
    if days <= 30:
        return 1.0
    if days <= 60:
        return 0.9
    if days <= 90:
        return 0.75
    if days <= 180:
        return 0.5
    return 0.25

def availability_multiplier(row):
    rr = float(row.get('recruiter_response_rate') or 0.5)
    ic = float(row.get('interview_completion_rate') or 0.5)
    s = recency_factor(row.get('last_active_date')) * notice_period_factor(row.get('notice_period_days'))
    s *= (0.5 + 0.5 * np.clip(rr, 0, 1)) * (0.5 + 0.5 * np.clip(ic, 0, 1))
    if bool(row.get('open_to_work_flag')):
        s *= 1.15
    return float(np.clip(s, 0, 1))

def github_bonus(row):
    try:
        score = float(row.get('github_activity_score'))
    except Exception:
        return 0.0
    if score >= 70:
        return 0.05
    if score >= 40:
        return 0.02
    return 0.0

############################################################
# CELL 15 - Composite Scoring Function (unchanged)
############################################################
def score_candidate_row(row):
    base = (
        CFG['WEIGHT_ROLE']       * row.get('role_fit_score', 0.0)
        + CFG['WEIGHT_EXPERIENCE'] * experience_validity_score(row)
        + CFG['WEIGHT_SKILLS']     * skill_relevance_score(row)
        + CFG['WEIGHT_LOCATION']   * location_logistics_score(row)
        + CFG['WEIGHT_EDUCATION']  * education_score(row)
        + CFG['WEIGHT_GITHUB']     * github_bonus(row)
    )
    final = base * availability_multiplier(row)
    if row.get('role_tier') == 'D':
        final *= CFG['ROLE_D_KEEP_MULTIPLIER']
    return float(final)

############################################################
# CELL 16 - Embedding Generation (the actual bottleneck fix)
#
# Root cause of the original 12+ min embedding stage: it called
# SentenceTransformer(..., backend="onnx") then torch.set_num_threads(...).
# That sets PyTorch's thread pool, but backend="onnx" runs inference
# through onnxruntime, not PyTorch, so that line did nothing. Combined
# with fp32 weights and no explicit ORT session threading, the model was
# very likely running on 1-2 threads.
#
# Fix: bypass sentence-transformers, drive onnxruntime directly via
# optimum, control SessionOptions ourselves, quantize to int8, batch by
# sorted length, overlap tokenization with inference, and cache both the
# exported model on disk and the resulting embeddings by content hash.
############################################################
from optimum.onnxruntime import ORTModelForFeatureExtraction, ORTQuantizer, AutoQuantizationConfig
from transformers import AutoTokenizer
import onnxruntime as ort
import hashlib

ONNX_DIR  = OUT_DIR / "onnx_fp32"
QUANT_DIR = OUT_DIR / "onnx_int8"


def prepare_onnx_model(model_id, use_int8):
    target_dir = QUANT_DIR if use_int8 else ONNX_DIR
    model_file = "model_quantized.onnx" if use_int8 else "model.onnx"

    # Cached on disk -> only paid once ever (or once per fresh Drive/runtime),
    # not on every execution.
    if not (target_dir / model_file).exists():
        target_dir.mkdir(parents=True, exist_ok=True)
        base_model = ORTModelForFeatureExtraction.from_pretrained(model_id, export=True)
        base_model.save_pretrained(ONNX_DIR)
        if use_int8:
            quantizer = ORTQuantizer.from_pretrained(ONNX_DIR)
            qconfig = AutoQuantizationConfig.avx2(is_static=False, per_channel=False)
            quantizer.quantize(save_dir=QUANT_DIR, quantization_config=qconfig)

    sess_opts = ort.SessionOptions()
    sess_opts.intra_op_num_threads = _CPU_CORES
    sess_opts.inter_op_num_threads = 1
    sess_opts.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
    sess_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    ort_model = ORTModelForFeatureExtraction.from_pretrained(
        target_dir,
        file_name=model_file,
        provider="CPUExecutionProvider",
        session_options=sess_opts,
        use_io_binding=False,
    )
    return tokenizer, ort_model


def mean_pool_normalize(last_hidden_state, attention_mask):
    mask = attention_mask[..., None].astype(np.float32)
    summed = (last_hidden_state * mask).sum(axis=1)
    counts = np.clip(mask.sum(axis=1), 1e-9, None)
    pooled = summed / counts
    norms = np.clip(np.linalg.norm(pooled, axis=1, keepdims=True), 1e-9, None)
    return (pooled / norms).astype(np.float32)


def encode_texts(texts, tokenizer, ort_model, batch_size, max_length, desc="Embedding"):
    n = len(texts)
    if n == 0:
        return np.zeros((0, ort_model.config.hidden_size), dtype='float32')

    order = np.argsort([-len(t) for t in texts])
    sorted_texts = [texts[i] for i in order]
    num_batches = math.ceil(n / batch_size)

    def tokenize_batch(bidx):
        chunk = sorted_texts[bidx * batch_size:(bidx + 1) * batch_size]
        return tokenizer(chunk, padding=True, truncation=True, max_length=max_length, return_tensors="np")

    results = [None] * num_batches
    with ThreadPoolExecutor(max_workers=2) as tok_pool:
        prefetch_depth = 2
        futures = {i: tok_pool.submit(tokenize_batch, i) for i in range(min(prefetch_depth, num_batches))}
        next_submit = prefetch_depth
        pbar = tqdm(total=num_batches, desc=desc, mininterval=1.0)
        for i in range(num_batches):
            enc = futures.pop(i).result()
            if next_submit < num_batches:
                futures[next_submit] = tok_pool.submit(tokenize_batch, next_submit)
                next_submit += 1

            out = ort_model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"])
            pooled = mean_pool_normalize(out.last_hidden_state, enc["attention_mask"])
            results[i] = pooled
            pbar.update(1)
        pbar.close()

    embeddings_sorted = np.vstack(results).astype('float32')
    embeddings = np.empty_like(embeddings_sorted)
    embeddings[order] = embeddings_sorted
    return embeddings


def load_embed_cache():
    if not CFG['CACHE_EMBEDDINGS'] or not PATHS['embed_cache'].exists():
        return {}
    try:
        data = np.load(PATHS['embed_cache'], allow_pickle=False)
        return {h: data[h] for h in data.files}
    except Exception:
        return {}

def save_embed_cache(cache_dict):
    if not CFG['CACHE_EMBEDDINGS']:
        return
    try:
        np.savez_compressed(PATHS['embed_cache'], **cache_dict)
    except Exception as e:
        print('Could not save embedding cache:', e)

def text_hash(t):
    return hashlib.blake2b(t.encode('utf-8'), digest_size=16).hexdigest()


embed_mask = (~df['honeypot_suspect']) & (df['role_tier'] != 'D')
embed_df = df[embed_mask].copy()
print(f'Embedding {len(embed_df)}/{len(df)} candidates (Tier-D + honeypots excluded)')

t_embed = time.time()
tokenizer, ort_model = prepare_onnx_model(CFG['EMBED_MODEL'], CFG['USE_INT8'])

corpus = embed_df['candidate_text'].fillna('').astype(str).tolist()
hashes = [text_hash(t) for t in corpus]

cache = load_embed_cache()
to_encode_idx = [i for i, h in enumerate(hashes) if h not in cache]
print(f'{len(corpus) - len(to_encode_idx)} candidates hit cache, encoding {len(to_encode_idx)} fresh')

if to_encode_idx:
    fresh_texts = [corpus[i] for i in to_encode_idx]
    fresh_embeddings = encode_texts(
        fresh_texts, tokenizer, ort_model,
        batch_size=CFG['EMBED_BATCH_SIZE'],
        max_length=CFG['MAX_SEQ_LEN'],
        desc='Embedding candidates',
    )
    for local_i, global_i in enumerate(to_encode_idx):
        cache[hashes[global_i]] = fresh_embeddings[local_i]
    save_embed_cache(cache)

embeddings = np.vstack([cache[h] for h in hashes]).astype('float32')
embed_index_map = embed_df.index.to_numpy()
print(f'Embedding done in {(time.time()-t_embed):.1f}s for {len(corpus)} candidates')

############################################################
# CELL 17 - JD Embedding + Fast Top-K Retrieval
############################################################
jd_embedding = encode_texts([JD_TEXT], tokenizer, ort_model,
                             batch_size=1, max_length=CFG['MAX_SEQ_LEN'], desc='Embedding JD')[0]

def top_k_indices(scores, k):
    k = min(k, len(scores))
    if k <= 0:
        return np.array([], dtype=int)
    part = np.argpartition(-scores, k - 1)[:k]
    return part[np.argsort(-scores[part])]

embed_scores = (embeddings @ jd_embedding).reshape(-1)
top_local = top_k_indices(embed_scores, CFG['EMBED_TOP_K'])
candidate_positions = embed_index_map[top_local]

print(f'Retrieved {len(candidate_positions)} candidates for full scoring')

############################################################
# CELL 18 - Score Shortlist and Rank (only ~4000 rows, cheap, unchanged)
############################################################
t_score = time.time()
shortlist = df.loc[candidate_positions].copy()
shortlist['score'] = [score_candidate_row(r) for r in shortlist.to_dict('records')]

top_n = CFG['TOP_N']
ranked = shortlist.sort_values(['score', 'candidate_id'], ascending=[False, True]).head(top_n).copy()
ranked['rank'] = np.arange(1, len(ranked) + 1)

print(f'Scoring done in {time.time()-t_score:.1f}s')
print(f"Honeypots in top {top_n}: {int(ranked['honeypot_suspect'].sum())}")
print(f"Tier-D in top {top_n}: {int((ranked['role_tier'] == 'D').sum())}")
print('\n--- TOP RESULTS ---')
print(ranked[['candidate_id', 'current_title', 'role_tier', 'score']].to_string(index=False))

############################################################
# CELL 19 - Reasoning Generation (unchanged)
############################################################
def top_relevant_skills(row, n=2):
    scored_skills = sorted(
        [(skill_relevance_value(s.get('name')), safe_str(s.get('name')))
         for s in row.get('skills') or [] if skill_relevance_value(s.get('name')) > 0],
        key=lambda x: (-x[0], x[1].lower())
    )
    seen, names = set(), []
    for _, name in scored_skills:
        if name.lower() not in seen:
            seen.add(name.lower())
            names.append(name)
        if len(names) >= n:
            break
    return names

def notice_note(row):
    days = row.get('notice_period_days')
    if days is None or (isinstance(days, float) and math.isnan(days)):
        return 'notice period is not listed'
    days = int(days)
    return f'{days}-day notice period fits the JD timeline' if days <= 30 else f'{days}-day notice period is a logistics concern'

def last_active_note(row):
    dt = parse_date(row.get('last_active_date'))
    if dt is None:
        return 'activity recency is not available'
    days = max(0, (CFG['TODAY'] - dt).days)
    if days <= 30:
        return f'active recently ({days} days ago)'
    if days <= 90:
        return f'last active {days} days ago'
    return f'activity is stale at {days} days since last active'

def generate_reasoning(row):
    title = safe_str(row.get('current_title')) or 'candidate'
    yoe = float(row.get('years_of_experience') or 0.0)
    skills = top_relevant_skills(row)
    skill_text = ', '.join(skills) if skills else 'relevant ML/search signals in career history'

    concern = None
    if notice_period_factor(row.get('notice_period_days')) < 0.8:
        concern = notice_note(row) + '.'
    elif recency_factor(row.get('last_active_date')) < 0.75:
        concern = last_active_note(row).capitalize() + '.'
    elif row.get('role_tier') == 'C':
        concern = 'Role identity is adjacent rather than a direct ML/search engineering match.'

    v = int(row.get('rank', 0)) % 4
    if v == 0:
        text = f'{title} with {yoe:.1f} years shows fit through {skill_text}; {notice_note(row)} and {last_active_note(row)}.'
    elif v == 1:
        text = f'Current title is {title}, with {yoe:.1f} years and strongest skills around {skill_text}. Availability: {notice_note(row)}.'
    elif v == 2:
        text = f'{title} ranked for ML/search-adjacent career evidence plus {skill_text}. {last_active_note(row).capitalize()}.'
    else:
        text = f'Fit from {title} role identity, {yoe:.1f} years experience, skills including {skill_text}. {notice_note(row).capitalize()}.'

    if concern and concern not in text:
        text += ' ' + concern

    return re.sub(r'\s+', ' ', text).strip()[:900]

ranked['reasoning'] = ranked.apply(generate_reasoning, axis=1)

############################################################
# CELL 20 - Build and Write Submission CSV (unchanged)
############################################################
submission = ranked[['candidate_id', 'rank', 'score', 'reasoning']].copy()
submission['score'] = submission['score'].astype(float)
submission = submission.sort_values('rank').reset_index(drop=True)

assert list(submission.columns) == ['candidate_id', 'rank', 'score', 'reasoning']
assert len(submission) == top_n
if not TEST_MODE:
    assert submission['rank'].tolist() == list(range(1, 101))

submission.to_csv(PATHS['submission'], index=False, encoding='utf-8')
print(f'Submission written: {PATHS["submission"]}')
print(submission[['candidate_id', 'rank', 'score']].head(10).to_string(index=False))

############################################################
# CELL 21 - Validate Submission (unchanged)
############################################################
if not TEST_MODE:
    result = subprocess.run(
        ['python', str(PATHS['validator']), str(PATHS['submission'])],
        capture_output=True, text=True
    )
    print('Validator stdout:', result.stdout)
    print('Validator stderr:', result.stderr)
    print('Validator return code:', result.returncode)
    if result.returncode != 0:
        raise RuntimeError("Submission failed validation. See validator output above.")

############################################################
# CELL 22 - Copy to Drive, Download, and clean up the process pool
############################################################
import shutil

try:
    shutil.copy2(str(PATHS['submission']), str(PATHS['submission_drive']))
    print('Copied to Drive:', PATHS['submission_drive'])
except Exception as e:
    print('Could not copy to Drive:', e)

from google.colab import files
files.download(str(PATHS['submission']))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
TEST_MODE = False  (rows = ALL)
CPU cores detected: 2
Resolved paths:
  candidates_jsonl: /content/drive/MyDrive/data/candidates.jsonl
  job_description: /content/drive/MyDrive/data/job_description.docx
  signals_doc: /content/drive/MyDrive/data/redrob_signals_doc.docx
  submission_spec: /content/drive/MyDrive/data/submission_spec.docx
  candidate_schema: /content/drive/MyDrive/data/candidate_schema.json
  sample_candidates: /content/drive/MyDrive/data/sample_candidates.json
  sample_submission: /content/drive/MyDrive/data/sample_submission.csv
  validator: /content/drive/MyDrive/data/validate_submission.py
  submission: /content/redrob_cache/full_submission.csv
  submission_drive: /content/drive/MyDrive/data/submission.csv
  embed_cache: /content/redrob_cache/embed_cache.npz
JD loaded: 9564 chars


Loading candidates (parallel):   0%|          | 0/2 [00:00<?, ?it/s]

Loaded 100000 candidates in 40.1s
role_tier
D    81287
C    13965
B     3867
A      881
Name: count, dtype: int64


Honeypot scan:   0%|          | 0/18713 [00:00<?, ?it/s]

Honeypot suspects flagged: 26 / 100000


...

Embedding done in 801.8s for 18687 candidates


Embedding JD:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieved 4000 candidates for full scoring
Scoring done in 6.4s
Honeypots in top 100: 0
Tier-D in top 100: 0

--- TOP RESULTS ---
candidate_id                    current_title role_tier    score
CAND_0002025               Senior AI Engineer         A 0.756893
CAND_0081846                 Lead AI Engineer         A 0.755868
CAND_0055905 Senior Machine Learning Engineer         A 0.650761
CAND_0046525 Senior Machine Learning Engineer         A 0.625502
CAND_0046064              Senior NLP Engineer         A 0.588860
CAND_0078042              Applied ML Engineer         A 0.519862
CAND_0086022         Senior Applied Scientist         A 0.498131
CAND_0071974               Senior AI Engineer         A 0.495466
CAND_0098846                      AI Engineer         A 0.404770
CAND_0003717                Software Engineer         B 0.372501
CAND_0071941                Software Engineer         B 0.371754
CAND_0032279                   Java Developer         C 0.369902
CAND_0090153             

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
############################################################
# CELL 1 - Install Dependencies
############################################################
!pip -q install "optimum[onnxruntime]" orjson numpy pandas tqdm python-docx pyarrow

############################################################
# CELL 2 - Imports
############################################################
import json
import math
import os
import re
import subprocess
import time
import warnings
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from datetime import datetime
from pathlib import Path

import numpy as np
import orjson
import pandas as pd
from docx import Document
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# Must be set before onnxruntime / tokenizers spin up their thread pools.
_CPU_CORES = os.cpu_count() or 4
os.environ["OMP_NUM_THREADS"] = str(_CPU_CORES)
os.environ["TOKENIZERS_PARALLELISM"] = "true"

_STAGE_TIMES = {}
def _tic():
    return time.time()
def _toc(label, t0):
    dt = time.time() - t0
    _STAGE_TIMES[label] = dt
    print(f"[TIMER] {label}: {dt:.1f}s")

_T_TOTAL = _tic()

############################################################
# CELL 3 - Mount Google Drive
############################################################
from google.colab import drive
drive.mount('/content/drive')

############################################################
# CELL 4 - Configuration
############################################################
folder = "/content/drive/MyDrive/data"
TEST_MODE = False
TEST_ROWS = 5

CFG = {
    'EMBED_MODEL':            'sentence-transformers/all-MiniLM-L6-v2',
    'MAX_SEQ_LEN':             96,
    'EMBED_BATCH_SIZE':        256,
    'TEXT_WORD_LIMIT':         128,
    'EMBED_TOP_K':             TEST_ROWS if TEST_MODE else 4000,
    'WEIGHT_ROLE':              0.40,
    'WEIGHT_EXPERIENCE':        0.25,
    'WEIGHT_SKILLS':            0.15,
    'WEIGHT_LOCATION':          0.10,
    'WEIGHT_EDUCATION':         0.05,
    'WEIGHT_GITHUB':            0.05,
    'SKILL_SATURATION':         6.0,
    'ROLE_D_KEEP_MULTIPLIER':   0.02,
    'TODAY':                    datetime.now(),
    'TOP_N':                    TEST_ROWS if TEST_MODE else 100,
    'USE_INT8':                 True,   # set False for fp32 (safer ranking parity, still fast)
    'PARALLEL_MIN_ROWS':        1500,   # below this, process-pool overhead isn't worth it (lowered)
    'CACHE_EMBEDDINGS':         True,   # skip re-encoding candidates whose text hasn't changed

    # --- NEW: caps the embedding stage, which is the true bottleneck on 100k rows ---
    # Only the top-N (by role_fit_score) embed-eligible candidates get encoded.
    # role_fit_score already carries the heaviest weight (0.40) in the final score,
    # so candidates cut here have near-zero chance of reaching the top 100 anyway.
    # Set to None to disable the cap (embed everyone, original behavior).
    'EMBED_MAX_CANDIDATES':     25000,
}

print(f"TEST_MODE = {TEST_MODE}  (rows = {TEST_ROWS if TEST_MODE else 'ALL'})")
print(f"CPU cores detected: {_CPU_CORES}")

############################################################
# CELL 5 - Path Resolution
############################################################
FOLDER = Path(folder)

def find_file(stem_candidates, extensions):
    for stem in stem_candidates:
        for ext in extensions:
            p = FOLDER / f"{stem}{ext}"
            if p.exists():
                return p
    tried = [str(FOLDER / f"{s}{e}") for s in stem_candidates for e in extensions]
    existing = "\n".join(sorted(p.name for p in FOLDER.iterdir())) if FOLDER.exists() else "FOLDER NOT FOUND"
    raise FileNotFoundError("Could not find any of:\n" + "\n".join(tried) + f"\n\nFolder contents:\n{existing}")

_cand_path = FOLDER / 'candidates.jsonl'
if not _cand_path.exists():
    raise FileNotFoundError(f"candidates.jsonl not found in: {FOLDER}")

PATHS = {
    'candidates_jsonl':  _cand_path,
    'job_description':   find_file(['job_description'],     ['.md', '.txt', '.docx']),
    'signals_doc':       find_file(['redrob_signals_doc'],  ['.md', '.txt', '.docx']),
    'submission_spec':   find_file(['submission_spec'],     ['.md', '.txt', '.docx']),
    'candidate_schema':  find_file(['candidate_schema'],    ['.json']),
    'sample_candidates': find_file(['sample_candidates'],   ['.json']),
    'sample_submission': find_file(['sample_submission'],   ['.csv']),
    'validator':         find_file(['validate_submission'], ['.py']),
}

OUT_DIR = Path('/content/redrob_cache')
OUT_DIR.mkdir(parents=True, exist_ok=True)

suffix = 'test' if TEST_MODE else 'full'
PATHS['submission']       = OUT_DIR / f'{suffix}_submission.csv'
PATHS['submission_drive'] = FOLDER  / ('test_submission.csv' if TEST_MODE else 'submission.csv')
PATHS['embed_cache']      = OUT_DIR / 'embed_cache.npz'

print("Resolved paths:")
for k, v in PATHS.items():
    print(f"  {k}: {v}")

############################################################
# CELL 6 - Read Text Docs (negligible cost, unchanged)
############################################################
def read_text_doc(path):
    if path.suffix.lower() == '.docx':
        doc = Document(str(path))
        parts = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
        for table in doc.tables:
            for row in table.rows:
                cells = [c.text.strip() for c in row.cells if c.text.strip()]
                if cells:
                    parts.append(' | '.join(cells))
        return '\n'.join(parts)
    return path.read_text(encoding='utf-8')

JD_TEXT              = read_text_doc(PATHS['job_description'])
SIGNALS_DOC_TEXT     = read_text_doc(PATHS['signals_doc'])
SUBMISSION_SPEC_TEXT = read_text_doc(PATHS['submission_spec'])

with open(PATHS['candidate_schema'], 'r', encoding='utf-8') as f:
    CANDIDATE_SCHEMA = json.load(f)

print(f"JD loaded: {len(JD_TEXT)} chars")

############################################################
# CELL 7 - Helper Functions
############################################################
def safe_str(v):
    return '' if v is None or (isinstance(v, float) and math.isnan(v)) else str(v)

def parse_date(value):
    if not value:
        return None
    try:
        dt = pd.to_datetime(value, errors='coerce')
        if pd.isna(dt):
            return None
        return dt.to_pydatetime()
    except Exception:
        return None

def truncate_words(text, limit):
    words = safe_str(text).split()
    return ' '.join(words[:limit])

############################################################
# CELL 8 - Load candidates.jsonl -> Flattened DataFrame
############################################################
def flatten_candidate(obj):
    profile   = obj.get('profile') or {}
    career    = obj.get('career_history') or []
    education = obj.get('education') or []
    skills    = obj.get('skills') or []
    signals   = obj.get('redrob_signals') or {}

    career_titles = ' '.join(safe_str(j.get('title')) for j in career if isinstance(j, dict))
    career_desc   = ' '.join(safe_str(j.get('description')) for j in career if isinstance(j, dict))

    candidate_text_parts = [profile.get('headline'), profile.get('summary'),
                             profile.get('current_title'), profile.get('current_industry')]
    for job in career:
        if isinstance(job, dict):
            candidate_text_parts.extend([job.get('title'), job.get('industry'), job.get('description')])
    candidate_text = truncate_words(' '.join(safe_str(p) for p in candidate_text_parts if p), CFG['TEXT_WORD_LIMIT'])

    return {
        'candidate_id':              obj.get('candidate_id'),
        'current_title':             safe_str(profile.get('current_title')),
        'headline':                  safe_str(profile.get('headline')),
        'summary':                   safe_str(profile.get('summary')),
        'location':                  safe_str(profile.get('location')),
        'country':                   safe_str(profile.get('country')),
        'years_of_experience':       float(profile.get('years_of_experience') or 0.0),
        'current_company':           safe_str(profile.get('current_company')),
        'current_company_size':      safe_str(profile.get('current_company_size')),
        'current_industry':          safe_str(profile.get('current_industry')),
        'career_history':            career,
        'education':                 education,
        'skills':                    skills,
        'redrob_signals':            signals,
        'career_text':                truncate_words(career_desc, CFG['TEXT_WORD_LIMIT']),
        'career_titles_text':        career_titles,
        'candidate_text':            candidate_text,
        'last_active_date':          signals.get('last_active_date'),
        'notice_period_days':        signals.get('notice_period_days'),
        'recruiter_response_rate':   signals.get('recruiter_response_rate'),
        'interview_completion_rate': signals.get('interview_completion_rate'),
        'open_to_work_flag':         signals.get('open_to_work_flag'),
        'preferred_work_mode':       signals.get('preferred_work_mode'),
        'willing_to_relocate':       signals.get('willing_to_relocate'),
        'github_activity_score':     signals.get('github_activity_score'),
        'skill_assessment_scores':   signals.get('skill_assessment_scores') or {},
    }

def _flatten_chunk(lines):
    return [flatten_candidate(orjson.loads(l)) for l in lines]

def chunk_list(items, n_chunks):
    n_chunks = max(1, n_chunks)
    chunk_size = math.ceil(len(items) / n_chunks)
    return [items[i:i + chunk_size] for i in range(0, len(items), chunk_size)]

def parallel_row_apply(records, chunk_fn, desc):
    """Generic parallel map over a list of dict records, chunked across cores.
    Used for every heavy per-row Python loop in this pipeline (honeypot, role-fit, etc)."""
    if len(records) < CFG['PARALLEL_MIN_ROWS']:
        return chunk_fn(records)
    chunks = chunk_list(records, _CPU_CORES)
    with ProcessPoolExecutor(max_workers=_CPU_CORES) as executor:
        results = list(
            tqdm(
                executor.map(chunk_fn, chunks),
                total=len(chunks),
                desc=desc,
                mininterval=1.0
            )
        )
    out = []
    for r in results:
        out.extend(r)
    return out

def load_candidates_to_df(limit=None):
    t0 = time.time()
    with open(PATHS['candidates_jsonl'], 'rb') as f:
        raw = f.read()
    lines = [l for l in raw.splitlines() if l.strip()]
    if limit:
        lines = lines[:limit]

    if len(lines) >= CFG['PARALLEL_MIN_ROWS']:
        chunks = chunk_list(lines, _CPU_CORES)
        with ProcessPoolExecutor(max_workers=_CPU_CORES) as executor:
            chunk_results = list(
                tqdm(
                    executor.map(_flatten_chunk, chunks),
                    total=len(chunks),
                    desc="Loading candidates (parallel)",
                    mininterval=1.0
                )
            )
        rows = [r for chunk in chunk_results for r in chunk]
    else:
        rows = [flatten_candidate(orjson.loads(l)) for l in lines]

    df = pd.DataFrame(rows)
    df['years_of_experience'] = df['years_of_experience'].astype('float32')
    print(f'Loaded {len(df)} candidates in {time.time()-t0:.1f}s')
    return df

_t = _tic()
df = load_candidates_to_df(limit=TEST_ROWS if TEST_MODE else None)
_toc('Cell8_load_candidates', _t)

############################################################
# CELL 9 - Vectorized Role-Tier Classification (unchanged, already fast)
############################################################
TIER_A_PATTERN = r'\b(machine learning engineer|ml engineer|ai engineer|applied scientist|research engineer|recommendation engineer|ranking engineer|recommender systems engineer|search engineer|nlp engineer|computer vision engineer)\b'
TIER_B_PATTERN = r'\b(software engineer|backend engineer|back end engineer|data engineer|platform engineer|systems engineer|python developer|data scientist)\b'
TIER_C_PATTERN = r'\b(full.?stack|frontend|front end|software developer|backend developer|developer)\b'
TIER_D_PATTERN = r'\b(hr|human resources|recruiter|marketing manager|sales executive|content writer|graphic designer|accountant|finance|civil engineer|mechanical engineer|electrical engineer|operations manager|customer support|business analyst|project manager|teacher|nurse|doctor|legal|lawyer)\b'
ML_EVIDENCE_PATTERN = r'\b(machine learning|deep learning|nlp|bert|transformer|embedding|vector database|retrieval|ranking|recommendation|recommender|rag|llm|model serving|feature store|pytorch|tensorflow|xgboost|scikit|search relevance)\b'

def compute_role_tier_vectorized(dframe):
    title_lc = dframe['current_title'].str.lower()
    evidence_lc = (dframe['summary'] + ' ' + dframe['career_text']).str.lower()

    has_d = title_lc.str.contains(TIER_D_PATTERN, regex=True, na=False)
    has_a = title_lc.str.contains(TIER_A_PATTERN, regex=True, na=False)
    is_ds = title_lc.str.contains(r'\bdata scientist\b', regex=True, na=False)
    has_ml_evidence = evidence_lc.str.contains(ML_EVIDENCE_PATTERN, regex=True, na=False)
    has_b = title_lc.str.contains(TIER_B_PATTERN, regex=True, na=False)
    has_c = title_lc.str.contains(TIER_C_PATTERN, regex=True, na=False)

    is_d_final = has_d & ~has_a
    is_a_final = has_a
    is_ds_ml   = is_ds & has_ml_evidence & ~is_a_final
    is_b_final = has_b & has_ml_evidence & ~is_a_final & ~is_ds_ml
    is_c_final = has_c & ~is_a_final & ~is_ds_ml & ~is_b_final

    tier = np.select(
        [is_d_final, is_a_final, is_ds_ml, is_b_final, is_c_final],
        ['D', 'A', 'A', 'B', 'C'],
        default='D'
    )
    title_score = np.select(
        [is_d_final, is_a_final, is_ds_ml, is_b_final, is_c_final],
        [0.0, 1.0, 0.9, 0.7, 0.35],
        default=0.0
    )
    return tier, title_score

def role_title_tier_single(title, evidence_text=''):
    """Used for scoring recent career_history entries (shortlist + role-fit pass)."""
    title = safe_str(title)
    if re.search(TIER_D_PATTERN, title, re.I) and not re.search(TIER_A_PATTERN, title, re.I):
        return 'D', 0.0
    if re.search(TIER_A_PATTERN, title, re.I):
        return 'A', 1.0
    ev = safe_str(evidence_text)
    if re.search(r'\bdata scientist\b', title, re.I) and re.search(ML_EVIDENCE_PATTERN, ev, re.I):
        return 'A', 0.9
    if re.search(TIER_B_PATTERN, title, re.I) and re.search(ML_EVIDENCE_PATTERN, ev, re.I):
        return 'B', 0.7
    if re.search(TIER_C_PATTERN, title, re.I):
        return 'C', 0.35
    return 'D', 0.0

_t = _tic()
_tier_arr, _score_arr = compute_role_tier_vectorized(df)
df['role_tier'] = _tier_arr
df['role_title_score'] = _score_arr
_toc('Cell9_role_tier', _t)

print(df['role_tier'].value_counts())

############################################################
# CELL 10 - Honeypot Detection + Role-Fit Scoring
# (MERGED into one parallel pass, was two separate passes with two
#  separate `.to_dict('records')` conversions over the same rows —
#  that conversion alone is expensive at 100k rows and was being paid twice)
############################################################
def honeypot_check(row):
    zero_expert = sum(
        1 for s in (row['skills'] or [])
        if safe_str(s.get('proficiency')).lower() in {'advanced', 'expert'}
        and int(s.get('duration_months') or 0) == 0
    )
    if zero_expert >= 3:
        return True

    yoe = row['years_of_experience']
    career = row['career_history'] or []

    starts = [parse_date(j.get('start_date')) for j in career]
    starts = [d for d in starts if d is not None]
    if starts:
        earliest_span_years = max(0.0, (CFG['TODAY'] - min(starts)).days / 365.25)
        if earliest_span_years + 3.0 < yoe:
            return True

    if yoe > 0:
        for j in career:
            if int(j.get('duration_months') or 0) > yoe * 12 + 1:
                return True

    return False

def _role_fit_single(row):
    title_score = row['role_title_score']
    recent_jobs = list(row['career_history'] or [])[:3]
    recent_scores = [role_title_tier_single(j.get('title'), safe_str(j.get('description')))[1] for j in recent_jobs]
    identity = max([title_score] + recent_scores) if recent_scores else title_score
    return float(np.clip(0.7 * title_score + 0.3 * identity, 0, 1))

def _honeypot_and_rolefit(row):
    return honeypot_check(row), _role_fit_single(row)

def _hp_rf_chunk(records):
    return [_honeypot_and_rolefit(r) for r in records]

_t = _tic()
non_d_mask = df['role_tier'] != 'D'
non_d_records = df.loc[non_d_mask].to_dict('records')  # single conversion, reused below

combined_results = parallel_row_apply(non_d_records, _hp_rf_chunk, "Honeypot+RoleFit (parallel)")

df['honeypot_suspect'] = False
df['role_fit_score'] = df['role_title_score']

if combined_results:
    hp_vals, rf_vals = zip(*combined_results)
    df.loc[non_d_mask, 'honeypot_suspect'] = hp_vals
    df.loc[non_d_mask, 'role_fit_score'] = rf_vals

_toc('Cell10_honeypot_rolefit', _t)
print(f"Honeypot suspects flagged: {int(df['honeypot_suspect'].sum())} / {len(df)}")

############################################################
# CELL 11 - (folded into Cell 10 above, kept as no-op for numbering parity)
############################################################

############################################################
# CELL 12 - Experience Validity Scoring (only runs on final shortlist, unchanged)
############################################################
CONCEPT_PATTERNS = [
    re.compile(r'\b(embedding|sentence transformer|semantic search|vector search|vector database|pinecone|weaviate|qdrant|milvus|faiss|retrieval)\b', re.I),
    re.compile(r'\b(ndcg|mrr|map@|ranking|reranking|learning to rank|search relevance)\b', re.I),
    re.compile(r'\b(python|fastapi|flask|django|docker|kubernetes|microservice|production|deployed|mlops)\b', re.I),
    re.compile(r'\b(hybrid search|bm25|elasticsearch|opensearch|lucene)\b', re.I),
    re.compile(r'\b(llm|large language model|fine.?tun|lora|qlora|peft|rag|langchain)\b', re.I),
    re.compile(r'\b(pytorch|tensorflow|scikit|xgboost|transformer|bert|neural network|deep learning)\b', re.I),
]
N_CONCEPTS = len(CONCEPT_PATTERNS)
PRODUCTION_PATTERN = re.compile(r'\b(deployed|production|scaled|served|serving|latency|live|monitoring|a/b test|sla|throughput)\b', re.I)

def role_months_score(career_history):
    months = 0.0
    today = CFG['TODAY']
    for job in career_history or []:
        _, s = role_title_tier_single(job.get('title'), safe_str(job.get('description')))
        if s >= 0.7:
            end = parse_date(job.get('end_date')) or today
            age = max(0.0, (today - end).days / 365.25)
            rw = 1.0 if age <= 3 else max(0.35, 1.0 - 0.12 * (age - 3))
            months += int(job.get('duration_months') or 0) * rw * s
    return float(np.clip(1.0 - abs(months - 72) / 72, 0, 1))

def experience_validity_score(row):
    text = safe_str(row.get('career_text')) + ' ' + safe_str(row.get('summary'))
    hits = sum(1 for p in CONCEPT_PATTERNS if p.search(text))
    yoe = float(row.get('years_of_experience') or 0.0)
    ys = 1.0 if 5 <= yoe <= 9 else (max(0.0, yoe / 5) if yoe < 5 else max(0.35, 1.0 - (yoe - 9) / 12))
    prod = 0.08 if PRODUCTION_PATTERN.search(text) else 0.0
    return float(np.clip(
        0.45 * (hits / N_CONCEPTS) + 0.35 * role_months_score(row.get('career_history')) + 0.20 * ys + prod,
        0, 1
    ))

############################################################
# CELL 13 - Skill Relevance Scoring (unchanged, cheap, shortlist only)
############################################################
SKILL_TAXONOMY = {
    'python': 0.9, 'pandas': 0.35, 'numpy': 0.35, 'scikit-learn': 0.55, 'sklearn': 0.55,
    'machine learning': 1.0, 'ml': 0.8, 'deep learning': 0.9, 'nlp': 1.0,
    'pytorch': 0.85, 'tensorflow': 0.75, 'transformers': 0.95, 'bert': 0.9,
    'embeddings': 1.0, 'sentence transformers': 1.0, 'semantic search': 1.0,
    'vector database': 1.0, 'pinecone': 0.95, 'weaviate': 0.95, 'qdrant': 0.95,
    'milvus': 0.95, 'faiss': 0.9, 'elasticsearch': 0.75, 'opensearch': 0.75,
    'bm25': 0.8, 'hybrid search': 1.0, 'ranking': 0.9, 'learning to rank': 1.0,
    'ndcg': 1.0, 'mrr': 1.0, 'rag': 0.9, 'langchain': 0.65, 'llm': 0.85,
    'fine tuning': 0.8, 'lora': 0.85, 'qlora': 0.85, 'peft': 0.85, 'mlops': 0.75,
    'docker': 0.35, 'kubernetes': 0.45, 'fastapi': 0.35,
}
PROFICIENCY_WEIGHT = {'beginner': 0.3, 'intermediate': 0.6, 'advanced': 0.85, 'expert': 1.0}

def skill_relevance_value(name):
    n = safe_str(name).lower().strip()
    if not n:
        return 0.0
    if n in SKILL_TAXONOMY:
        return SKILL_TAXONOMY[n]
    for key, w in SKILL_TAXONOMY.items():
        if key in n or n in key:
            return w * 0.85
    return 0.0

def duration_trust(months):
    m = int(months or 0)
    if m == 0:
        return 0.1
    if m <= 6:
        return 0.5
    if m <= 12:
        return 0.75
    return 1.0

def skill_relevance_score(row):
    total = 0.0
    assessments = {
        safe_str(k).lower(): float(v)
        for k, v in (row.get('skill_assessment_scores') or {}).items() if v is not None
    }
    for skill in row.get('skills') or []:
        rel = skill_relevance_value(skill.get('name'))
        if rel <= 0:
            continue
        key = safe_str(skill.get('name')).lower()
        if key in assessments:
            trust = np.clip(assessments[key] / 100.0, 0, 1)
        else:
            trust = PROFICIENCY_WEIGHT.get(safe_str(skill.get('proficiency')).lower(), 0.45) * duration_trust(skill.get('duration_months'))
        total += rel * trust
    return float(1.0 - math.exp(-total / CFG['SKILL_SATURATION']))

############################################################
# CELL 14 - Location, Education, Availability, Github Scoring (unchanged)
############################################################
def location_logistics_score(row):
    loc = safe_str(row.get('location')).lower()
    country = safe_str(row.get('country')).lower()
    if re.search(r'\b(pune|noida)\b', loc):
        ls = 1.0
    elif re.search(r'\b(bangalore|bengaluru|hyderabad|mumbai|delhi|gurgaon|gurugram|ncr)\b', loc):
        ls = 0.85
    elif country in {'india', 'in'} or 'india' in loc:
        ls = 0.6
    else:
        ls = 0.5 if bool(row.get('willing_to_relocate')) else 0.2
    ms = {'hybrid': 1.0, 'flexible': 1.0, 'onsite': 0.8, 'remote': 0.6}.get(
        safe_str(row.get('preferred_work_mode')).lower(), 0.75
    )
    return float(0.7 * ls + 0.3 * ms)

RELEVANT_EDU_PATTERN = re.compile(
    r'\b(computer|cs|software|information technology|artificial intelligence|'
    r'machine learning|data science|statistics|mathematics|math|electronics|ece|analytics)\b', re.I
)

def education_score(row):
    edu = row.get('education') or []
    if not edu:
        return 0.5
    best = 0.4
    for e in edu:
        field = safe_str(e.get('degree')) + ' ' + safe_str(e.get('field_of_study'))
        rel = bool(RELEVANT_EDU_PATTERN.search(field))
        tier = safe_str(e.get('tier')).lower()
        s = 1.0 if rel and tier == 'tier_1' else 0.8 if rel and tier == 'tier_2' else 0.65 if rel else 0.4
        best = max(best, s)
    return float(best)

def notice_period_factor(days):
    try:
        days = float(days)
    except Exception:
        return 0.75
    if days <= 30:
        return 1.0
    if days <= 60:
        return 0.8
    if days <= 90:
        return 0.6
    return 0.45

def recency_factor(last_active_date):
    dt = parse_date(last_active_date)
    if dt is None:
        return 0.5
    days = max(0, (CFG['TODAY'] - dt).days)
    if days <= 30:
        return 1.0
    if days <= 60:
        return 0.9
    if days <= 90:
        return 0.75
    if days <= 180:
        return 0.5
    return 0.25

def availability_multiplier(row):
    rr = float(row.get('recruiter_response_rate') or 0.5)
    ic = float(row.get('interview_completion_rate') or 0.5)
    s = recency_factor(row.get('last_active_date')) * notice_period_factor(row.get('notice_period_days'))
    s *= (0.5 + 0.5 * np.clip(rr, 0, 1)) * (0.5 + 0.5 * np.clip(ic, 0, 1))
    if bool(row.get('open_to_work_flag')):
        s *= 1.15
    return float(np.clip(s, 0, 1))

def github_bonus(row):
    try:
        score = float(row.get('github_activity_score'))
    except Exception:
        return 0.0
    if score >= 70:
        return 0.05
    if score >= 40:
        return 0.02
    return 0.0

############################################################
# CELL 15 - Composite Scoring Function (unchanged)
############################################################
def score_candidate_row(row):
    base = (
        CFG['WEIGHT_ROLE']       * row.get('role_fit_score', 0.0)
        + CFG['WEIGHT_EXPERIENCE'] * experience_validity_score(row)
        + CFG['WEIGHT_SKILLS']     * skill_relevance_score(row)
        + CFG['WEIGHT_LOCATION']   * location_logistics_score(row)
        + CFG['WEIGHT_EDUCATION']  * education_score(row)
        + CFG['WEIGHT_GITHUB']     * github_bonus(row)
    )
    final = base * availability_multiplier(row)
    if row.get('role_tier') == 'D':
        final *= CFG['ROLE_D_KEEP_MULTIPLIER']
    return float(final)

############################################################
# CELL 16 - Embedding Generation (the real bottleneck fix)
#
# On top of the original ONNX/int8 fix, this version also CAPS how many
# candidates get embedded (CFG['EMBED_MAX_CANDIDATES']), prioritized by
# role_fit_score. Encoding is O(n) in pool size, so if 100k rows shrink to
# e.g. 60k after Tier-D/honeypot filtering, that's still 60k forward passes.
# Capping to the top ~25k by role_fit_score keeps encode time bounded while
# barely touching the final top-100 (role weight = 0.40, the heaviest term).
############################################################
from optimum.onnxruntime import ORTModelForFeatureExtraction, ORTQuantizer, AutoQuantizationConfig
from transformers import AutoTokenizer
import onnxruntime as ort
import hashlib

ONNX_DIR  = OUT_DIR / "onnx_fp32"
QUANT_DIR = OUT_DIR / "onnx_int8"


def prepare_onnx_model(model_id, use_int8):
    target_dir = QUANT_DIR if use_int8 else ONNX_DIR
    model_file = "model_quantized.onnx" if use_int8 else "model.onnx"

    if not (target_dir / model_file).exists():
        target_dir.mkdir(parents=True, exist_ok=True)
        base_model = ORTModelForFeatureExtraction.from_pretrained(model_id, export=True)
        base_model.save_pretrained(ONNX_DIR)
        if use_int8:
            quantizer = ORTQuantizer.from_pretrained(ONNX_DIR)
            qconfig = AutoQuantizationConfig.avx2(is_static=False, per_channel=False)
            quantizer.quantize(save_dir=QUANT_DIR, quantization_config=qconfig)

    sess_opts = ort.SessionOptions()
    sess_opts.intra_op_num_threads = _CPU_CORES
    sess_opts.inter_op_num_threads = 1
    sess_opts.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
    sess_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    sess_opts.enable_mem_pattern = True
    sess_opts.enable_cpu_mem_arena = True

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    ort_model = ORTModelForFeatureExtraction.from_pretrained(
        target_dir,
        file_name=model_file,
        provider="CPUExecutionProvider",
        session_options=sess_opts,
        use_io_binding=False,
    )
    return tokenizer, ort_model


def mean_pool_normalize(last_hidden_state, attention_mask):
    mask = attention_mask[..., None].astype(np.float32)
    summed = (last_hidden_state * mask).sum(axis=1)
    counts = np.clip(mask.sum(axis=1), 1e-9, None)
    pooled = summed / counts
    norms = np.clip(np.linalg.norm(pooled, axis=1, keepdims=True), 1e-9, None)
    return (pooled / norms).astype(np.float32)


def encode_texts(texts, tokenizer, ort_model, batch_size, max_length, desc="Embedding"):
    n = len(texts)
    if n == 0:
        return np.zeros((0, ort_model.config.hidden_size), dtype='float32')

    order = np.argsort([-len(t) for t in texts])
    sorted_texts = [texts[i] for i in order]
    num_batches = math.ceil(n / batch_size)

    def tokenize_batch(bidx):
        chunk = sorted_texts[bidx * batch_size:(bidx + 1) * batch_size]
        return tokenizer(chunk, padding=True, truncation=True, max_length=max_length, return_tensors="np")

    results = [None] * num_batches
    with ThreadPoolExecutor(max_workers=2) as tok_pool:
        prefetch_depth = 2
        futures = {i: tok_pool.submit(tokenize_batch, i) for i in range(min(prefetch_depth, num_batches))}
        next_submit = prefetch_depth
        pbar = tqdm(total=num_batches, desc=desc, mininterval=1.0)
        for i in range(num_batches):
            enc = futures.pop(i).result()
            if next_submit < num_batches:
                futures[next_submit] = tok_pool.submit(tokenize_batch, next_submit)
                next_submit += 1

            out = ort_model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"])
            pooled = mean_pool_normalize(out.last_hidden_state, enc["attention_mask"])
            results[i] = pooled
            pbar.update(1)
        pbar.close()

    embeddings_sorted = np.vstack(results).astype('float32')
    embeddings = np.empty_like(embeddings_sorted)
    embeddings[order] = embeddings_sorted
    return embeddings


def load_embed_cache():
    if not CFG['CACHE_EMBEDDINGS'] or not PATHS['embed_cache'].exists():
        return {}
    try:
        data = np.load(PATHS['embed_cache'], allow_pickle=False)
        return {h: data[h] for h in data.files}
    except Exception:
        return {}

def save_embed_cache(cache_dict):
    if not CFG['CACHE_EMBEDDINGS']:
        return
    try:
        np.savez_compressed(PATHS['embed_cache'], **cache_dict)
    except Exception as e:
        print('Could not save embedding cache:', e)

def text_hash(t):
    return hashlib.blake2b(t.encode('utf-8'), digest_size=16).hexdigest()


_t = _tic()

embed_mask = (~df['honeypot_suspect']) & (df['role_tier'] != 'D')
embed_df = df[embed_mask].copy()
print(f'Embed-eligible after honeypot/Tier-D filter: {len(embed_df)}/{len(df)}')

cap = CFG.get('EMBED_MAX_CANDIDATES')
if cap and len(embed_df) > cap:
    embed_df = embed_df.sort_values('role_fit_score', ascending=False).head(cap)
    print(f'Capped embedding pool to top {cap} by role_fit_score')

print(f'Embedding {len(embed_df)} candidates')

tokenizer, ort_model = prepare_onnx_model(CFG['EMBED_MODEL'], CFG['USE_INT8'])

corpus = embed_df['candidate_text'].fillna('').astype(str).tolist()
hashes = [text_hash(t) for t in corpus]

cache = load_embed_cache()
to_encode_idx = [i for i, h in enumerate(hashes) if h not in cache]
print(f'{len(corpus) - len(to_encode_idx)} candidates hit cache, encoding {len(to_encode_idx)} fresh')

if to_encode_idx:
    fresh_texts = [corpus[i] for i in to_encode_idx]
    fresh_embeddings = encode_texts(
        fresh_texts, tokenizer, ort_model,
        batch_size=CFG['EMBED_BATCH_SIZE'],
        max_length=CFG['MAX_SEQ_LEN'],
        desc='Embedding candidates',
    )
    for local_i, global_i in enumerate(to_encode_idx):
        cache[hashes[global_i]] = fresh_embeddings[local_i]
    save_embed_cache(cache)

embeddings = np.vstack([cache[h] for h in hashes]).astype('float32')
embed_index_map = embed_df.index.to_numpy()

_toc('Cell16_embedding', _t)

############################################################
# CELL 17 - JD Embedding + Fast Top-K Retrieval
############################################################
_t = _tic()

jd_embedding = encode_texts([JD_TEXT], tokenizer, ort_model,
                             batch_size=1, max_length=CFG['MAX_SEQ_LEN'], desc='Embedding JD')[0]

def top_k_indices(scores, k):
    k = min(k, len(scores))
    if k <= 0:
        return np.array([], dtype=int)
    part = np.argpartition(-scores, k - 1)[:k]
    return part[np.argsort(-scores[part])]

embed_scores = (embeddings @ jd_embedding).reshape(-1)
top_local = top_k_indices(embed_scores, CFG['EMBED_TOP_K'])
candidate_positions = embed_index_map[top_local]

print(f'Retrieved {len(candidate_positions)} candidates for full scoring')
_toc('Cell17_jd_retrieval', _t)

############################################################
# CELL 18 - Score Shortlist and Rank (only ~4000 rows, cheap, unchanged)
############################################################
_t = _tic()
shortlist = df.loc[candidate_positions].copy()
shortlist['score'] = [score_candidate_row(r) for r in shortlist.to_dict('records')]

top_n = CFG['TOP_N']
ranked = shortlist.sort_values(['score', 'candidate_id'], ascending=[False, True]).head(top_n).copy()
ranked['rank'] = np.arange(1, len(ranked) + 1)
_toc('Cell18_shortlist_scoring', _t)

print(f"Honeypots in top {top_n}: {int(ranked['honeypot_suspect'].sum())}")
print(f"Tier-D in top {top_n}: {int((ranked['role_tier'] == 'D').sum())}")
print('\n--- TOP RESULTS ---')
print(ranked[['candidate_id', 'current_title', 'role_tier', 'score']].to_string(index=False))

############################################################
# CELL 19 - Reasoning Generation (unchanged, cheap, shortlist only)
############################################################
def top_relevant_skills(row, n=2):
    scored_skills = sorted(
        [(skill_relevance_value(s.get('name')), safe_str(s.get('name')))
         for s in row.get('skills') or [] if skill_relevance_value(s.get('name')) > 0],
        key=lambda x: (-x[0], x[1].lower())
    )
    seen, names = set(), []
    for _, name in scored_skills:
        if name.lower() not in seen:
            seen.add(name.lower())
            names.append(name)
        if len(names) >= n:
            break
    return names

def notice_note(row):
    days = row.get('notice_period_days')
    if days is None or (isinstance(days, float) and math.isnan(days)):
        return 'notice period is not listed'
    days = int(days)
    return f'{days}-day notice period fits the JD timeline' if days <= 30 else f'{days}-day notice period is a logistics concern'

def last_active_note(row):
    dt = parse_date(row.get('last_active_date'))
    if dt is None:
        return 'activity recency is not available'
    days = max(0, (CFG['TODAY'] - dt).days)
    if days <= 30:
        return f'active recently ({days} days ago)'
    if days <= 90:
        return f'last active {days} days ago'
    return f'activity is stale at {days} days since last active'

def generate_reasoning(row):
    title = safe_str(row.get('current_title')) or 'candidate'
    yoe = float(row.get('years_of_experience') or 0.0)
    skills = top_relevant_skills(row)
    skill_text = ', '.join(skills) if skills else 'relevant ML/search signals in career history'

    concern = None
    if notice_period_factor(row.get('notice_period_days')) < 0.8:
        concern = notice_note(row) + '.'
    elif recency_factor(row.get('last_active_date')) < 0.75:
        concern = last_active_note(row).capitalize() + '.'
    elif row.get('role_tier') == 'C':
        concern = 'Role identity is adjacent rather than a direct ML/search engineering match.'

    v = int(row.get('rank', 0)) % 4
    if v == 0:
        text = f'{title} with {yoe:.1f} years shows fit through {skill_text}; {notice_note(row)} and {last_active_note(row)}.'
    elif v == 1:
        text = f'Current title is {title}, with {yoe:.1f} years and strongest skills around {skill_text}. Availability: {notice_note(row)}.'
    elif v == 2:
        text = f'{title} ranked for ML/search-adjacent career evidence plus {skill_text}. {last_active_note(row).capitalize()}.'
    else:
        text = f'Fit from {title} role identity, {yoe:.1f} years experience, skills including {skill_text}. {notice_note(row).capitalize()}.'

    if concern and concern not in text:
        text += ' ' + concern

    return re.sub(r'\s+', ' ', text).strip()[:900]

ranked['reasoning'] = ranked.apply(generate_reasoning, axis=1)

############################################################
# CELL 20 - Build and Write Submission CSV (unchanged)
############################################################
submission = ranked[['candidate_id', 'rank', 'score', 'reasoning']].copy()
submission['score'] = submission['score'].astype(float)
submission = submission.sort_values('rank').reset_index(drop=True)

assert list(submission.columns) == ['candidate_id', 'rank', 'score', 'reasoning']
assert len(submission) == top_n
if not TEST_MODE:
    assert submission['rank'].tolist() == list(range(1, 101))

submission.to_csv(PATHS['submission'], index=False, encoding='utf-8')
print(f'Submission written: {PATHS["submission"]}')
print(submission[['candidate_id', 'rank', 'score']].head(10).to_string(index=False))

############################################################
# CELL 21 - Validate Submission (unchanged)
############################################################
if not TEST_MODE:
    result = subprocess.run(
        ['python', str(PATHS['validator']), str(PATHS['submission'])],
        capture_output=True, text=True
    )
    print('Validator stdout:', result.stdout)
    print('Validator stderr:', result.stderr)
    print('Validator return code:', result.returncode)
    if result.returncode != 0:
        raise RuntimeError("Submission failed validation. See validator output above.")

############################################################
# CELL 22 - Copy to Drive, Download, and clean up
############################################################
import shutil

try:
    shutil.copy2(str(PATHS['submission']), str(PATHS['submission_drive']))
    print('Copied to Drive:', PATHS['submission_drive'])
except Exception as e:
    print('Could not copy to Drive:', e)

_toc('TOTAL_RUNTIME', _T_TOTAL)
print('\n--- STAGE TIME BREAKDOWN ---')
for k, v in _STAGE_TIMES.items():
    print(f'  {k}: {v:.1f}s')

from google.colab import files
files.download(str(PATHS['submission']))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.2/194.2 kB 6.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
Drive already mounted at /content/drive; to attempt to forcibly remount, c

Loading candidates (parallel):   0%|          | 0/2 [00:02<?, ?it/s]

Loaded 100000 candidates in 69.5s
[TIMER] Cell8_load_candidates: 70.1s
[TIMER] Cell9_role_tier: 10.1s
role_tier
D    81287
C    13965
B     3867
A      881
Name: count, dtype: int64


Honeypot+RoleFit (parallel):   0%|          | 0/2 [00:01<?, ?it/s]

[TIMER] Cell10_honeypot_rolefit: 23.9s
Honeypot suspects flagged: 26 / 100000


...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

0 candidates hit cache, encoding 18687 fresh


Embedding candidates:   0%|          | 0/73 [00:00<?, ?it/s]

In [1]:
############################################################
# CELL 1 - Install Dependencies
############################################################
!pip -q install "optimum[onnxruntime]" orjson numpy pandas tqdm python-docx pyarrow

############################################################
# CELL 2 - Imports
############################################################
import json
import math
import os
import re
import subprocess
import time
import warnings
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from datetime import datetime
from pathlib import Path

import numpy as np
import orjson
import pandas as pd
from docx import Document
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# Must be set before onnxruntime / tokenizers spin up their thread pools.
_CPU_CORES = os.cpu_count() or 4
os.environ["OMP_NUM_THREADS"] = str(_CPU_CORES)
os.environ["TOKENIZERS_PARALLELISM"] = "true"

_STAGE_TIMES = {}
def _tic():
    return time.time()
def _toc(label, t0):
    dt = time.time() - t0
    _STAGE_TIMES[label] = dt
    print(f"[TIMER] {label}: {dt:.1f}s")

_T_TOTAL = _tic()

############################################################
# CELL 3 - Mount Google Drive
############################################################
from google.colab import drive
drive.mount('/content/drive')

############################################################
# CELL 4 - Configuration
#
# SPEED CHANGES vs original (only embedding-stage knobs touched — nothing
# that affects honeypot detection, role-tier logic, or the final scoring
# formula, so ranking behaviour stays the same for anyone who was already
# going to land in the shortlist):
#   - EMBED_MAX_CANDIDATES  25000 -> 8000   (biggest win, embedding is the
#                                            true bottleneck; role_fit_score
#                                            already carries 0.40 weight so
#                                            candidates cut here rarely had
#                                            a shot at top-100 anyway)
#   - MAX_SEQ_LEN           96    -> 64     (less compute per candidate;
#                                            candidate_text is already
#                                            truncated to 128 words upstream)
#   - EMBED_BATCH_SIZE      256   -> 512    (fewer forward-pass calls =
#                                            less fixed per-batch overhead)
############################################################
folder = "/content/drive/MyDrive/data"
TEST_MODE = False
TEST_ROWS = 5

CFG = {
    'EMBED_MODEL':            'sentence-transformers/all-MiniLM-L6-v2',
    'MAX_SEQ_LEN':             64,
    'EMBED_BATCH_SIZE':        512,
    'TEXT_WORD_LIMIT':         128,
    'EMBED_TOP_K':             TEST_ROWS if TEST_MODE else 4000,
    'WEIGHT_ROLE':              0.40,
    'WEIGHT_EXPERIENCE':        0.25,
    'WEIGHT_SKILLS':            0.15,
    'WEIGHT_LOCATION':          0.10,
    'WEIGHT_EDUCATION':         0.05,
    'WEIGHT_GITHUB':            0.05,
    'SKILL_SATURATION':         6.0,
    'ROLE_D_KEEP_MULTIPLIER':   0.02,
    'TODAY':                    datetime.now(),
    'TOP_N':                    TEST_ROWS if TEST_MODE else 100,
    'USE_INT8':                 True,   # set False for fp32 (safer ranking parity, still fast)
    'PARALLEL_MIN_ROWS':        1500,   # below this, process-pool overhead isn't worth it
    'CACHE_EMBEDDINGS':         True,   # skip re-encoding candidates whose text hasn't changed

    # Caps the embedding stage, which is the true bottleneck on 100k rows.
    # Only the top-N (by role_fit_score) embed-eligible candidates get encoded.
    # Set to None to disable the cap (embed everyone, original behavior).
    'EMBED_MAX_CANDIDATES':     8000,
}

print(f"TEST_MODE = {TEST_MODE}  (rows = {TEST_ROWS if TEST_MODE else 'ALL'})")
print(f"CPU cores detected: {_CPU_CORES}")

############################################################
# CELL 5 - Path Resolution
############################################################
FOLDER = Path(folder)

def find_file(stem_candidates, extensions):
    for stem in stem_candidates:
        for ext in extensions:
            p = FOLDER / f"{stem}{ext}"
            if p.exists():
                return p
    tried = [str(FOLDER / f"{s}{e}") for s in stem_candidates for e in extensions]
    existing = "\n".join(sorted(p.name for p in FOLDER.iterdir())) if FOLDER.exists() else "FOLDER NOT FOUND"
    raise FileNotFoundError("Could not find any of:\n" + "\n".join(tried) + f"\n\nFolder contents:\n{existing}")

_cand_path = FOLDER / 'candidates.jsonl'
if not _cand_path.exists():
    raise FileNotFoundError(f"candidates.jsonl not found in: {FOLDER}")

PATHS = {
    'candidates_jsonl':  _cand_path,
    'job_description':   find_file(['job_description'],     ['.md', '.txt', '.docx']),
    'signals_doc':       find_file(['redrob_signals_doc'],  ['.md', '.txt', '.docx']),
    'submission_spec':   find_file(['submission_spec'],     ['.md', '.txt', '.docx']),
    'candidate_schema':  find_file(['candidate_schema'],    ['.json']),
    'sample_candidates': find_file(['sample_candidates'],   ['.json']),
    'sample_submission': find_file(['sample_submission'],   ['.csv']),
    'validator':         find_file(['validate_submission'], ['.py']),
}

OUT_DIR = Path('/content/redrob_cache')
OUT_DIR.mkdir(parents=True, exist_ok=True)

suffix = 'test' if TEST_MODE else 'full'
PATHS['submission']       = OUT_DIR / f'{suffix}_submission.csv'
PATHS['submission_drive'] = FOLDER  / ('test_submission.csv' if TEST_MODE else 'submission.csv')
PATHS['embed_cache']      = OUT_DIR / 'embed_cache.npz'

print("Resolved paths:")
for k, v in PATHS.items():
    print(f"  {k}: {v}")

############################################################
# CELL 6 - Read Text Docs (negligible cost, unchanged)
############################################################
def read_text_doc(path):
    if path.suffix.lower() == '.docx':
        doc = Document(str(path))
        parts = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
        for table in doc.tables:
            for row in table.rows:
                cells = [c.text.strip() for c in row.cells if c.text.strip()]
                if cells:
                    parts.append(' | '.join(cells))
        return '\n'.join(parts)
    return path.read_text(encoding='utf-8')

JD_TEXT              = read_text_doc(PATHS['job_description'])
SIGNALS_DOC_TEXT     = read_text_doc(PATHS['signals_doc'])
SUBMISSION_SPEC_TEXT = read_text_doc(PATHS['submission_spec'])

with open(PATHS['candidate_schema'], 'r', encoding='utf-8') as f:
    CANDIDATE_SCHEMA = json.load(f)

print(f"JD loaded: {len(JD_TEXT)} chars")

############################################################
# CELL 7 - Helper Functions
############################################################
def safe_str(v):
    return '' if v is None or (isinstance(v, float) and math.isnan(v)) else str(v)

def parse_date(value):
    if not value:
        return None
    try:
        dt = pd.to_datetime(value, errors='coerce')
        if pd.isna(dt):
            return None
        return dt.to_pydatetime()
    except Exception:
        return None

def truncate_words(text, limit):
    words = safe_str(text).split()
    return ' '.join(words[:limit])

############################################################
# CELL 8 - Load candidates.jsonl -> Flattened DataFrame
############################################################
def flatten_candidate(obj):
    profile   = obj.get('profile') or {}
    career    = obj.get('career_history') or []
    education = obj.get('education') or []
    skills    = obj.get('skills') or []
    signals   = obj.get('redrob_signals') or {}

    career_titles = ' '.join(safe_str(j.get('title')) for j in career if isinstance(j, dict))
    career_desc   = ' '.join(safe_str(j.get('description')) for j in career if isinstance(j, dict))

    candidate_text_parts = [profile.get('headline'), profile.get('summary'),
                             profile.get('current_title'), profile.get('current_industry')]
    for job in career:
        if isinstance(job, dict):
            candidate_text_parts.extend([job.get('title'), job.get('industry'), job.get('description')])
    candidate_text = truncate_words(' '.join(safe_str(p) for p in candidate_text_parts if p), CFG['TEXT_WORD_LIMIT'])

    return {
        'candidate_id':              obj.get('candidate_id'),
        'current_title':             safe_str(profile.get('current_title')),
        'headline':                  safe_str(profile.get('headline')),
        'summary':                   safe_str(profile.get('summary')),
        'location':                  safe_str(profile.get('location')),
        'country':                   safe_str(profile.get('country')),
        'years_of_experience':       float(profile.get('years_of_experience') or 0.0),
        'current_company':           safe_str(profile.get('current_company')),
        'current_company_size':      safe_str(profile.get('current_company_size')),
        'current_industry':          safe_str(profile.get('current_industry')),
        'career_history':            career,
        'education':                 education,
        'skills':                    skills,
        'redrob_signals':            signals,
        'career_text':                truncate_words(career_desc, CFG['TEXT_WORD_LIMIT']),
        'career_titles_text':        career_titles,
        'candidate_text':            candidate_text,
        'last_active_date':          signals.get('last_active_date'),
        'notice_period_days':        signals.get('notice_period_days'),
        'recruiter_response_rate':   signals.get('recruiter_response_rate'),
        'interview_completion_rate': signals.get('interview_completion_rate'),
        'open_to_work_flag':         signals.get('open_to_work_flag'),
        'preferred_work_mode':       signals.get('preferred_work_mode'),
        'willing_to_relocate':       signals.get('willing_to_relocate'),
        'github_activity_score':     signals.get('github_activity_score'),
        'skill_assessment_scores':   signals.get('skill_assessment_scores') or {},
    }

def _flatten_chunk(lines):
    return [flatten_candidate(orjson.loads(l)) for l in lines]

def chunk_list(items, n_chunks):
    n_chunks = max(1, n_chunks)
    chunk_size = math.ceil(len(items) / n_chunks)
    return [items[i:i + chunk_size] for i in range(0, len(items), chunk_size)]

def parallel_row_apply(records, chunk_fn, desc):
    """Generic parallel map over a list of dict records, chunked across cores.
    Used for every heavy per-row Python loop in this pipeline (honeypot, role-fit, etc)."""
    if len(records) < CFG['PARALLEL_MIN_ROWS']:
        return chunk_fn(records)
    chunks = chunk_list(records, _CPU_CORES)
    with ProcessPoolExecutor(max_workers=_CPU_CORES) as executor:
        results = list(
            tqdm(
                executor.map(chunk_fn, chunks),
                total=len(chunks),
                desc=desc,
                mininterval=1.0
            )
        )
    out = []
    for r in results:
        out.extend(r)
    return out

def load_candidates_to_df(limit=None):
    t0 = time.time()
    with open(PATHS['candidates_jsonl'], 'rb') as f:
        raw = f.read()
    lines = [l for l in raw.splitlines() if l.strip()]
    if limit:
        lines = lines[:limit]

    if len(lines) >= CFG['PARALLEL_MIN_ROWS']:
        chunks = chunk_list(lines, _CPU_CORES)
        with ProcessPoolExecutor(max_workers=_CPU_CORES) as executor:
            chunk_results = list(
                tqdm(
                    executor.map(_flatten_chunk, chunks),
                    total=len(chunks),
                    desc="Loading candidates (parallel)",
                    mininterval=1.0
                )
            )
        rows = [r for chunk in chunk_results for r in chunk]
    else:
        rows = [flatten_candidate(orjson.loads(l)) for l in lines]

    df = pd.DataFrame(rows)
    df['years_of_experience'] = df['years_of_experience'].astype('float32')
    print(f'Loaded {len(df)} candidates in {time.time()-t0:.1f}s')
    return df

_t = _tic()
df = load_candidates_to_df(limit=TEST_ROWS if TEST_MODE else None)
_toc('Cell8_load_candidates', _t)

############################################################
# CELL 9 - Vectorized Role-Tier Classification (unchanged, already fast)
############################################################
TIER_A_PATTERN = r'\b(machine learning engineer|ml engineer|ai engineer|applied scientist|research engineer|recommendation engineer|ranking engineer|recommender systems engineer|search engineer|nlp engineer|computer vision engineer)\b'
TIER_B_PATTERN = r'\b(software engineer|backend engineer|back end engineer|data engineer|platform engineer|systems engineer|python developer|data scientist)\b'
TIER_C_PATTERN = r'\b(full.?stack|frontend|front end|software developer|backend developer|developer)\b'
TIER_D_PATTERN = r'\b(hr|human resources|recruiter|marketing manager|sales executive|content writer|graphic designer|accountant|finance|civil engineer|mechanical engineer|electrical engineer|operations manager|customer support|business analyst|project manager|teacher|nurse|doctor|legal|lawyer)\b'
ML_EVIDENCE_PATTERN = r'\b(machine learning|deep learning|nlp|bert|transformer|embedding|vector database|retrieval|ranking|recommendation|recommender|rag|llm|model serving|feature store|pytorch|tensorflow|xgboost|scikit|search relevance)\b'

def compute_role_tier_vectorized(dframe):
    title_lc = dframe['current_title'].str.lower()
    evidence_lc = (dframe['summary'] + ' ' + dframe['career_text']).str.lower()

    has_d = title_lc.str.contains(TIER_D_PATTERN, regex=True, na=False)
    has_a = title_lc.str.contains(TIER_A_PATTERN, regex=True, na=False)
    is_ds = title_lc.str.contains(r'\bdata scientist\b', regex=True, na=False)
    has_ml_evidence = evidence_lc.str.contains(ML_EVIDENCE_PATTERN, regex=True, na=False)
    has_b = title_lc.str.contains(TIER_B_PATTERN, regex=True, na=False)
    has_c = title_lc.str.contains(TIER_C_PATTERN, regex=True, na=False)

    is_d_final = has_d & ~has_a
    is_a_final = has_a
    is_ds_ml   = is_ds & has_ml_evidence & ~is_a_final
    is_b_final = has_b & has_ml_evidence & ~is_a_final & ~is_ds_ml
    is_c_final = has_c & ~is_a_final & ~is_ds_ml & ~is_b_final

    tier = np.select(
        [is_d_final, is_a_final, is_ds_ml, is_b_final, is_c_final],
        ['D', 'A', 'A', 'B', 'C'],
        default='D'
    )
    title_score = np.select(
        [is_d_final, is_a_final, is_ds_ml, is_b_final, is_c_final],
        [0.0, 1.0, 0.9, 0.7, 0.35],
        default=0.0
    )
    return tier, title_score

def role_title_tier_single(title, evidence_text=''):
    """Used for scoring recent career_history entries (shortlist + role-fit pass)."""
    title = safe_str(title)
    if re.search(TIER_D_PATTERN, title, re.I) and not re.search(TIER_A_PATTERN, title, re.I):
        return 'D', 0.0
    if re.search(TIER_A_PATTERN, title, re.I):
        return 'A', 1.0
    ev = safe_str(evidence_text)
    if re.search(r'\bdata scientist\b', title, re.I) and re.search(ML_EVIDENCE_PATTERN, ev, re.I):
        return 'A', 0.9
    if re.search(TIER_B_PATTERN, title, re.I) and re.search(ML_EVIDENCE_PATTERN, ev, re.I):
        return 'B', 0.7
    if re.search(TIER_C_PATTERN, title, re.I):
        return 'C', 0.35
    return 'D', 0.0

_t = _tic()
_tier_arr, _score_arr = compute_role_tier_vectorized(df)
df['role_tier'] = _tier_arr
df['role_title_score'] = _score_arr
_toc('Cell9_role_tier', _t)

print(df['role_tier'].value_counts())

############################################################
# CELL 10 - Honeypot Detection + Role-Fit Scoring
# (MERGED into one parallel pass, was two separate passes with two
#  separate `.to_dict('records')` conversions over the same rows —
#  that conversion alone is expensive at 100k rows and was being paid twice)
############################################################
def honeypot_check(row):
    zero_expert = sum(
        1 for s in (row['skills'] or [])
        if safe_str(s.get('proficiency')).lower() in {'advanced', 'expert'}
        and int(s.get('duration_months') or 0) == 0
    )
    if zero_expert >= 3:
        return True

    yoe = row['years_of_experience']
    career = row['career_history'] or []

    starts = [parse_date(j.get('start_date')) for j in career]
    starts = [d for d in starts if d is not None]
    if starts:
        earliest_span_years = max(0.0, (CFG['TODAY'] - min(starts)).days / 365.25)
        if earliest_span_years + 3.0 < yoe:
            return True

    if yoe > 0:
        for j in career:
            if int(j.get('duration_months') or 0) > yoe * 12 + 1:
                return True

    return False

def _role_fit_single(row):
    title_score = row['role_title_score']
    recent_jobs = list(row['career_history'] or [])[:3]
    recent_scores = [role_title_tier_single(j.get('title'), safe_str(j.get('description')))[1] for j in recent_jobs]
    identity = max([title_score] + recent_scores) if recent_scores else title_score
    return float(np.clip(0.7 * title_score + 0.3 * identity, 0, 1))

def _honeypot_and_rolefit(row):
    return honeypot_check(row), _role_fit_single(row)

def _hp_rf_chunk(records):
    return [_honeypot_and_rolefit(r) for r in records]

_t = _tic()
non_d_mask = df['role_tier'] != 'D'
non_d_records = df.loc[non_d_mask].to_dict('records')  # single conversion, reused below

combined_results = parallel_row_apply(non_d_records, _hp_rf_chunk, "Honeypot+RoleFit (parallel)")

df['honeypot_suspect'] = False
df['role_fit_score'] = df['role_title_score']

if combined_results:
    hp_vals, rf_vals = zip(*combined_results)
    df.loc[non_d_mask, 'honeypot_suspect'] = hp_vals
    df.loc[non_d_mask, 'role_fit_score'] = rf_vals

_toc('Cell10_honeypot_rolefit', _t)
print(f"Honeypot suspects flagged: {int(df['honeypot_suspect'].sum())} / {len(df)}")

############################################################
# CELL 11 - (folded into Cell 10 above, kept as no-op for numbering parity)
############################################################

############################################################
# CELL 12 - Experience Validity Scoring (only runs on final shortlist, unchanged)
############################################################
CONCEPT_PATTERNS = [
    re.compile(r'\b(embedding|sentence transformer|semantic search|vector search|vector database|pinecone|weaviate|qdrant|milvus|faiss|retrieval)\b', re.I),
    re.compile(r'\b(ndcg|mrr|map@|ranking|reranking|learning to rank|search relevance)\b', re.I),
    re.compile(r'\b(python|fastapi|flask|django|docker|kubernetes|microservice|production|deployed|mlops)\b', re.I),
    re.compile(r'\b(hybrid search|bm25|elasticsearch|opensearch|lucene)\b', re.I),
    re.compile(r'\b(llm|large language model|fine.?tun|lora|qlora|peft|rag|langchain)\b', re.I),
    re.compile(r'\b(pytorch|tensorflow|scikit|xgboost|transformer|bert|neural network|deep learning)\b', re.I),
]
N_CONCEPTS = len(CONCEPT_PATTERNS)
PRODUCTION_PATTERN = re.compile(r'\b(deployed|production|scaled|served|serving|latency|live|monitoring|a/b test|sla|throughput)\b', re.I)

def role_months_score(career_history):
    months = 0.0
    today = CFG['TODAY']
    for job in career_history or []:
        _, s = role_title_tier_single(job.get('title'), safe_str(job.get('description')))
        if s >= 0.7:
            end = parse_date(job.get('end_date')) or today
            age = max(0.0, (today - end).days / 365.25)
            rw = 1.0 if age <= 3 else max(0.35, 1.0 - 0.12 * (age - 3))
            months += int(job.get('duration_months') or 0) * rw * s
    return float(np.clip(1.0 - abs(months - 72) / 72, 0, 1))

def experience_validity_score(row):
    text = safe_str(row.get('career_text')) + ' ' + safe_str(row.get('summary'))
    hits = sum(1 for p in CONCEPT_PATTERNS if p.search(text))
    yoe = float(row.get('years_of_experience') or 0.0)
    ys = 1.0 if 5 <= yoe <= 9 else (max(0.0, yoe / 5) if yoe < 5 else max(0.35, 1.0 - (yoe - 9) / 12))
    prod = 0.08 if PRODUCTION_PATTERN.search(text) else 0.0
    return float(np.clip(
        0.45 * (hits / N_CONCEPTS) + 0.35 * role_months_score(row.get('career_history')) + 0.20 * ys + prod,
        0, 1
    ))

############################################################
# CELL 13 - Skill Relevance Scoring (unchanged, cheap, shortlist only)
############################################################
SKILL_TAXONOMY = {
    'python': 0.9, 'pandas': 0.35, 'numpy': 0.35, 'scikit-learn': 0.55, 'sklearn': 0.55,
    'machine learning': 1.0, 'ml': 0.8, 'deep learning': 0.9, 'nlp': 1.0,
    'pytorch': 0.85, 'tensorflow': 0.75, 'transformers': 0.95, 'bert': 0.9,
    'embeddings': 1.0, 'sentence transformers': 1.0, 'semantic search': 1.0,
    'vector database': 1.0, 'pinecone': 0.95, 'weaviate': 0.95, 'qdrant': 0.95,
    'milvus': 0.95, 'faiss': 0.9, 'elasticsearch': 0.75, 'opensearch': 0.75,
    'bm25': 0.8, 'hybrid search': 1.0, 'ranking': 0.9, 'learning to rank': 1.0,
    'ndcg': 1.0, 'mrr': 1.0, 'rag': 0.9, 'langchain': 0.65, 'llm': 0.85,
    'fine tuning': 0.8, 'lora': 0.85, 'qlora': 0.85, 'peft': 0.85, 'mlops': 0.75,
    'docker': 0.35, 'kubernetes': 0.45, 'fastapi': 0.35,
}
PROFICIENCY_WEIGHT = {'beginner': 0.3, 'intermediate': 0.6, 'advanced': 0.85, 'expert': 1.0}

def skill_relevance_value(name):
    n = safe_str(name).lower().strip()
    if not n:
        return 0.0
    if n in SKILL_TAXONOMY:
        return SKILL_TAXONOMY[n]
    for key, w in SKILL_TAXONOMY.items():
        if key in n or n in key:
            return w * 0.85
    return 0.0

def duration_trust(months):
    m = int(months or 0)
    if m == 0:
        return 0.1
    if m <= 6:
        return 0.5
    if m <= 12:
        return 0.75
    return 1.0

def skill_relevance_score(row):
    total = 0.0
    assessments = {
        safe_str(k).lower(): float(v)
        for k, v in (row.get('skill_assessment_scores') or {}).items() if v is not None
    }
    for skill in row.get('skills') or []:
        rel = skill_relevance_value(skill.get('name'))
        if rel <= 0:
            continue
        key = safe_str(skill.get('name')).lower()
        if key in assessments:
            trust = np.clip(assessments[key] / 100.0, 0, 1)
        else:
            trust = PROFICIENCY_WEIGHT.get(safe_str(skill.get('proficiency')).lower(), 0.45) * duration_trust(skill.get('duration_months'))
        total += rel * trust
    return float(1.0 - math.exp(-total / CFG['SKILL_SATURATION']))

############################################################
# CELL 14 - Location, Education, Availability, Github Scoring (unchanged)
############################################################
def location_logistics_score(row):
    loc = safe_str(row.get('location')).lower()
    country = safe_str(row.get('country')).lower()
    if re.search(r'\b(pune|noida)\b', loc):
        ls = 1.0
    elif re.search(r'\b(bangalore|bengaluru|hyderabad|mumbai|delhi|gurgaon|gurugram|ncr)\b', loc):
        ls = 0.85
    elif country in {'india', 'in'} or 'india' in loc:
        ls = 0.6
    else:
        ls = 0.5 if bool(row.get('willing_to_relocate')) else 0.2
    ms = {'hybrid': 1.0, 'flexible': 1.0, 'onsite': 0.8, 'remote': 0.6}.get(
        safe_str(row.get('preferred_work_mode')).lower(), 0.75
    )
    return float(0.7 * ls + 0.3 * ms)

RELEVANT_EDU_PATTERN = re.compile(
    r'\b(computer|cs|software|information technology|artificial intelligence|'
    r'machine learning|data science|statistics|mathematics|math|electronics|ece|analytics)\b', re.I
)

def education_score(row):
    edu = row.get('education') or []
    if not edu:
        return 0.5
    best = 0.4
    for e in edu:
        field = safe_str(e.get('degree')) + ' ' + safe_str(e.get('field_of_study'))
        rel = bool(RELEVANT_EDU_PATTERN.search(field))
        tier = safe_str(e.get('tier')).lower()
        s = 1.0 if rel and tier == 'tier_1' else 0.8 if rel and tier == 'tier_2' else 0.65 if rel else 0.4
        best = max(best, s)
    return float(best)

def notice_period_factor(days):
    try:
        days = float(days)
    except Exception:
        return 0.75
    if days <= 30:
        return 1.0
    if days <= 60:
        return 0.8
    if days <= 90:
        return 0.6
    return 0.45

def recency_factor(last_active_date):
    dt = parse_date(last_active_date)
    if dt is None:
        return 0.5
    days = max(0, (CFG['TODAY'] - dt).days)
    if days <= 30:
        return 1.0
    if days <= 60:
        return 0.9
    if days <= 90:
        return 0.75
    if days <= 180:
        return 0.5
    return 0.25

def availability_multiplier(row):
    rr = float(row.get('recruiter_response_rate') or 0.5)
    ic = float(row.get('interview_completion_rate') or 0.5)
    s = recency_factor(row.get('last_active_date')) * notice_period_factor(row.get('notice_period_days'))
    s *= (0.5 + 0.5 * np.clip(rr, 0, 1)) * (0.5 + 0.5 * np.clip(ic, 0, 1))
    if bool(row.get('open_to_work_flag')):
        s *= 1.15
    return float(np.clip(s, 0, 1))

def github_bonus(row):
    try:
        score = float(row.get('github_activity_score'))
    except Exception:
        return 0.0
    if score >= 70:
        return 0.05
    if score >= 40:
        return 0.02
    return 0.0

############################################################
# CELL 15 - Composite Scoring Function (unchanged)
############################################################
def score_candidate_row(row):
    base = (
        CFG['WEIGHT_ROLE']       * row.get('role_fit_score', 0.0)
        + CFG['WEIGHT_EXPERIENCE'] * experience_validity_score(row)
        + CFG['WEIGHT_SKILLS']     * skill_relevance_score(row)
        + CFG['WEIGHT_LOCATION']   * location_logistics_score(row)
        + CFG['WEIGHT_EDUCATION']  * education_score(row)
        + CFG['WEIGHT_GITHUB']     * github_bonus(row)
    )
    final = base * availability_multiplier(row)
    if row.get('role_tier') == 'D':
        final *= CFG['ROLE_D_KEEP_MULTIPLIER']
    return float(final)

############################################################
# CELL 16 - Embedding Generation (the real bottleneck fix)
#
# On top of the original ONNX/int8 fix, this version also CAPS how many
# candidates get embedded (CFG['EMBED_MAX_CANDIDATES']), prioritized by
# role_fit_score. Encoding is O(n) in pool size, so if 100k rows shrink to
# e.g. 60k after Tier-D/honeypot filtering, that's still 60k forward passes.
# Capping to the top ~8k by role_fit_score keeps encode time bounded while
# barely touching the final top-100 (role weight = 0.40, the heaviest term).
############################################################
from optimum.onnxruntime import ORTModelForFeatureExtraction, ORTQuantizer, AutoQuantizationConfig
from transformers import AutoTokenizer
import onnxruntime as ort
import hashlib

ONNX_DIR  = OUT_DIR / "onnx_fp32"
QUANT_DIR = OUT_DIR / "onnx_int8"


def prepare_onnx_model(model_id, use_int8):
    target_dir = QUANT_DIR if use_int8 else ONNX_DIR
    model_file = "model_quantized.onnx" if use_int8 else "model.onnx"

    if not (target_dir / model_file).exists():
        target_dir.mkdir(parents=True, exist_ok=True)
        base_model = ORTModelForFeatureExtraction.from_pretrained(model_id, export=True)
        base_model.save_pretrained(ONNX_DIR)
        if use_int8:
            quantizer = ORTQuantizer.from_pretrained(ONNX_DIR)
            qconfig = AutoQuantizationConfig.avx2(is_static=False, per_channel=False)
            quantizer.quantize(save_dir=QUANT_DIR, quantization_config=qconfig)

    sess_opts = ort.SessionOptions()
    sess_opts.intra_op_num_threads = _CPU_CORES
    sess_opts.inter_op_num_threads = 1
    sess_opts.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
    sess_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    sess_opts.enable_mem_pattern = True
    sess_opts.enable_cpu_mem_arena = True

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    ort_model = ORTModelForFeatureExtraction.from_pretrained(
        target_dir,
        file_name=model_file,
        provider="CPUExecutionProvider",
        session_options=sess_opts,
        use_io_binding=False,
    )
    return tokenizer, ort_model


def mean_pool_normalize(last_hidden_state, attention_mask):
    mask = attention_mask[..., None].astype(np.float32)
    summed = (last_hidden_state * mask).sum(axis=1)
    counts = np.clip(mask.sum(axis=1), 1e-9, None)
    pooled = summed / counts
    norms = np.clip(np.linalg.norm(pooled, axis=1, keepdims=True), 1e-9, None)
    return (pooled / norms).astype(np.float32)


def encode_texts(texts, tokenizer, ort_model, batch_size, max_length, desc="Embedding"):
    n = len(texts)
    if n == 0:
        return np.zeros((0, ort_model.config.hidden_size), dtype='float32')

    order = np.argsort([-len(t) for t in texts])
    sorted_texts = [texts[i] for i in order]
    num_batches = math.ceil(n / batch_size)

    def tokenize_batch(bidx):
        chunk = sorted_texts[bidx * batch_size:(bidx + 1) * batch_size]
        return tokenizer(chunk, padding=True, truncation=True, max_length=max_length, return_tensors="np")

    results = [None] * num_batches
    with ThreadPoolExecutor(max_workers=2) as tok_pool:
        prefetch_depth = 2
        futures = {i: tok_pool.submit(tokenize_batch, i) for i in range(min(prefetch_depth, num_batches))}
        next_submit = prefetch_depth
        pbar = tqdm(total=num_batches, desc=desc, mininterval=1.0)
        for i in range(num_batches):
            enc = futures.pop(i).result()
            if next_submit < num_batches:
                futures[next_submit] = tok_pool.submit(tokenize_batch, next_submit)
                next_submit += 1

            out = ort_model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"])
            pooled = mean_pool_normalize(out.last_hidden_state, enc["attention_mask"])
            results[i] = pooled
            pbar.update(1)
        pbar.close()

    embeddings_sorted = np.vstack(results).astype('float32')
    embeddings = np.empty_like(embeddings_sorted)
    embeddings[order] = embeddings_sorted
    return embeddings


def load_embed_cache():
    if not CFG['CACHE_EMBEDDINGS'] or not PATHS['embed_cache'].exists():
        return {}
    try:
        data = np.load(PATHS['embed_cache'], allow_pickle=False)
        return {h: data[h] for h in data.files}
    except Exception:
        return {}

def save_embed_cache(cache_dict):
    if not CFG['CACHE_EMBEDDINGS']:
        return
    try:
        np.savez_compressed(PATHS['embed_cache'], **cache_dict)
    except Exception as e:
        print('Could not save embedding cache:', e)

def text_hash(t):
    return hashlib.blake2b(t.encode('utf-8'), digest_size=16).hexdigest()


_t = _tic()

embed_mask = (~df['honeypot_suspect']) & (df['role_tier'] != 'D')
embed_df = df[embed_mask].copy()
print(f'Embed-eligible after honeypot/Tier-D filter: {len(embed_df)}/{len(df)}')

cap = CFG.get('EMBED_MAX_CANDIDATES')
if cap and len(embed_df) > cap:
    embed_df = embed_df.sort_values('role_fit_score', ascending=False).head(cap)
    print(f'Capped embedding pool to top {cap} by role_fit_score')

print(f'Embedding {len(embed_df)} candidates')

tokenizer, ort_model = prepare_onnx_model(CFG['EMBED_MODEL'], CFG['USE_INT8'])

corpus = embed_df['candidate_text'].fillna('').astype(str).tolist()
hashes = [text_hash(t) for t in corpus]

cache = load_embed_cache()
to_encode_idx = [i for i, h in enumerate(hashes) if h not in cache]
print(f'{len(corpus) - len(to_encode_idx)} candidates hit cache, encoding {len(to_encode_idx)} fresh')

if to_encode_idx:
    fresh_texts = [corpus[i] for i in to_encode_idx]
    fresh_embeddings = encode_texts(
        fresh_texts, tokenizer, ort_model,
        batch_size=CFG['EMBED_BATCH_SIZE'],
        max_length=CFG['MAX_SEQ_LEN'],
        desc='Embedding candidates',
    )
    for local_i, global_i in enumerate(to_encode_idx):
        cache[hashes[global_i]] = fresh_embeddings[local_i]
    save_embed_cache(cache)

embeddings = np.vstack([cache[h] for h in hashes]).astype('float32')
embed_index_map = embed_df.index.to_numpy()

_toc('Cell16_embedding', _t)

############################################################
# CELL 17 - JD Embedding + Fast Top-K Retrieval
############################################################
_t = _tic()

jd_embedding = encode_texts([JD_TEXT], tokenizer, ort_model,
                             batch_size=1, max_length=CFG['MAX_SEQ_LEN'], desc='Embedding JD')[0]

def top_k_indices(scores, k):
    k = min(k, len(scores))
    if k <= 0:
        return np.array([], dtype=int)
    part = np.argpartition(-scores, k - 1)[:k]
    return part[np.argsort(-scores[part])]

embed_scores = (embeddings @ jd_embedding).reshape(-1)
top_local = top_k_indices(embed_scores, CFG['EMBED_TOP_K'])
candidate_positions = embed_index_map[top_local]

print(f'Retrieved {len(candidate_positions)} candidates for full scoring')
_toc('Cell17_jd_retrieval', _t)

############################################################
# CELL 18 - Score Shortlist and Rank (only ~4000 rows, cheap, unchanged)
############################################################
_t = _tic()
shortlist = df.loc[candidate_positions].copy()
shortlist['score'] = [score_candidate_row(r) for r in shortlist.to_dict('records')]

top_n = CFG['TOP_N']
ranked = shortlist.sort_values(['score', 'candidate_id'], ascending=[False, True]).head(top_n).copy()
ranked['rank'] = np.arange(1, len(ranked) + 1)
_toc('Cell18_shortlist_scoring', _t)

print(f"Honeypots in top {top_n}: {int(ranked['honeypot_suspect'].sum())}")
print(f"Tier-D in top {top_n}: {int((ranked['role_tier'] == 'D').sum())}")
print('\n--- TOP RESULTS ---')
print(ranked[['candidate_id', 'current_title', 'role_tier', 'score']].to_string(index=False))

############################################################
# CELL 19 - Reasoning Generation (unchanged, cheap, shortlist only)
############################################################
def top_relevant_skills(row, n=2):
    scored_skills = sorted(
        [(skill_relevance_value(s.get('name')), safe_str(s.get('name')))
         for s in row.get('skills') or [] if skill_relevance_value(s.get('name')) > 0],
        key=lambda x: (-x[0], x[1].lower())
    )
    seen, names = set(), []
    for _, name in scored_skills:
        if name.lower() not in seen:
            seen.add(name.lower())
            names.append(name)
        if len(names) >= n:
            break
    return names

def notice_note(row):
    days = row.get('notice_period_days')
    if days is None or (isinstance(days, float) and math.isnan(days)):
        return 'notice period is not listed'
    days = int(days)
    return f'{days}-day notice period fits the JD timeline' if days <= 30 else f'{days}-day notice period is a logistics concern'

def last_active_note(row):
    dt = parse_date(row.get('last_active_date'))
    if dt is None:
        return 'activity recency is not available'
    days = max(0, (CFG['TODAY'] - dt).days)
    if days <= 30:
        return f'active recently ({days} days ago)'
    if days <= 90:
        return f'last active {days} days ago'
    return f'activity is stale at {days} days since last active'

def generate_reasoning(row):
    title = safe_str(row.get('current_title')) or 'candidate'
    yoe = float(row.get('years_of_experience') or 0.0)
    skills = top_relevant_skills(row)
    skill_text = ', '.join(skills) if skills else 'relevant ML/search signals in career history'

    concern = None
    if notice_period_factor(row.get('notice_period_days')) < 0.8:
        concern = notice_note(row) + '.'
    elif recency_factor(row.get('last_active_date')) < 0.75:
        concern = last_active_note(row).capitalize() + '.'
    elif row.get('role_tier') == 'C':
        concern = 'Role identity is adjacent rather than a direct ML/search engineering match.'

    v = int(row.get('rank', 0)) % 4
    if v == 0:
        text = f'{title} with {yoe:.1f} years shows fit through {skill_text}; {notice_note(row)} and {last_active_note(row)}.'
    elif v == 1:
        text = f'Current title is {title}, with {yoe:.1f} years and strongest skills around {skill_text}. Availability: {notice_note(row)}.'
    elif v == 2:
        text = f'{title} ranked for ML/search-adjacent career evidence plus {skill_text}. {last_active_note(row).capitalize()}.'
    else:
        text = f'Fit from {title} role identity, {yoe:.1f} years experience, skills including {skill_text}. {notice_note(row).capitalize()}.'

    if concern and concern not in text:
        text += ' ' + concern

    return re.sub(r'\s+', ' ', text).strip()[:900]

ranked['reasoning'] = ranked.apply(generate_reasoning, axis=1)

############################################################
# CELL 20 - Build and Write Submission CSV (unchanged)
# This assert block is the real guarantee that your final CSV is valid:
# exactly 100 rows, ranked 1-100, correct columns — before it ever
# reaches the validator or gets downloaded.
############################################################
submission = ranked[['candidate_id', 'rank', 'score', 'reasoning']].copy()
submission['score'] = submission['score'].astype(float)
submission = submission.sort_values('rank').reset_index(drop=True)

assert list(submission.columns) == ['candidate_id', 'rank', 'score', 'reasoning']
assert len(submission) == top_n
if not TEST_MODE:
    assert submission['rank'].tolist() == list(range(1, 101))

submission.to_csv(PATHS['submission'], index=False, encoding='utf-8')
print(f'Submission written: {PATHS["submission"]}')
print(submission[['candidate_id', 'rank', 'score']].head(10).to_string(index=False))

############################################################
# CELL 21 - Validate Submission (unchanged)
############################################################
if not TEST_MODE:
    result = subprocess.run(
        ['python', str(PATHS['validator']), str(PATHS['submission'])],
        capture_output=True, text=True
    )
    print('Validator stdout:', result.stdout)
    print('Validator stderr:', result.stderr)
    print('Validator return code:', result.returncode)
    if result.returncode != 0:
        raise RuntimeError("Submission failed validation. See validator output above.")

############################################################
# CELL 22 - Copy to Drive, Download, and clean up
# files.download() here is what actually pushes the final submission.csv
# to your browser's download folder — this only runs if Cell 20's asserts
# and Cell 21's validator both passed, so if you see the download, the
# CSV is submission-ready.
############################################################
import shutil

try:
    shutil.copy2(str(PATHS['submission']), str(PATHS['submission_drive']))
    print('Copied to Drive:', PATHS['submission_drive'])
except Exception as e:
    print('Could not copy to Drive:', e)

_toc('TOTAL_RUNTIME', _T_TOTAL)
print('\n--- STAGE TIME BREAKDOWN ---')
for k, v in _STAGE_TIMES.items():
    print(f'  {k}: {v:.1f}s')

from google.colab import files
files.download(str(PATHS['submission']))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
TEST_MODE = False  (rows = ALL)
CPU cores detected: 2
Resolved paths:
  candidates_jsonl: /content/drive/MyDrive/data/candidates.jsonl
  job_description: /content/drive/MyDrive/data/job_description.docx
  signals_doc: /content/drive/MyDrive/data/redrob_signals_doc.docx
  submission_spec: /content/drive/MyDrive/data/submission_spec.docx
  candidate_schema: /content/drive/MyDrive/data/candidate_schema.json
  sample_candidates: /content/drive/MyDrive/data/sample_candidates.json
  sample_submission: /content/drive/MyDrive/data/sample_submission.csv
  validator: /content/drive/MyDrive/data/validate_submission.py
  submission: /content/redrob_cache/full_submission.csv
  submission_drive: /content/drive/MyDrive/data/submission.csv
  embed_cache: /content/redrob_cache/embed_cache.npz
JD loaded: 9564 chars


Loading candidates (parallel):   0%|          | 0/2 [00:00<?, ?it/s]

Loaded 100000 candidates in 30.9s
[TIMER] Cell8_load_candidates: 31.1s
[TIMER] Cell9_role_tier: 8.1s
role_tier
D    81287
C    13965
B     3867
A      881
Name: count, dtype: int64


Honeypot+RoleFit (parallel):   0%|          | 0/2 [00:00<?, ?it/s]

[TIMER] Cell10_honeypot_rolefit: 29.7s
Honeypot suspects flagged: 26 / 100000


...

[TIMER] Cell16_embedding: 227.0s


Embedding JD:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieved 4000 candidates for full scoring
[TIMER] Cell17_jd_retrieval: 0.1s
[TIMER] Cell18_shortlist_scoring: 5.7s
Honeypots in top 100: 0
Tier-D in top 100: 0

--- TOP RESULTS ---
candidate_id                    current_title role_tier    score
CAND_0002025               Senior AI Engineer         A 0.756893
CAND_0081846                 Lead AI Engineer         A 0.755868
CAND_0025640             AI Research Engineer         A 0.752457
CAND_0011687              Senior NLP Engineer         A 0.751585
CAND_0043860               Junior ML Engineer         A 0.724493
CAND_0099806                      AI Engineer         A 0.704127
CAND_0053605    Senior Software Engineer (ML)         B 0.697266
CAND_0080534                      ML Engineer         A 0.689934
CAND_0040887        Machine Learning Engineer         A 0.679794
CAND_0018499 Senior Machine Learning Engineer         A 0.670936
CAND_0093912            Senior Data Scientist         A 0.662455
CAND_0030031                      AI E

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>